# CS-781 Phase 3 (Rhetorical Role Prediction) -- E0: Full fine-tuning (reference baseline, not counted)

Full fine-tuning of every trunk parameter (reference baseline: NOT on the brief's technique menu, never counted toward the 5 techniques).

Accelerator: **GPU T4 x2**. Jobs run as a per-GPU queue (two models in parallel); a GPU that fails its sanity check is dropped (fallback to one GPU). Each job: smoke test with real-model self-checks -> overfit-32 (first job) -> LR probe x{1/3,1,3} -> training with 30-minute checkpoints -> dev report + bias decision -> validated `submission.zip`. Then best-of-4 into `phase3_out/E0_full_ft/BEST/`.

To split the four models across two accounts, add e.g. `--models M4,M2` here and `--models M3,M1` on the other account.

**Attach these inputs** (Add Input -> Datasets / Notebook Output):
1. The four Phase-1 outputs (`rudra1610/01-train-gpt2style`, `02-train-llamastyle`,
   `03-train-mistral-sliding`, `04-train-bloom-alibi`; each has `final/model.pt`,
   `final/config.json`, `final/bpe_model.json`). Models are recognised by the `arch` field of
   `config.json`, so the dataset names do not matter.
2. The official Phase-3 data (`Phase_3_RR`: `Train/CL_train.json`, `CL_dev.json`,
   `IT_train.json`, `IT_dev.json`, `label_vocab.json`, `Test/CL_test.json`, `IT_test.json`) as a
   Kaggle dataset. *Alternative:* a read-only Hugging Face token as the Kaggle secret `HF` (+
   internet on); the files are then downloaded once with plain `urllib`.
3. Recommended: the output of `00_data_tests_pilot` (`PILOT.json` = causal-vs-bidir winner, and
   the `phase3_data` cache). Without it the attention mode defaults to `bidir`.
4. Optional: the output of an earlier session of THIS notebook: finished jobs are skipped and an
   unfinished one resumes from its `last.pt`.


In [ ]:
import os, sys, json, hashlib, subprocess, glob
os.makedirs("/kaggle/working/phase3", exist_ok=True)
print(sys.version.split()[0])
try:
    import torch
    print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count(),
          [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
except Exception as e:
    print("torch not importable:", e)
print("inputs:", sorted(os.listdir("/kaggle/input")) if os.path.isdir("/kaggle/input") else "none")
EXPECTED_MD5 = {
 "bpe_tokenizer.py": "906d8a37d55bf0081dd5e8d7944c0b4e",
 "phase1_models.py": "1ae9785176b01f67a305e1504b0f7ae6",
 "ner_common.py": "020d134c97ddb4ef8fc97810501a107c",
 "ner_models.py": "6e3e619bf13a1248284f5d75184923e0",
 "ner_peft.py": "d418dc954f786d538ba288e4f34ca454",
 "rr_common.py": "557a5c95227b7de8ef94a1642dfde21d",
 "rr_data.py": "9e4c1a55dc623cfe68c4b43d60ee4cb3",
 "rr_eval.py": "903575ecaf37ef7272b7f82863bb0174",
 "rr_models.py": "1f466c1edb4ba7c1d528da01ad191537",
 "rr_peft.py": "75398185ca66bb57e61088b126ff3eed",
 "rr_kd.py": "7cbb284069c5fedfd3efa3d75792470f",
 "rr_train.py": "87130b1226f8ef0de30e0374078b7770",
 "rr_predict.py": "15094a6f651300a5c9ffa99925c4c5d8",
 "rr_select.py": "61fc5fcb916cbaa39dcc2c08749cb204",
 "rr_aggregate.py": "4a57655c95a25d7c727b625969372eb5",
 "rr_ensemble.py": "91e1a8a8f8deddfded9ca75bbe29c767",
 "run_experiment.py": "46a35e0cd6068d6900cc20ca3dded98f",
 "selftest.py": "d5bb9e33d8e5faf4008a8d8a561e5ab8"
}


In [ ]:
%%writefile /kaggle/working/phase3/bpe_tokenizer.py
"""
bpe_tokenizer.py -- from-scratch byte-level BPE (pure Python, stdlib + optional numpy/torch).

No `tokenizers`, `transformers`, `sentencepiece`, `tiktoken` or any other tokenizer library.
Only: re, heapq, json, collections, random, time, gc.

Token-id layout (matches the original notebook's special-token ids):
    0 <|endoftext|>   1 <|pad|>   2 <|unk|>   3..258 = raw bytes 0..255   259.. = merges
Internally training works with "internal ids" (byte b -> b, merge k -> 256+k);
the public API adds NUM_SPECIAL (=3) to every id.
"""
import gc
import heapq
import json
import random
import re
import time
from collections import Counter
from pathlib import Path

SPECIALS = ["<|endoftext|>", "<|pad|>", "<|unk|>"]
NUM_SPECIAL = len(SPECIALS)

# GPT-2 style pre-tokenisation, expressed with the stdlib `re` module
# (\p{L} -> [^\W\d_],  \p{N} -> \d,  punctuation/symbols -> [^\s\w] or '_').
PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)"      # English contractions
    r"| ?[^\W\d_]+"               # optional space + letters
    r"| ?\d+"                     # optional space + digits
    r"| ?(?:[^\s\w]|_)+"          # optional space + punctuation / symbols
    r"|\s+(?!\S)"                 # whitespace run not followed by a non-space
    r"|\s+"                       # remaining whitespace
)

MAX_PRETOKEN_CHARS = 512          # pathological runs (e.g. 50k dashes) are cut into pieces


# =============================================================================
# 1. TRAINING
# =============================================================================
def count_pretokens(texts, max_unique=6_000_000, max_word_chars=64, log_every=20000,
                    log=print):
    """Stream documents -> Counter of pre-tokens, with a hard memory ceiling.

    When the Counter grows past `max_unique`, singletons (then rarer and rarer words) are
    pruned, so RAM stays bounded no matter how large the corpus is.
    """
    counter = Counter()
    findall = PRETOKEN_RE.findall
    n_docs = n_chars = 0
    t0 = time.time()
    for text in texts:
        counter.update(findall(text))
        n_docs += 1
        n_chars += len(text)
        if len(counter) > max_unique:
            thr = 2
            while len(counter) > int(max_unique * 0.7):
                counter = Counter({k: v for k, v in counter.items() if v >= thr})
                thr += 1
            gc.collect()
            log(f"   [count] pruned to {len(counter):,} unique pre-tokens (min count {thr - 1})")
        if n_docs % log_every == 0:
            log(f"   [count] {n_docs:,} docs | {n_chars/1e6:,.0f}M chars | "
                f"{len(counter):,} unique | {time.time()-t0:.0f}s")
    # drop absurdly long pre-tokens (junk); they carry no useful merge statistics
    if max_word_chars:
        for k in [k for k in counter if len(k) > max_word_chars]:
            del counter[k]
    return counter


def train_bpe(word_counts, vocab_size, max_words=2_000_000, min_word_count=2,
              min_pair_count=2, n_verify=5000, seed=0, log_every=1000, log=print):
    """Learn BPE merges from a Counter of pre-tokens.

    Classic incremental algorithm: exact pair counts + inverted index (pair -> words that
    contain it) + lazy max-heap, so each merge only touches the words that contain the pair.

    Returns (merges, verify) where
        merges : list[(a, b)] in rank order, in INTERNAL ids (byte b -> b, merge k -> 256+k)
        verify : list[(pretoken_str, final_internal_ids)] used to cross-check the encoder
    """
    num_merges = vocab_size - NUM_SPECIAL - 256
    assert num_merges > 0, "vocab_size too small"

    # ---- 1. pick the training vocabulary of words ---------------------------------
    items = [(w, c) for w, c in word_counts.items() if c >= min_word_count]
    if len(items) > max_words:
        items.sort(key=lambda kv: kv[1], reverse=True)
        items = items[:max_words]
    words, freqs, strs = [], [], []
    for w, c in items:
        ids = list(w.encode("utf-8", "replace"))
        if len(ids) > 1:
            words.append(ids); freqs.append(c); strs.append(w)
    del items
    log(f"   [bpe] training on {len(words):,} unique words "
        f"({sum(freqs)/1e6:,.1f}M word occurrences)")

    # ---- 2. initial pair statistics ------------------------------------------------
    pc = {}        # pair -> total count
    where = {}     # pair -> set(word index)
    for idx, ids in enumerate(words):
        f = freqs[idx]
        for p in zip(ids, ids[1:]):
            if p in pc:
                pc[p] += f
                where[p].add(idx)
            else:
                pc[p] = f
                where[p] = {idx}
    heap = [(-c, p) for p, c in pc.items()]
    heapq.heapify(heap)

    merges = []
    t0 = time.time()
    next_id = 256
    while next_id < 256 + num_merges:
        # ---- pick the most frequent pair (lazy heap: skip stale entries) -----------
        best = None
        while heap:
            negc, p = heapq.heappop(heap)
            if pc.get(p, 0) == -negc:
                best = (-negc, p)
                break
        if best is None:
            log("   [bpe] no more pairs to merge -- stopping early")
            break
        count, (a, b) = best
        if count < min_pair_count:
            log(f"   [bpe] best pair count {count} < {min_pair_count} -- stopping early")
            break

        merges.append((a, b))
        new = next_id
        next_id += 1

        # ---- apply the merge to every word that contains it ------------------------
        idxs = where.pop((a, b), ())
        changed = {}                      # pair -> count before this merge
        for idx in idxs:
            ids = words[idx]
            n = len(ids)
            out = []
            i = 0
            hit = False
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new); i += 2; hit = True
                else:
                    out.append(ids[i]); i += 1
            if not hit:
                continue
            f = freqs[idx]
            old_pairs = list(zip(ids, ids[1:]))
            new_pairs = list(zip(out, out[1:]))
            for q in old_pairs:
                if q not in changed:
                    changed[q] = pc.get(q, 0)
                pc[q] -= f
            for q in new_pairs:
                if q not in changed:
                    changed[q] = pc.get(q, 0)
                pc[q] = pc.get(q, 0) + f
            new_set = set(new_pairs)
            for q in set(old_pairs) - new_set:
                s = where.get(q)
                if s is not None:
                    s.discard(idx)
            for q in new_set:
                s = where.get(q)
                if s is None:
                    where[q] = {idx}
                else:
                    s.add(idx)
            words[idx] = out

        # ---- publish the changed pair counts ---------------------------------------
        for q, orig in changed.items():
            c = pc.get(q, 0)
            if c <= 0:
                pc.pop(q, None)
                where.pop(q, None)
            elif c != orig:
                heapq.heappush(heap, (-c, q))
        # keep the heap from growing without bound
        if len(heap) > max(4_000_000, 8 * len(pc)):
            heap = [(-c, q) for q, c in pc.items()]
            heapq.heapify(heap)

        k = len(merges)
        if k % log_every == 0 or k == num_merges:
            log(f"   [bpe] merge {k:,}/{num_merges:,} | last pair count {count:,} | "
                f"{len(pc):,} live pairs | {time.time()-t0:.0f}s")

    # ---- 3. sample of final segmentations (to verify the encoder) ----------------
    rng = random.Random(seed)
    pick = rng.sample(range(len(words)), min(n_verify, len(words)))
    verify = [(strs[i], list(words[i])) for i in pick]
    del words, freqs, strs, pc, where, heap
    gc.collect()
    return merges, verify


# =============================================================================
# 2. TOKENIZER (encode / decode / save / load)
# =============================================================================
class BPETokenizer:
    """Byte-level BPE tokenizer with a small HuggingFace-like surface:
    tok(text, add_special_tokens=False)["input_ids"], tok.encode, tok.decode,
    len(tok), eos_token_id / pad_token_id, padding_side / truncation_side.
    """

    def __init__(self, merges, cache_max=2_000_000):
        self.merges = [tuple(m) for m in merges]
        self.vocab_size = NUM_SPECIAL + 256 + len(self.merges)
        self._merge_map = {m: 256 + r for r, m in enumerate(self.merges)}   # pair -> internal id
        # id -> bytes (internal ids)
        self._bytes = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            self._bytes.append(self._bytes[a] + self._bytes[b])
        self._special_bytes = [s.encode("utf-8") for s in SPECIALS]

        self._cache = {}
        self._cache_max = cache_max

        self.eos_token, self.bos_token = SPECIALS[0], SPECIALS[0]
        self.pad_token, self.unk_token = SPECIALS[1], SPECIALS[2]
        self.eos_token_id = self.bos_token_id = 0
        self.pad_token_id = 1
        self.unk_token_id = 2
        self.padding_side = "left"        # decoder-only batched generation
        self.truncation_side = "left"     # an over-long prompt loses its beginning

    # ------------------------------------------------------------------ basics
    def __len__(self):
        return self.vocab_size

    def get_vocab(self):
        v = {s: i for i, s in enumerate(SPECIALS)}
        for i, bs in enumerate(self._bytes):
            v[bs.decode("latin-1") + f"#{i}"] = i + NUM_SPECIAL   # unique readable key
        return v

    # ------------------------------------------------------------------ encoding
    def _bpe_word(self, b):
        """BPE-merge one pre-token given as bytes -> tuple of PUBLIC ids."""
        ids = list(b)
        mm = self._merge_map
        while len(ids) > 1:
            best = None
            bp = None
            for p in zip(ids, ids[1:]):
                m = mm.get(p)
                if m is not None and (best is None or m < best):
                    best, bp = m, p
            if best is None:
                break
            a, c = bp
            out = []
            i, n = 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == c:
                    out.append(best); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        return tuple(i + NUM_SPECIAL for i in ids)

    def _encode_pretoken(self, w):
        if len(w) > MAX_PRETOKEN_CHARS:
            res = ()
            for s in range(0, len(w), MAX_PRETOKEN_CHARS):
                res += self._bpe_word(w[s:s + MAX_PRETOKEN_CHARS].encode("utf-8", "replace"))
            return res
        return self._bpe_word(w.encode("utf-8", "replace"))

    def encode(self, text, add_special_tokens=False):
        cache = self._cache
        out = []
        ext = out.extend
        for w in PRETOKEN_RE.findall(text):
            t = cache.get(w)
            if t is None:
                t = self._encode_pretoken(w)
                if len(cache) >= self._cache_max:
                    cache.clear()                 # bounded RAM
                cache[w] = t
            ext(t)
        return out

    def __call__(self, text, add_special_tokens=False, padding=False, truncation=False,
                 max_length=None, return_tensors=None, **_ignored):
        single = isinstance(text, str)
        batch = [text] if single else list(text)
        enc = [self.encode(t) for t in batch]
        if truncation and max_length is not None:
            if self.truncation_side == "left":
                enc = [e[-max_length:] if len(e) > max_length else e for e in enc]
            else:
                enc = [e[:max_length] for e in enc]
        masks = [[1] * len(e) for e in enc]
        if padding:
            target = max_length if (padding == "max_length" and max_length) else max(len(e) for e in enc)
            for k, e in enumerate(enc):
                gap = target - len(e)
                if gap > 0:
                    if self.padding_side == "left":
                        enc[k] = [self.pad_token_id] * gap + e
                        masks[k] = [0] * gap + masks[k]
                    else:
                        enc[k] = e + [self.pad_token_id] * gap
                        masks[k] = masks[k] + [0] * gap
        if return_tensors in ("pt", "np"):
            if return_tensors == "pt":
                import torch
                mk = lambda x: torch.tensor(x, dtype=torch.long)
            else:
                import numpy as np
                mk = lambda x: np.asarray(x, dtype=np.int64)
            return {"input_ids": mk(enc), "attention_mask": mk(masks)}
        if single:
            return {"input_ids": enc[0], "attention_mask": masks[0]}
        return {"input_ids": enc, "attention_mask": masks}

    # ------------------------------------------------------------------ decoding
    def decode(self, ids, skip_special_tokens=False):
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        parts = []
        nb = len(self._bytes)
        for i in ids:
            i = int(i)
            if i < NUM_SPECIAL:
                if not skip_special_tokens and i >= 0:
                    parts.append(self._special_bytes[i])
            elif i - NUM_SPECIAL < nb:
                parts.append(self._bytes[i - NUM_SPECIAL])
        return b"".join(parts).decode("utf-8", errors="replace")

    def batch_decode(self, seqs, skip_special_tokens=False):
        return [self.decode(s, skip_special_tokens) for s in seqs]

    # ------------------------------------------------------------------ persistence
    def save(self, path):
        obj = {"type": "byte_level_bpe", "version": 1, "specials": SPECIALS,
               "num_special": NUM_SPECIAL, "vocab_size": self.vocab_size,
               "pretoken_regex": PRETOKEN_RE.pattern, "merges": self.merges}
        with open(path, "w") as f:
            json.dump(obj, f)

    def save_pretrained(self, out_dir):
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        self.save(out_dir / "bpe_model.json")
        with open(out_dir / "tokenizer_config.json", "w") as f:
            json.dump({"eos_token": self.eos_token, "pad_token": self.pad_token,
                       "unk_token": self.unk_token, "eos_token_id": 0, "pad_token_id": 1,
                       "unk_token_id": 2, "vocab_size": self.vocab_size,
                       "padding_side": self.padding_side,
                       "truncation_side": self.truncation_side}, f, indent=2)

    @classmethod
    def load(cls, path, **kw):
        with open(path) as f:
            obj = json.load(f)
        assert obj.get("type") == "byte_level_bpe", "not a byte_level_bpe file"
        assert obj["pretoken_regex"] == PRETOKEN_RE.pattern, "pre-tokenizer regex mismatch"
        return cls(obj["merges"], **kw)

    @classmethod
    def from_pretrained(cls, directory, **kw):
        return cls.load(Path(directory) / "bpe_model.json", **kw)


In [ ]:
%%writefile /kaggle/working/phase3/phase1_models.py
"""
phase1_models.py -- the Phase-1 model code, copied VERBATIM from the Phase-1 training notebooks.

  * cell 13 of every notebook (identical in all four): shared core
    (CausalLMOutput, positions_from_mask, causal_allowed, build_attn_mask, init_weights,
     decoding helpers, CausalLMBase). The only change: its trailing print() was removed so
     that importing this module is silent.
  * cell 15, class definitions only (everything before `MODEL_CLASS, CONFIG_CLASS = ...`):
      - 01_train_gpt2style       -> GPT2StyleConfig, GPT2Attention, GELUMLP, GPT2Block, GPT2StyleLM
      - 02_train_llamastyle      -> LlamaStyleConfig, RMSNorm, RotaryEmbedding, GQAttention, SwiGLU,
                                    LlamaBlock, LlamaStyleLM
        (03_train_mistral_sliding defines the SAME classes; it differs only by a module-level
         SLIDING_WINDOW constant and one comment. The window comes from config.json, so one
         copy serves both M2 and M3.)
      - 04_train_bloom_alibi     -> BloomStyleConfig, alibi_slopes, alibi_bias, BloomAttention,
                                    BloomMLP, BloomBlock, BloomStyleLM

DO NOT EDIT the copied code: state_dict keys of the Phase-1 `model.pt` files depend on it.
Everything NER-specific lives in ner_models.py.
"""

# =============================================================================================
# notebook cell 13 (shared core)
# =============================================================================================
import math
from dataclasses import dataclass, asdict, field
from typing import Optional, List, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from numpy.lib.stride_tricks import sliding_window_view


@dataclass
class CausalLMOutput:
    loss: Optional[torch.Tensor] = None
    logits: Optional[torch.Tensor] = None
    past_key_values: Optional[list] = None      # per layer: (k, v), each (B, n_kv_heads, S, head_dim)


def positions_from_mask(attention_mask, past_len, T, B, device):
    """Position ids for the T new tokens. attention_mask (if given) covers ALL keys: (B, past_len+T)."""
    if attention_mask is None:
        return torch.arange(past_len, past_len + T, device=device).unsqueeze(0).expand(B, T)
    pos = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)
    return pos[:, -T:]


def causal_allowed(T, past_len, attention_mask, window, device):
    """Bool (B|1, 1, T, S): True where query t may attend key s. Always materialised."""
    S = past_len + T
    q = torch.arange(past_len, S, device=device)[:, None]      # absolute index of each query
    k = torch.arange(S, device=device)[None, :]
    allowed = k <= q                                             # causal
    if window is not None:
        allowed = allowed & (q - k < window)                     # sliding window: last `window` keys incl. self
    allowed = allowed[None, None]
    if attention_mask is not None:
        allowed = allowed & attention_mask[:, None, None, :].bool()   # drop padding keys
        allowed = allowed | (q == k)[None, None]                 # a pad query would see nothing -> NaN; let it see itself
    return allowed


def build_attn_mask(T, past_len, attention_mask, window, device):
    """Returns (mask, is_causal) for F.scaled_dot_product_attention."""
    S = past_len + T
    window_active = window is not None and window < S
    no_pad = attention_mask is None or bool(attention_mask.all())
    if no_pad and not window_active:
        if past_len == 0:
            return None, True        # fast fused causal kernel (training path)
        if T == 1:
            return None, False       # single cached query sees every key
    return causal_allowed(T, past_len, None if no_pad else attention_mask,
                          window if window_active else None, device), False


def init_weights(model, n_layer, std=0.02):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=std)
    for name, p in model.named_parameters():       # residual-branch output projections
        if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
            nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * n_layer))


# ------------------------------------------------------------------ decoding helpers
def _sample(logits, temperature, top_p, top_k):
    logits = logits / max(float(temperature), 1e-5)
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(int(top_k), logits.size(-1)), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p is not None and top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        probs = sl.softmax(-1)
        remove = probs.cumsum(-1) - probs > top_p           # keep the smallest set with mass >= top_p
        sl = sl.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(1, si, sl)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


def _banned_ngram_tokens(row, n):
    """Tokens that would complete an n-gram already present in `row` (list of ints)."""
    if n < 2 or len(row) < n:
        return []
    arr = np.asarray(row)
    win = sliding_window_view(arr[:-1], n - 1)           # window i = arr[i:i+n-1], followed by arr[i+n-1]
    hit = np.nonzero((win == arr[-(n - 1):]).all(1))[0]
    return arr[hit + n - 1].tolist()


class CausalLMBase(nn.Module):
    """Loss + generation shared by all four architectures."""

    @staticmethod
    def lm_loss(logits, labels):
        return F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)).float(),
                               labels[:, 1:].reshape(-1), ignore_index=-100)

    def _process_logits(self, logits, seq, rows, repetition_penalty, no_repeat_ngram_size, suppress):
        if repetition_penalty and repetition_penalty != 1.0:
            sc = torch.gather(logits, 1, seq)
            sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
            logits = logits.scatter(1, seq, sc)
        if no_repeat_ngram_size and no_repeat_ngram_size > 1:
            for b, row in enumerate(rows):
                banned = _banned_ngram_tokens(row, no_repeat_ngram_size)
                if banned:
                    logits[b, banned] = float("-inf")
        if suppress:
            logits[:, list(suppress)] = float("-inf")
        return logits

    @torch.inference_mode()
    def generate(self, input_ids, attention_mask=None, max_new_tokens=256, do_sample=False,
                 temperature=1.0, top_p=1.0, top_k=0, repetition_penalty=1.0,
                 no_repeat_ngram_size=0, num_beams=1, suppress_token_ids=(),
                 stop_fn=None, stop_check_every=16, **_ignored):
        """Returns ONLY the new tokens, shape (B, n_new). stop_fn(new_tokens) -> True ends early."""
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        attention_mask = attention_mask.long()
        rows = [ids[m.bool()].tolist() for ids, m in zip(input_ids, attention_mask)]   # real tokens only
        if num_beams > 1:
            return self._beam_search(input_ids, attention_mask, rows, max_new_tokens, num_beams,
                                     repetition_penalty, no_repeat_ngram_size, suppress_token_ids,
                                     stop_fn, stop_check_every)
        out = self(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out.past_key_values, out.logits[:, -1].float()
        seq, new = input_ids, []
        for step in range(max_new_tokens):
            logits = self._process_logits(logits, seq, rows, repetition_penalty,
                                          no_repeat_ngram_size, suppress_token_ids)
            nxt = _sample(logits, temperature, top_p, top_k) if do_sample else logits.argmax(-1)
            new.append(nxt)
            seq = torch.cat([seq, nxt[:, None]], 1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(nxt)[:, None]], 1)
            for row, t in zip(rows, nxt.tolist()):
                row.append(t)
            if step == max_new_tokens - 1:
                break
            if stop_fn is not None and (step + 1) % stop_check_every == 0 and stop_fn(torch.stack(new, 1)):
                break
            out = self(nxt[:, None], attention_mask=attention_mask, past_key_values=past, use_cache=True)
            past, logits = out.past_key_values, out.logits[:, -1].float()
        return torch.stack(new, 1)

    def _beam_search(self, input_ids, attention_mask, rows, max_new_tokens, K, repetition_penalty,
                     no_repeat_ngram_size, suppress, stop_fn, stop_check_every):
        B = input_ids.size(0)
        out = self(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past = [(k.repeat_interleave(K, 0), v.repeat_interleave(K, 0)) for k, v in out.past_key_values]
        logits = out.logits[:, -1].float().repeat_interleave(K, 0)
        seq = input_ids.repeat_interleave(K, 0)
        am = attention_mask.repeat_interleave(K, 0)
        rows = [list(r) for r in rows for _ in range(K)]
        scores = torch.zeros(B, K, device=input_ids.device)
        scores[:, 1:] = -1e9                       # all beams start identical: expand from beam 0 only
        gen = seq[:, :0]
        for step in range(max_new_tokens):
            logits = self._process_logits(logits, seq, rows, repetition_penalty, no_repeat_ngram_size, suppress)
            logp = logits.log_softmax(-1)
            V = logp.size(-1)
            cand = (scores.view(-1, 1) + logp).view(B, K * V)
            top, idx = cand.topk(K, dim=1)
            src = (torch.arange(B, device=idx.device)[:, None] * K + idx // V).view(-1)
            tok = (idx % V).view(-1)
            scores = top
            gen = torch.cat([gen[src], tok[:, None]], 1)
            seq = torch.cat([seq[src], tok[:, None]], 1)
            am = torch.cat([am[src], torch.ones_like(tok)[:, None]], 1)
            rows = [rows[i] + [t] for i, t in zip(src.tolist(), tok.tolist())]
            past = [(k[src], v[src]) for k, v in past]
            if step == max_new_tokens - 1:
                break
            if stop_fn is not None and (step + 1) % stop_check_every == 0 and stop_fn(gen):
                break
            out = self(tok[:, None], attention_mask=am, past_key_values=past, use_cache=True)
            past, logits = out.past_key_values, out.logits[:, -1].float()
        best = scores.argmax(1)
        return gen.view(B, K, -1)[torch.arange(B, device=best.device), best]


# =============================================================================================
# 01_train_gpt2style, cell 15 (classes)
# =============================================================================================
@dataclass
class GPT2StyleConfig:
    vocab_size: int
    n_positions: int = 2048
    n_embd: int = 768
    n_layer: int = 14
    n_head: int = 12
    mlp_ratio: int = 4
    layer_norm_eps: float = 1e-5
    initializer_range: float = 0.02


class GPT2Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.n_embd % c.n_head == 0
        self.n_head, self.head_dim, self.d = c.n_head, c.n_embd // c.n_head, c.n_embd
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd)
        self.o_proj = nn.Linear(c.n_embd, c.n_embd)

    def forward(self, x, mask, is_causal, past=None):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).split(self.d, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)      # (B, H, T, hd)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if past is not None:
            k, v = torch.cat([past[0], k], 2), torch.cat([past[1], v], 2)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=is_causal)
        y = y.transpose(1, 2).reshape(B, T, self.d)                        # merge heads
        return self.o_proj(y), (k, v)


class GELUMLP(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.up_proj = nn.Linear(d, hidden)
        self.down_proj = nn.Linear(hidden, d)

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class GPT2Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln_1 = nn.LayerNorm(c.n_embd, eps=c.layer_norm_eps)
        self.attn = GPT2Attention(c)
        self.ln_2 = nn.LayerNorm(c.n_embd, eps=c.layer_norm_eps)
        self.mlp = GELUMLP(c.n_embd, c.mlp_ratio * c.n_embd)

    def forward(self, x, mask, is_causal, past=None):
        a, present = self.attn(self.ln_1(x), mask, is_causal, past)       # pre-LN residual
        x = x + a
        x = x + self.mlp(self.ln_2(x))
        return x, present


class GPT2StyleLM(CausalLMBase):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.wte = nn.Embedding(c.vocab_size, c.n_embd)
        self.wpe = nn.Embedding(c.n_positions, c.n_embd)
        self.blocks = nn.ModuleList([GPT2Block(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.n_embd, eps=c.layer_norm_eps)
        self.lm_head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        init_weights(self, c.n_layer, c.initializer_range)
        self.lm_head.weight = self.wte.weight                              # tied input/output embeddings

    def get_input_embeddings(self):
        return self.wte

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None,
                use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = past_key_values[0][0].size(2) if past_key_values else 0
        pos = positions_from_mask(attention_mask, past_len, T, B, input_ids.device)
        pos = pos.clamp(max=self.config.n_positions - 1)
        mask, is_causal = build_attn_mask(T, past_len, attention_mask, None, input_ids.device)
        x = self.wte(input_ids) + self.wpe(pos)
        presents = []
        for i, blk in enumerate(self.blocks):
            x, pr = blk(x, mask, is_causal, past_key_values[i] if past_key_values else None)
            presents.append(pr)
        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = self.lm_loss(logits, labels) if labels is not None else None
        return CausalLMOutput(loss=loss, logits=logits, past_key_values=presents if use_cache else None)


# =============================================================================================
# 02_train_llamastyle, cell 15 (classes) -- also used for 03_train_mistral_sliding
# =============================================================================================
@dataclass
class LlamaStyleConfig:
    vocab_size: int
    max_position_embeddings: int = 2048
    hidden_size: int = 768
    intermediate_size: int = 3072
    num_hidden_layers: int = 12
    num_attention_heads: int = 12
    num_key_value_heads: int = 4
    rope_theta: float = 10000.0
    rms_norm_eps: float = 1e-5
    sliding_window: Optional[int] = None      # None = full causal attention
    initializer_range: float = 0.02


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).to(x.dtype)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, pos):                                   # pos: (B, T) int
        freqs = pos[..., None].float() * self.inv_freq        # (B, T, hd/2)
        emb = torch.cat([freqs, freqs], dim=-1)               # (B, T, hd): angle for dim i and i+hd/2
        return emb.cos()[:, None], emb.sin()[:, None]         # (B, 1, T, hd) -> broadcast over heads


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    return (x * cos + rotate_half(x) * sin).to(x.dtype)


class GQAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.hidden_size % c.num_attention_heads == 0
        assert c.num_attention_heads % c.num_key_value_heads == 0, "KV heads must divide query heads"
        self.h, self.kv = c.num_attention_heads, c.num_key_value_heads
        self.hd = c.hidden_size // c.num_attention_heads
        self.n_rep = self.h // self.kv
        self.q_proj = nn.Linear(c.hidden_size, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(c.hidden_size, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(c.hidden_size, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, c.hidden_size, bias=False)

    def forward(self, x, cos, sin, mask, is_causal, past=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)    # (B, 12, T, hd)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)   # (B,  4, T, hd)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)           # rotate Q and K only
        if past is not None:
            k, v = torch.cat([past[0], k], 2), torch.cat([past[1], v], 2)
        present = (k, v)                                                  # cache stays at 4 heads
        if self.n_rep > 1:                                                # kv head j serves q heads j*3..j*3+2
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=is_causal)
        return self.o_proj(y.transpose(1, 2).reshape(B, T, self.h * self.hd)), present


class SwiGLU(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.gate_proj = nn.Linear(d, hidden, bias=False)
        self.up_proj = nn.Linear(d, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, d, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class LlamaBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.input_norm = RMSNorm(c.hidden_size, c.rms_norm_eps)
        self.attn = GQAttention(c)
        self.post_attn_norm = RMSNorm(c.hidden_size, c.rms_norm_eps)
        self.mlp = SwiGLU(c.hidden_size, c.intermediate_size)

    def forward(self, x, cos, sin, mask, is_causal, past=None):
        a, present = self.attn(self.input_norm(x), cos, sin, mask, is_causal, past)
        x = x + a
        x = x + self.mlp(self.post_attn_norm(x))
        return x, present


class LlamaStyleLM(CausalLMBase):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.embed_tokens = nn.Embedding(c.vocab_size, c.hidden_size)
        self.rotary = RotaryEmbedding(c.hidden_size // c.num_attention_heads, c.rope_theta)
        self.layers = nn.ModuleList([LlamaBlock(c) for _ in range(c.num_hidden_layers)])
        self.norm = RMSNorm(c.hidden_size, c.rms_norm_eps)
        self.lm_head = nn.Linear(c.hidden_size, c.vocab_size, bias=False)
        init_weights(self, c.num_hidden_layers, c.initializer_range)
        self.lm_head.weight = self.embed_tokens.weight

    def get_input_embeddings(self):
        return self.embed_tokens

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None,
                use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = past_key_values[0][0].size(2) if past_key_values else 0
        pos = positions_from_mask(attention_mask, past_len, T, B, input_ids.device)
        cos, sin = self.rotary(pos)
        mask, is_causal = build_attn_mask(T, past_len, attention_mask,
                                          self.config.sliding_window, input_ids.device)
        x = self.embed_tokens(input_ids)
        presents = []
        for i, layer in enumerate(self.layers):
            x, pr = layer(x, cos, sin, mask, is_causal, past_key_values[i] if past_key_values else None)
            presents.append(pr)
        x = self.norm(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = self.lm_loss(logits, labels) if labels is not None else None
        return CausalLMOutput(loss=loss, logits=logits, past_key_values=presents if use_cache else None)


# =============================================================================================
# 04_train_bloom_alibi, cell 15 (classes)
# =============================================================================================
@dataclass
class BloomStyleConfig:
    vocab_size: int
    hidden_size: int = 768
    n_layer: int = 14
    n_head: int = 12
    mlp_ratio: int = 4
    layer_norm_eps: float = 1e-5
    initializer_range: float = 0.02


def alibi_slopes(n_head):
    def pow2_slopes(n):
        start = 2.0 ** (-(2.0 ** -(math.log2(n) - 3)))
        return [start * start ** i for i in range(n)]
    if math.log2(n_head).is_integer():
        return pow2_slopes(n_head)
    closest = 2 ** math.floor(math.log2(n_head))
    return pow2_slopes(closest) + alibi_slopes(2 * closest)[0::2][: n_head - closest]


def alibi_bias(pos_q, pos_k, slopes, allowed):
    """pos_q (B,T), pos_k (B,S), slopes (H,), allowed bool (B|1,1,T,S) -> float bias (B,H,T,S)."""
    dist = (pos_q[:, None, :, None] - pos_k[:, None, None, :]).clamp(min=0).float()
    bias = -slopes.view(1, -1, 1, 1) * dist
    return bias.masked_fill(~allowed, float("-inf"))


class BloomAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.hidden_size % c.n_head == 0
        self.h, self.hd, self.d = c.n_head, c.hidden_size // c.n_head, c.hidden_size
        self.query_key_value = nn.Linear(c.hidden_size, 3 * c.hidden_size)
        self.o_proj = nn.Linear(c.hidden_size, c.hidden_size)

    def forward(self, x, bias, past=None):
        B, T, _ = x.shape
        q, k, v = self.query_key_value(x).split(self.d, dim=-1)
        q = q.view(B, T, self.h, self.hd).transpose(1, 2)
        k = k.view(B, T, self.h, self.hd).transpose(1, 2)
        v = v.view(B, T, self.h, self.hd).transpose(1, 2)
        if past is not None:
            k, v = torch.cat([past[0], k], 2), torch.cat([past[1], v], 2)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=bias.to(q.dtype))   # additive ALiBi bias
        return self.o_proj(y.transpose(1, 2).reshape(B, T, self.d)), (k, v)


class BloomMLP(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.up_proj = nn.Linear(d, hidden)
        self.down_proj = nn.Linear(hidden, d)

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class BloomBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.input_layernorm = nn.LayerNorm(c.hidden_size, eps=c.layer_norm_eps)
        self.self_attention = BloomAttention(c)
        self.post_attention_layernorm = nn.LayerNorm(c.hidden_size, eps=c.layer_norm_eps)
        self.mlp = BloomMLP(c.hidden_size, c.mlp_ratio * c.hidden_size)

    def forward(self, x, bias, past=None):
        a, present = self.self_attention(self.input_layernorm(x), bias, past)
        x = x + a
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x, present


class BloomStyleLM(CausalLMBase):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.word_embeddings = nn.Embedding(c.vocab_size, c.hidden_size)
        self.blocks = nn.ModuleList([BloomBlock(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.hidden_size, eps=c.layer_norm_eps)
        self.lm_head = nn.Linear(c.hidden_size, c.vocab_size, bias=False)
        self.register_buffer("slopes", torch.tensor(alibi_slopes(c.n_head), dtype=torch.float32),
                             persistent=False)
        init_weights(self, c.n_layer, c.initializer_range)
        self.lm_head.weight = self.word_embeddings.weight

    def get_input_embeddings(self):
        return self.word_embeddings

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None,
                use_cache=False, last_only=False):
        B, T = input_ids.shape
        dev = input_ids.device
        past_len = past_key_values[0][0].size(2) if past_key_values else 0
        S = past_len + T
        if attention_mask is None:                          # all rows identical -> batch dim 1
            pos_k = torch.arange(S, device=dev)[None]
            allowed = causal_allowed(T, past_len, None, None, dev)
        else:
            pos_k = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)
            allowed = causal_allowed(T, past_len, attention_mask, None, dev)
        bias = alibi_bias(pos_k[:, -T:], pos_k, self.slopes, allowed)    # computed once, shared by all layers
        x = self.word_embeddings(input_ids)                  # no position embedding, no embedding LayerNorm (see note)
        presents = []
        for i, blk in enumerate(self.blocks):
            x, pr = blk(x, bias, past_key_values[i] if past_key_values else None)
            presents.append(pr)
        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = self.lm_loss(logits, labels) if labels is not None else None
        return CausalLMOutput(loss=loss, logits=logits, past_key_values=presents if use_cache else None)


# ---------------------------------------------------------------------------------------------
# Registry (added for Phase 2; not part of the copied code)
# ---------------------------------------------------------------------------------------------
MODEL_CLASSES = {
    "GPT2StyleLM": (GPT2StyleLM, GPT2StyleConfig),
    "LlamaStyleLM": (LlamaStyleLM, LlamaStyleConfig),
    "BloomStyleLM": (BloomStyleLM, BloomStyleConfig),
}


In [ ]:
%%writefile /kaggle/working/phase3/ner_common.py
"""
ner_common.py -- small torch-free helpers shared by every Phase-2 module.

  * registries: the four Phase-1 models (M1..M4) and the experiments (E1..E6, extras)
  * file discovery under /kaggle/input (or any root), atomic JSON writes
  * disk guard and output-size accounting (no Hugging Face Hub anywhere in this project)
"""
import glob
import hashlib
import json
import os
import shutil
import time

# ---------------------------------------------------------------------------------------------
# Registries
# ---------------------------------------------------------------------------------------------
# `arch` must equal the "arch" field the Phase-1 notebooks wrote into final/config.json.
MODELS = {
    "M1": {"arch": "gpt2style", "folder": "M1_gpt2style", "kind": "gpt2"},
    "M2": {"arch": "llamastyle", "folder": "M2_llamastyle", "kind": "llama"},
    "M3": {"arch": "mistral-sliding", "folder": "M3_mistral_sliding", "kind": "llama"},
    "M4": {"arch": "bloom-alibi", "folder": "M4_bloom_alibi", "kind": "bloom"},
}
DEFAULT_MODEL_ORDER = ["M4", "M2", "M3", "M1"]          # strongest Phase-1 LM first

# Layer subsets for the layer-pruned KD students (every second layer; section 8 of the brief).
STUDENT_LAYERS = {
    "M1": [0, 2, 4, 6, 8, 10, 12],      # 7 of 14
    "M2": [0, 2, 4, 6, 8, 10],          # 6 of 12
    "M3": [0, 2, 4, 6, 8, 10],          # 6 of 12
    "M4": [0, 2, 4, 6, 8, 10, 12],      # 7 of 14
}

# The six required experiments + extras. `technique` selects the code path in ner_peft.py.
EXPERIMENTS = {
    "E1": {"folder": "E1_full_ft", "technique": "full_ft", "title": "Full fine-tuning"},
    "E2": {"folder": "E2_lorma", "technique": "lorma", "title": "LoRMA+ (compulsory)"},
    "E3": {"folder": "E3_lora", "technique": "lora", "title": "LoRA"},
    "E4": {"folder": "E4_adapters", "technique": "adapters", "title": "Houlsby adapters"},
    "E5": {"folder": "E5_partial_ft", "technique": "partial_ft", "title": "Partial / layer-wise FT"},
    "E6": {"folder": "E6_kd", "technique": "kd", "title": "Knowledge distillation"},
    # extras (not counted among the six)
    "X1": {"folder": "X1_bitfit", "technique": "bitfit", "title": "BitFit (M1/M4 only)",
           "models": ["M4", "M1"]},
    "X2": {"folder": "X2_student_no_kd", "technique": "student_hard",
           "title": "Pruned student, hard labels only (KD control)"},
}
REQUIRED_EXPS = ["E1", "E2", "E3", "E4", "E5", "E6"]

# Default learning rates (section 9.3). The LR probe multiplies `lr` by {1/3, 1, 3}.
DEFAULT_LR = {
    "full_ft": 5e-5, "lorma": 1e-3, "lora": 1e-3, "adapters": 1e-3, "bitfit": 2e-3,
    "partial_ft": 1e-4, "kd": 1e-4, "student_hard": 1e-4,
}
DEFAULT_EPOCHS = {
    "full_ft": 6, "kd": 6, "student_hard": 6, "partial_ft": 8,
    "lorma": 10, "lora": 10, "adapters": 10, "bitfit": 10,
}


# ---------------------------------------------------------------------------------------------
# JSON / files
# ---------------------------------------------------------------------------------------------
def write_json(path, obj, indent=2):
    """Atomic JSON write: a crash never leaves a half-written file behind."""
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    tmp = f"{path}.tmp{os.getpid()}"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=indent, ensure_ascii=False, default=_json_default)
    os.replace(tmp, path)


def _json_default(o):
    try:
        import numpy as np
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
    except ImportError:
        pass
    raise TypeError(f"not JSON serialisable: {type(o)}")


def read_json(path, default=None):
    if not os.path.exists(path):
        return default
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def find_files(name, roots, max_hits=50):
    """All files called `name` below any of `roots` (recursive), sorted for determinism."""
    hits = []
    for r in roots:
        if r and os.path.isdir(r):
            hits.extend(glob.glob(os.path.join(r, "**", name), recursive=True))
    return sorted(set(hits))[:max_hits]


def default_input_roots():
    """Where attached Kaggle datasets live; PHASE2_INPUT_ROOTS (os.pathsep-separated) overrides."""
    env = os.environ.get("PHASE2_INPUT_ROOTS")
    if env:
        return [p for p in env.split(os.pathsep) if p]
    return ["/kaggle/input"]


def iter_finished_runs(out_root, input_roots=()):
    """Yield (exp_folder, model_folder, run_dir) for every finished run (DONE.json):
    directly under `out_root` (<exp>/<model>/DONE.json, whatever out_root is called) and under
    any `phase2_out` folder inside the attached input datasets. First occurrence wins."""
    seen = set()
    out_root = os.path.abspath(out_root) if out_root else None
    if out_root and os.path.isdir(out_root):
        for d in sorted(glob.glob(os.path.join(out_root, "*", "*", "DONE.json"))):
            exp_f, mod_f = d.split(os.sep)[-3], d.split(os.sep)[-2]
            if mod_f != "BEST" and (exp_f, mod_f) not in seen:
                seen.add((exp_f, mod_f))
                yield exp_f, mod_f, os.path.dirname(d)
    for d in find_files("DONE.json", list(input_roots), max_hits=5000):
        parts = d.split(os.sep)
        if "phase2_out" not in parts:
            continue
        i = len(parts) - 1 - parts[::-1].index("phase2_out")
        if len(parts) - i != 4 or parts[-2] == "BEST":
            continue
        if (parts[i + 1], parts[-2]) not in seen:
            seen.add((parts[i + 1], parts[-2]))
            yield parts[i + 1], parts[-2], os.path.dirname(d)


def find_phase1_models(roots=None):
    """Map M1..M4 -> directory holding that model's Phase-1 `model.pt` + `config.json`.

    Identified by the "arch" field the Phase-1 notebooks wrote into config.json, so the
    dataset/folder names do not matter.
    """
    roots = roots or default_input_roots()
    by_arch = {}
    for cfg_path in find_files("config.json", roots, max_hits=10000):
        d = os.path.dirname(cfg_path)
        if not os.path.exists(os.path.join(d, "model.pt")):
            continue
        try:
            cfg = read_json(cfg_path)
        except Exception:
            continue
        if isinstance(cfg, dict) and "arch" in cfg and "model_class" in cfg:
            by_arch.setdefault(cfg["arch"], []).append(d)
    out = {}
    for mid, m in MODELS.items():
        cands = by_arch.get(m["arch"], [])
        if cands:
            # prefer a directory literally called "final" (what the Phase-1 notebooks saved)
            cands = sorted(cands, key=lambda p: (os.path.basename(p) != "final", p))
            out[mid] = cands[0]
    return out


# ---------------------------------------------------------------------------------------------
# Disk guard (section 9.5)
# ---------------------------------------------------------------------------------------------
def free_gb(path="/kaggle/working"):
    p = path
    while p and not os.path.exists(p):
        p = os.path.dirname(p)
    return shutil.disk_usage(p or "/").free / 1e9


def dir_size_gb(path):
    tot = 0
    for dp, _dn, fns in os.walk(path):
        for fn in fns:
            fp = os.path.join(dp, fn)
            if not os.path.islink(fp):
                try:
                    tot += os.path.getsize(fp)
                except OSError:
                    pass
    return tot / 1e9


class DiskFull(RuntimeError):
    pass


def disk_guard(out_root, min_free_gb=4.0, log=print, keep=None):
    """Make sure at least `min_free_gb` is free under `out_root`.

    Deletion order: leftover last.pt / last.tmp of models that are finished (DONE.json) or not
    the one currently running (`keep` = its checkpoint dir), then tmp files. Never touches
    best.pt, submissions or reports. Raises DiskFull if still short (callers stop cleanly).
    """
    if free_gb(out_root) >= min_free_gb:
        return
    removed = []
    for pat in ("**/checkpoints/last.pt", "**/checkpoints/last.tmp", "**/*.tmp*"):
        for f in glob.glob(os.path.join(out_root, pat), recursive=True):
            if keep and os.path.abspath(f).startswith(os.path.abspath(keep)):
                continue
            try:
                sz = os.path.getsize(f) / 1e9
                os.remove(f)
                removed.append((f, sz))
            except OSError:
                pass
        if free_gb(out_root) >= min_free_gb:
            break
    for f, sz in removed:
        log(f"   [disk] removed {f} ({sz:.2f} GB)")
    if free_gb(out_root) < min_free_gb:
        raise DiskFull(f"only {free_gb(out_root):.1f} GB free under {out_root} "
                       f"(need {min_free_gb} GB) even after cleanup")


# ---------------------------------------------------------------------------------------------
# misc
# ---------------------------------------------------------------------------------------------
def now():
    return time.time()


def fmt_h(seconds):
    return f"{seconds / 3600:.2f} h"


def log_line(path, msg, also_print=True):
    """Append a timestamped line to a text log (and print it)."""
    line = time.strftime("%Y-%m-%d %H:%M:%S ") + msg
    if also_print:
        print(line, flush=True)
    if path:
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        with open(path, "a", encoding="utf-8") as f:
            f.write(line + "\n")


In [ ]:
%%writefile /kaggle/working/phase3/ner_models.py
"""
ner_models.py -- Phase-1 trunk wrappers + NER heads.

The Phase-1 models' forward() only returns LM logits and is always causal. For NER we need the
final hidden states, optionally with bidirectional attention. `Trunk` re-implements only the
tiny "embed -> blocks -> final norm" loop and REUSES the original submodule objects (no copies;
lm_head is dropped, which also avoids a (B, T, 32000) logits tensor).

Attention modes (section 5.4):
  causal : exactly the Phase-1 masks (verified equal to the original forward in tests)
  bidir  : M1/M2 key-padding mask only; M3 symmetric sliding window |q-k| < window;
           M4 ALiBi with symmetric distance -slope*|q-k|. Padding is on the right.
"""
import json
import os

import torch
import torch.nn as nn
import torch.nn.functional as F

from ner_common import MODELS
from phase1_models import (MODEL_CLASSES, RMSNorm, alibi_bias, build_attn_mask, causal_allowed,
                           positions_from_mask)

MAX_POSITIONS = 1024        # every Phase-1 model was trained on 1024-token blocks


def torch_load(path, map_location="cpu"):
    """torch.load that works on old and new torch (weights_only appeared in 1.13)."""
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)
    except Exception:
        # weights_only=True refuses non-tensor pickles (our resumable checkpoints)
        try:
            return torch.load(path, map_location=map_location, weights_only=False)
        except TypeError:
            return torch.load(path, map_location=map_location)


def load_phase1_lm(phase1_dir):
    """Rebuild a Phase-1 model from final/{config.json, model.pt}; strict key match."""
    with open(os.path.join(phase1_dir, "config.json")) as f:
        cfg = json.load(f)
    cls, cfg_cls = MODEL_CLASSES[cfg["model_class"]]
    lm = cls(cfg_cls(**cfg["config"]))
    sd = torch_load(os.path.join(phase1_dir, "model.pt"))
    lm.load_state_dict(sd, strict=True)
    return lm, cfg


def kind_of(lm):
    name = type(lm).__name__
    return {"GPT2StyleLM": "gpt2", "LlamaStyleLM": "llama", "BloomStyleLM": "bloom"}[name]


class Trunk(nn.Module):
    """embed -> selected blocks -> final norm, reusing the Phase-1 submodules."""

    def __init__(self, lm, attn_mode="bidir", layer_ids=None):
        super().__init__()
        assert attn_mode in ("causal", "bidir")
        self.kind = kind_of(lm)
        self.attn_mode = attn_mode
        c = lm.config
        self.window = None
        if self.kind == "gpt2":
            self.embed, self.wpe, blocks, self.norm = lm.wte, lm.wpe, lm.blocks, lm.ln_f
            self.n_positions = c.n_positions
            self.hidden_size = c.n_embd
        elif self.kind == "llama":
            self.embed, self.rotary, blocks, self.norm = lm.embed_tokens, lm.rotary, lm.layers, lm.norm
            self.window = c.sliding_window
            self.hidden_size = c.hidden_size
        else:
            self.embed, blocks, self.norm = lm.word_embeddings, lm.blocks, lm.ln_f
            self.register_buffer("slopes", lm.slopes.detach().clone(), persistent=False)
            self.hidden_size = c.hidden_size
        self.n_layers_original = len(blocks)
        self.layer_ids = list(range(len(blocks))) if layer_ids is None else list(layer_ids)
        assert all(0 <= i < len(blocks) for i in self.layer_ids)
        self.blocks = nn.ModuleList([blocks[i] for i in self.layer_ids])

    # ---------------------------------------------------------------- masks
    def _sdpa_mask(self, T, attention_mask, dev):
        """(mask, is_causal) for M1/M2/M3 blocks."""
        if self.attn_mode == "causal":
            return build_attn_mask(T, 0, attention_mask, self.window, dev)
        no_pad = bool(attention_mask.all())
        win = self.window if (self.window is not None and self.window < T) else None
        if no_pad and win is None:
            return None, False
        q = torch.arange(T, device=dev)[:, None]
        k = torch.arange(T, device=dev)[None, :]
        allowed = torch.ones(1, 1, T, T, dtype=torch.bool, device=dev)
        if win is not None:
            allowed = allowed & ((q - k).abs() < win)[None, None]
        if not no_pad:
            allowed = allowed & attention_mask[:, None, None, :].bool()
            allowed = allowed | (q == k)[None, None]          # no all-masked rows (pad queries)
        B = attention_mask.size(0)
        return allowed.expand(B, 1, T, T).contiguous(), False

    def _alibi(self, T, attention_mask, dev):
        pos_k = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)
        if self.attn_mode == "causal":                          # identical to BloomStyleLM.forward
            allowed = causal_allowed(T, 0, attention_mask, None, dev)
            return alibi_bias(pos_k[:, -T:], pos_k, self.slopes, allowed)
        dist = (pos_k[:, None, :, None] - pos_k[:, None, None, :]).abs().float()
        bias = -self.slopes.view(1, -1, 1, 1) * dist
        q = torch.arange(T, device=dev)[:, None]
        k = torch.arange(T, device=dev)[None, :]
        allowed = attention_mask[:, None, None, :].bool() | (q == k)[None, None]
        return bias.masked_fill(~allowed, float("-inf"))

    # ---------------------------------------------------------------- forward
    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        assert T <= MAX_POSITIONS, f"sequence of {T} tokens > {MAX_POSITIONS} (Phase-1 context)"
        dev = input_ids.device
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        pos = positions_from_mask(attention_mask, 0, T, B, dev)
        if self.kind == "gpt2":
            pos = pos.clamp(max=self.n_positions - 1)
            mask, is_causal = self._sdpa_mask(T, attention_mask, dev)
            x = self.embed(input_ids) + self.wpe(pos)
            for blk in self.blocks:
                x, _ = blk(x, mask, is_causal, None)
        elif self.kind == "llama":
            cos, sin = self.rotary(pos)
            mask, is_causal = self._sdpa_mask(T, attention_mask, dev)
            x = self.embed(input_ids)
            for blk in self.blocks:
                x, _ = blk(x, cos, sin, mask, is_causal, None)
        else:
            bias = self._alibi(T, attention_mask, dev)
            x = self.embed(input_ids)
            for blk in self.blocks:
                x, _ = blk(x, bias, None)
        return self.norm(x)

    def norm_modules(self):
        return [m for m in self.modules() if isinstance(m, (nn.LayerNorm, RMSNorm))]


class NERModel(nn.Module):
    """Trunk + dropout + Linear(H, C) [+ CRF]."""

    def __init__(self, trunk, n_labels, head="linear", dropout=0.1):
        super().__init__()
        self.trunk = trunk
        self.n_labels = n_labels
        self.num_tags = 1 + 2 * n_labels
        self.head_type = head
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(trunk.hidden_size, self.num_tags)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
        self.crf = None
        if head == "crf":
            from ner_crf import CRF
            self.crf = CRF(n_labels)

    def forward(self, input_ids, attention_mask=None, labels=None, class_weights=None):
        h = self.trunk(input_ids, attention_mask)
        logits = self.classifier(self.dropout(h)).float()
        loss = None
        if labels is not None:
            if self.crf is None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
                                       weight=class_weights, ignore_index=-100)
            else:                                   # drop the EOS position, mask = real labels
                lab = labels[:, 1:]
                loss = self.crf.neg_log_likelihood(logits[:, 1:], lab, lab != -100)
        return logits, loss


def build_ner_model(phase1_dir, n_labels, attn_mode="bidir", head="linear", layer_ids=None,
                    dropout=0.1):
    lm, cfg = load_phase1_lm(phase1_dir)
    trunk = Trunk(lm, attn_mode, layer_ids)
    del lm                                          # unused blocks / lm_head are freed here
    return NERModel(trunk, n_labels, head, dropout), cfg


def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable_only))


# ---------------------------------------------------------------------------------------------
# trained-model reconstruction (teacher for E6, prediction, ensembles)
# ---------------------------------------------------------------------------------------------
def trainable_state_dict(model, half=False):
    sd = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
    if half:
        sd = {n: (t.half() if t.is_floating_point() else t) for n, t in sd.items()}
    return sd


def load_trainable_state(model, sd):
    """Load a trainable-only state dict; every key must exist and match a trainable param."""
    params = dict(model.named_parameters())
    missing = [n for n in sd if n not in params]
    if missing:
        raise KeyError(f"checkpoint has unknown parameters: {missing[:5]}")
    with torch.no_grad():
        for n, t in sd.items():
            params[n].copy_(t.to(params[n].dtype))
    expected = {n for n, p in params.items() if p.requires_grad}
    absent = expected - set(sd)
    if absent:
        raise KeyError(f"checkpoint lacks trainable parameters: {sorted(absent)[:5]}")


def rebuild_trained_model(run_config, best_pt, phase1_dirs, device="cpu"):
    """Recreate the exact module structure of a finished run and load its best.pt."""
    from ner_peft import apply_technique
    mid = run_config["model"]
    p1 = phase1_dirs.get(mid) or run_config.get("phase1_dir")
    if not p1 or not os.path.exists(os.path.join(p1, "model.pt")):
        raise FileNotFoundError(f"Phase-1 weights for {mid} ({MODELS[mid]['arch']}) not found")
    model, _ = build_ner_model(p1, run_config["n_labels"], run_config["attn_mode"],
                               run_config["head"], run_config.get("layer_ids"))
    apply_technique(model, run_config["technique"], run_config, log=lambda *a: None)
    load_trainable_state(model, torch_load(best_pt))
    return model.to(device).eval()


In [ ]:
%%writefile /kaggle/working/phase3/ner_peft.py
"""
ner_peft.py -- fine-tuning / PEFT techniques, implemented from scratch (no `peft` library).

  full_ft      every trunk parameter + head            (E1; also KD students E6 / X2)
  lorma        LoRMA+  h = (I + s*B*A) W0 x  (+ bias)   (E2, compulsory)
  lora         LoRA    h = W0 x + s*B*A x   (+ bias)    (E3)
  adapters     Houlsby bottleneck after attention-out and MLP-out, + norms (E4)
  partial_ft   top-6 blocks + final norm + head, layer-wise LR decay (E5)
  bitfit       bias terms only (M1/M4)                  (extra X1)

`apply_technique(model, technique, cfg)` freezes everything, inserts the modules, unfreezes
exactly what the method trains, ASSERTS that the set of trainable parameter names equals the
expected set, and returns optimizer parameter groups (no weight decay on 1-D tensors).

LoRMA reference: Bihany, Patel & Modi (2025), "LoRMA: Low-Rank Multiplicative Adaptation for
LLMs", arXiv:2506.07621. We implement the additive-identity rank inflation (LoRMA+): the
frozen output y0 = W0 x is multiplied by (I + s*B*A) with B (d_out x r) = 0 at init, so the
layer starts as the exact frozen layer. The d_out x d_out product is never materialised:
y = y0 + s * (y0 @ A^T) @ B^T. For fused QKV projections each of q, k, v gets its own B, A.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from phase1_models import RMSNorm

# Linear layers adapted by LoRA / LoRMA, per architecture family (paths inside each block).
TARGETS = {
    "gpt2": ["attn.qkv", "attn.o_proj", "mlp.up_proj", "mlp.down_proj"],
    "llama": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj",
              "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"],
    "bloom": ["self_attention.query_key_value", "self_attention.o_proj",
              "mlp.up_proj", "mlp.down_proj"],
}
FUSED_QKV = {"gpt2": "attn.qkv", "bloom": "self_attention.query_key_value"}
ADAPTER_SITES = {
    "gpt2": ["attn.o_proj", "mlp.down_proj"],
    "llama": ["attn.o_proj", "mlp.down_proj"],
    "bloom": ["self_attention.o_proj", "mlp.down_proj"],
}


def _get(mod, path):
    for p in path.split("."):
        mod = getattr(mod, p)
    return mod


def _set(mod, path, new):
    parts = path.split(".")
    for p in parts[:-1]:
        mod = getattr(mod, p)
    setattr(mod, parts[-1], new)


# =============================================================================================
# modules
# =============================================================================================
class LoRALinear(nn.Module):
    """y = W0 x + b + (alpha/r) * B(A(dropout(x)))."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(self.drop(x), self.lora_A.to(x.dtype)),
                                         self.lora_B.to(x.dtype))

    @torch.no_grad()
    def merged_weight(self):
        return self.base.weight + self.scale * (self.lora_B @ self.lora_A).to(self.base.weight.dtype)


class LoRMALinear(nn.Module):
    """LoRMA+ : y = (I + s*B*A) (W0 x) + b, applied separately to each output slice."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, slices=None):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        d_out = base.out_features
        self.slices = slices or [(0, d_out)]
        assert self.slices[0][0] == 0 and self.slices[-1][1] == d_out
        assert all(a[1] == b[0] for a, b in zip(self.slices, self.slices[1:])), "slices must tile"
        self.lorma_A = nn.ParameterList()
        self.lorma_B = nn.ParameterList()
        for (s, e) in self.slices:
            A = nn.Parameter(torch.empty(r, e - s))
            nn.init.kaiming_uniform_(A, a=math.sqrt(5))
            self.lorma_A.append(A)
            self.lorma_B.append(nn.Parameter(torch.zeros(e - s, r)))

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)                  # frozen W0 x, bias added after
        parts = []
        for (s, e), A, B in zip(self.slices, self.lorma_A, self.lorma_B):
            ys = y0[..., s:e]
            parts.append(ys + self.scale * ((ys @ A.to(ys.dtype).t()) @ B.to(ys.dtype).t()))
        y = parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)
        if self.base.bias is not None:
            y = y + self.base.bias
        return y

    @torch.no_grad()
    def merged_weight(self):
        W0 = self.base.weight
        W = W0.clone()
        for (s, e), A, B in zip(self.slices, self.lorma_A, self.lorma_B):
            W[s:e] = W0[s:e] + self.scale * (B @ (A @ W0[s:e]))
        return W


class Adapter(nn.Module):
    """Houlsby bottleneck: h + up(gelu(down(h))); up ~ N(0, 1e-3) so it starts near identity."""

    def __init__(self, d, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02)
        nn.init.zeros_(self.down.bias)
        nn.init.normal_(self.up.weight, std=1e-3)
        nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class AdapterWrapped(nn.Module):
    """Wraps a sub-layer's output projection: adapter(base(x)), i.e. BEFORE the residual add."""

    def __init__(self, base: nn.Linear, bottleneck=64):
        super().__init__()
        self.base = base
        self.adapter = Adapter(base.out_features, bottleneck)

    def forward(self, x):
        return self.adapter(self.base(x))


# =============================================================================================
# applying a technique
# =============================================================================================
def _groups(named, lr, wd, tag):
    dec = [p for n, p in named if p.ndim >= 2]
    nodec = [p for n, p in named if p.ndim < 2]
    out = []
    if dec:
        out.append({"params": dec, "lr": lr, "weight_decay": wd, "name": f"{tag}/decay"})
    if nodec:
        out.append({"params": nodec, "lr": lr, "weight_decay": 0.0, "name": f"{tag}/no_decay"})
    return out


def apply_technique(model, technique, cfg, log=print):
    """Mutates `model` in place; returns optimizer param groups."""
    trunk = model.trunk
    kind = trunk.kind
    lr = float(cfg.get("lr", 1e-3))
    head_lr = float(cfg.get("head_lr", 1e-3))
    wd = float(cfg.get("weight_decay", 0.01))
    for p in model.parameters():
        p.requires_grad = False

    head_named = [(n, p) for n, p in model.named_parameters()
                  if n.startswith("classifier.") or n.startswith("crf.")]
    expected = {n for n, _ in head_named}
    groups = []

    if technique in ("full_ft", "kd", "student_hard"):
        named = [(n, p) for n, p in model.named_parameters() if n.startswith("trunk.")]
        expected |= {n for n, _ in named}
        for _, p in named:
            p.requires_grad = True
        groups += _groups(named, lr, wd, "trunk")

    elif technique in ("lora", "lorma"):
        r = int(cfg.get(f"{technique}_r", 16))
        alpha = float(cfg.get(f"{technique}_alpha", 32))
        for bi, blk in enumerate(trunk.blocks):
            for path in TARGETS[kind]:
                base = _get(blk, path)
                assert isinstance(base, nn.Linear), f"{path} is {type(base)}"
                if technique == "lora":
                    new = LoRALinear(base, r, alpha, float(cfg.get("lora_dropout", 0.05)))
                else:
                    slices = None
                    if FUSED_QKV.get(kind) == path:
                        d = trunk.hidden_size
                        assert base.out_features == 3 * d
                        slices = [(0, d), (d, 2 * d), (2 * d, 3 * d)]
                    new = LoRMALinear(base, r, alpha, slices)
                _set(blk, path, new)
        named = [(n, p) for n, p in model.named_parameters()
                 if ".lora_A" in n or ".lora_B" in n or ".lorma_A" in n or ".lorma_B" in n]
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, technique)

    elif technique == "adapters":
        bott = int(cfg.get("adapter_bottleneck", 64))
        for blk in trunk.blocks:
            for path in ADAPTER_SITES[kind]:
                _set(blk, path, AdapterWrapped(_get(blk, path), bott))
        named = [(n, p) for n, p in model.named_parameters() if ".adapter." in n]
        norm_ids = {id(p) for m in trunk.norm_modules() for p in m.parameters()}
        named += [(n, p) for n, p in model.named_parameters() if id(p) in norm_ids]
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, "adapters")

    elif technique == "bitfit":
        named = [(n, p) for n, p in model.named_parameters()
                 if n.startswith("trunk.") and n.endswith(".bias")]
        if not named:
            raise ValueError(f"BitFit needs bias terms; the {kind} architecture has none")
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, "bitfit")

    elif technique == "partial_ft":
        k = int(cfg.get("partial_top_k", 6))
        decay = float(cfg.get("layer_decay", 0.85))
        L = len(trunk.blocks)
        k = min(k, L)
        for i in range(L - k, L):
            named = [(f"trunk.blocks.{i}.{n}", p) for n, p in trunk.blocks[i].named_parameters()]
            for _, p in named:
                p.requires_grad = True
            expected |= {n for n, _ in named}
            groups += _groups(named, lr * decay ** (L - 1 - i), wd, f"block{i}")
        named = [(f"trunk.norm.{n}", p) for n, p in trunk.norm.named_parameters()]
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, "final_norm")
    else:
        raise ValueError(f"unknown technique {technique!r}")

    for _, p in head_named:
        p.requires_grad = True
    groups += _groups(head_named, head_lr, wd, "head")

    actual = {n for n, p in model.named_parameters() if p.requires_grad}
    if actual != expected:
        raise AssertionError(f"trainable-parameter mismatch: unexpected "
                             f"{sorted(actual - expected)[:5]}, missing {sorted(expected - actual)[:5]}")
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    log(f"   [{technique}] trainable {n_tr:,} / {n_all:,} params ({100 * n_tr / n_all:.3f}%) "
        f"in {len(actual)} tensors")
    return groups


def summarize_trainable(model, max_lines=40):
    rows = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
    lines = [f"{n}  {k:,}" for n, k in rows[:max_lines]]
    if len(rows) > max_lines:
        lines.append(f"... and {len(rows) - max_lines} more tensors")
    return lines


In [ ]:
%%writefile /kaggle/working/phase3/rr_common.py
"""
rr_common.py -- the Phase-3 (Rhetorical Role prediction) spec, registries and small helpers.

Torch-free. Every other rr_* module imports its constants from here, so a change in the brief
(it is a dynamic document) is a one-line edit of `SPEC` (and a regeneration of the notebooks,
whose md5 check then covers the new value everywhere).

Reused byte-identical from Phase 2 (never edited): ner_common.py (file discovery, atomic JSON,
disk guard, Phase-1 model discovery by the `arch` field), ner_models.py (Trunk), ner_peft.py
(LoRA, LoRMA+), phase1_models.py, bpe_tokenizer.py.

Inputs : nothing (constants) / environment variable PHASE3_INPUT_ROOTS (os.pathsep-separated)
Outputs: SPEC, EXPERIMENTS, TECHNIQUES, defaults, helpers
"""
import glob
import os
import time
from typing import Dict, Iterator, List, Optional, Tuple

from ner_common import (MODELS, STUDENT_LAYERS, DiskFull, dir_size_gb, disk_guard, file_md5,
                        find_files, find_phase1_models, free_gb, read_json, write_json)

__all__ = ["SPEC", "MODELS", "STUDENT_LAYERS", "EXPERIMENTS", "TECHNIQUES", "DEFAULT_LR",
           "DEFAULT_EPOCHS", "MAX_TRAINABLE_PCT", "DEFAULT_MODEL_ORDER", "OUT_ROOT", "DATA_DIR",
           "CODE_DIR", "EXIT", "GateFailure", "default_input_roots", "find_files",
           "find_phase1_models", "read_json", "write_json", "file_md5", "disk_guard", "DiskFull",
           "dir_size_gb", "free_gb", "module_md5s", "iter_finished_runs", "log_to"]

# =============================================================================================
# The brief (CS-781 Capstone, Phase 3 -- Rhetorical Role Prediction). Edit HERE only.
# =============================================================================================
SPEC: Dict = {
    "phase": 3,
    "task": "rhetorical_role_prediction",
    "hf_repo": "Exploration-Lab/CS-781-Capstone",
    "hf_folder": "Phase_3_RR",
    # key -> path inside hf_folder. Train/dev: [{"id", "text": [..], "labels": [..]}];
    # test: {id: [sentences]} (the loader also accepts [{"id", "text"}]).
    "files": {
        "CL_train": "Train/CL_train.json", "CL_dev": "Train/CL_dev.json",
        "IT_train": "Train/IT_train.json", "IT_dev": "Train/IT_dev.json",
        "label_vocab": "Train/label_vocab.json",
        "CL_test": "Test/CL_test.json", "IT_test": "Test/IT_test.json",
    },
    "domains": ["CL", "IT"],
    # the 13 official labels, in label_vocab.json index order, exact spelling ("None" is a string)
    "labels": ["Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "Dissent",
               "PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled",
               "RulingByLowerCourt", "RatioOfTheDecision", "RulingByPresentCourt", "None"],
    "metric": "macro_f1",                 # primary; macro precision / recall are secondary
    "optimise_pooling": "pooled",         # brief does not say pooled vs per-domain: log both
    "submission_entry": "prediction.json",  # must sit at the ROOT of submission.zip
    "submission_zip": "submission.zip",
    "submissions": {"min": 6, "max": 40, "max_per_day": 10},
    "rules": {
        "min_experiments": 6, "min_peft_techniques": 5, "compulsory": ["lorma"],
        "min_kd_experiments": 1, "all_models_required": True,
        "foundation_params": [50e6, 300e6],   # Phase-1 models; pruned KD students may be smaller
    },
    # technique menu of the brief (full_ft is NOT on it: reference only)
    "peft_menu": ["lora", "qlora", "adalora", "rslora", "lorma", "prefix", "prompt", "ia3"],
    "kd_technique": "kd",
    "context": 1024,                      # Phase-1 models were trained on 1024-token blocks
    "eos_id": 0, "pad_id": 1, "unk_id": 2,
    "deadline": "2026-10-16 23:59 (strict)",
    "codabench_user_format": "a_123456",
    "allowed_imports": ["torch", "numpy", "stdlib"],
}

# =============================================================================================
# Registries
# =============================================================================================
DEFAULT_MODEL_ORDER: List[str] = ["M4", "M2", "M3", "M1"]   # strongest Phase-1 LM first

# technique -> description; `peft` marks techniques that count toward the 5-technique rule
TECHNIQUES: Dict[str, Dict] = {
    "lorma":   {"peft": True,  "title": "LoRMA+ (multiplicative low-rank, compulsory)"},
    "lora":    {"peft": True,  "title": "LoRA"},
    "rslora":  {"peft": True,  "title": "rsLoRA (rank-stabilised LoRA)"},
    "ia3":     {"peft": True,  "title": "IA3"},
    "adalora": {"peft": True,  "title": "AdaLoRA (SVD + importance-based rank allocation)"},
    "qlora":   {"peft": True,  "title": "QLoRA (simulated NF4 base + LoRA)"},
    "prefix":  {"peft": True,  "title": "Prefix Tuning"},
    "prompt":  {"peft": True,  "title": "Prompt Tuning"},
    "kd":      {"peft": False, "title": "Knowledge distillation (pruned student)"},
    "full_ft": {"peft": False, "title": "Full fine-tuning (reference only)"},
}

EXPERIMENTS: Dict[str, Dict] = {
    "E0":  {"folder": "E0_full_ft", "technique": "full_ft", "counts": False, "priority": 3,
            "title": "Full fine-tuning (reference baseline, not counted)"},
    "E1":  {"folder": "E1_lorma", "technique": "lorma", "counts": True, "priority": 1,
            "title": "LoRMA+ (compulsory)"},
    "E2":  {"folder": "E2_lora", "technique": "lora", "counts": True, "priority": 1,
            "title": "LoRA"},
    "E3":  {"folder": "E3_rslora", "technique": "rslora", "counts": True, "priority": 2,
            "title": "rsLoRA"},
    "E4":  {"folder": "E4_ia3", "technique": "ia3", "counts": True, "priority": 2,
            "title": "IA3"},
    "E5":  {"folder": "E5_adalora", "technique": "adalora", "counts": True, "priority": 2,
            "title": "AdaLoRA"},
    "E6":  {"folder": "E6_kd", "technique": "kd", "counts": True, "priority": 1,
            "title": "Knowledge distillation (self-contained full-FT teacher -> pruned student)"},
    "E7":  {"folder": "E7_qlora", "technique": "qlora", "counts": True, "priority": 3,
            "title": "QLoRA (simulated NF4)"},
    "E8":  {"folder": "E8_prefix", "technique": "prefix", "counts": True, "priority": 3,
            "title": "Prefix Tuning"},
    "E9":  {"folder": "E9_prompt", "technique": "prompt", "counts": True, "priority": 3,
            "title": "Prompt Tuning"},
    "E10": {"folder": "E10_ablations", "technique": "ablation", "counts": False, "priority": 3,
            "title": "Ablations: context, head, loss, domain handling"},
    "E11": {"folder": "E11_ensemble", "technique": "ensemble", "counts": False, "priority": 9,
            "title": "Ensemble of BEST models + final report"},
}
PILOT_FOLDER = "P0_pilot"

# Section 6 starting points; the LR probe multiplies `lr` by {1/3, 1, 3}.
DEFAULT_LR: Dict[str, float] = {
    "lorma": 1e-3, "lora": 1e-3, "rslora": 1e-3, "adalora": 1e-3, "qlora": 1e-3,
    "ia3": 3e-3, "prefix": 2e-3, "prompt": 3e-3, "full_ft": 5e-5, "kd": 1e-4,
}
DEFAULT_EPOCHS: Dict[str, int] = {
    "lorma": 10, "lora": 10, "rslora": 10, "adalora": 10, "qlora": 10,
    "ia3": 12, "prefix": 15, "prompt": 15, "full_ft": 6, "kd": 8,
}
# upper bound on trainable % of the (real-size) model, asserted by selftest --full-size
MAX_TRAINABLE_PCT: Dict[str, float] = {
    "lorma": 5.0, "lora": 5.0, "rslora": 8.0, "adalora": 5.0, "qlora": 5.0,
    "ia3": 0.5, "prefix": 6.0, "prompt": 0.5,
}

# =============================================================================================
# Paths / exit codes
# =============================================================================================
CODE_DIR = os.path.dirname(os.path.abspath(__file__))
OUT_ROOT = "/kaggle/working/phase3_out"
DATA_DIR = "/kaggle/working/phase3_data"
OUT_MARKER = "phase3_out"                 # folder name used to find earlier outputs in inputs
EXIT = {"ok": 0, "error": 1, "missing_dependency": 2, "gate_failure": 3,
        "validation_failure": 4}

MODULE_FILES = ["bpe_tokenizer.py", "phase1_models.py", "ner_common.py", "ner_models.py",
                "ner_peft.py", "rr_common.py", "rr_data.py", "rr_eval.py", "rr_models.py",
                "rr_peft.py", "rr_kd.py", "rr_train.py", "rr_predict.py", "rr_select.py",
                "rr_aggregate.py", "rr_ensemble.py", "run_experiment.py", "selftest.py"]


class GateFailure(RuntimeError):
    """A 'stop and ask' condition: the pipeline must not produce a submission."""


def default_input_roots() -> List[str]:
    """Where attached Kaggle datasets live; PHASE3_INPUT_ROOTS (os.pathsep-separated) overrides."""
    env = os.environ.get("PHASE3_INPUT_ROOTS")
    if env:
        return [p for p in env.split(os.pathsep) if p]
    return ["/kaggle/input"]


def module_md5s(code_dir: str = CODE_DIR) -> Dict[str, Optional[str]]:
    """md5 of every shared module (recorded in each run_config.json; no git needed)."""
    out = {}
    for f in MODULE_FILES:
        p = os.path.join(code_dir, f)
        out[f] = file_md5(p) if os.path.exists(p) else None
    return out


def iter_finished_runs(out_root: Optional[str], input_roots=()) -> Iterator[Tuple[str, str, str]]:
    """Yield (exp_folder, job_folder, run_dir) for every finished run (DONE.json) directly under
    `out_root` (<exp>/<job>/DONE.json) and under any `phase3_out` folder inside the attached
    inputs. BEST folders are skipped. First occurrence wins."""
    seen = set()
    if out_root and os.path.isdir(out_root):
        for d in sorted(glob.glob(os.path.join(os.path.abspath(out_root), "*", "*", "DONE.json"))):
            exp_f, job_f = d.split(os.sep)[-3], d.split(os.sep)[-2]
            if job_f != "BEST" and (exp_f, job_f) not in seen:
                seen.add((exp_f, job_f))
                yield exp_f, job_f, os.path.dirname(d)
    for d in find_files("DONE.json", list(input_roots), max_hits=20000):
        parts = d.split(os.sep)
        if OUT_MARKER not in parts:
            continue
        i = len(parts) - 1 - parts[::-1].index(OUT_MARKER)
        if len(parts) - i != 4 or parts[-2] == "BEST":
            continue
        if (parts[i + 1], parts[-2]) not in seen:
            seen.add((parts[i + 1], parts[-2]))
            yield parts[i + 1], parts[-2], os.path.dirname(d)


def student_layers(mid: str, phase1_dir: str) -> List[int]:
    """Every second layer of the Phase-1 model (= STUDENT_LAYERS for the real 12/14-layer
    models; derived from config.json so that tiny test models work too)."""
    cfg = read_json(os.path.join(phase1_dir, "config.json"))["config"]
    n = cfg.get("n_layer", cfg.get("num_hidden_layers"))
    ids = list(range(0, n, 2))
    if n in (12, 14):
        assert ids == STUDENT_LAYERS[mid], (ids, STUDENT_LAYERS[mid])
    return ids


def log_to(path: Optional[str], msg: str, prefix: str = "") -> None:
    """Print a line and append it (timestamped) to a log file."""
    line = f"{prefix}{msg}"
    print(line, flush=True)
    if path:
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        with open(path, "a", encoding="utf-8") as f:
            f.write(time.strftime("%H:%M:%S ") + line + "\n")


In [ ]:
%%writefile /kaggle/working/phase3/rr_data.py
"""
rr_data.py -- Phase-3 data: loading, gates G0/G1/G2, tokenised cache, window packing.

Torch-free (numpy + stdlib). Run on CPU:

    python rr_data.py build [--out /kaggle/working/phase3_data]

Gates (logged to GATES.json in the cache dir; any failure raises GateFailure = stop and ask):
  G0 schema   every doc has an id and a list of sentences; train/dev: len(text) == len(labels),
              no empty docs, every label in label_vocab.json, label_vocab == SPEC["labels"];
              no duplicate ids; CL_test and IT_test ids must not collide (a single
              prediction.json dict would be ambiguous -> STOP, ask the TAs on Discord).
              Overlaps of test ids with train/dev ids are printed loudly (not fatal).
  G1 tokens   every sentence is encoded; length stats (mean/p50/p95/p99/max per sentence,
              tokens per doc); a sentence that encodes to zero tokens gets [unk] (counted);
              truncations at max_sent_tokens are counted.
  G2 output   (a) window packing round trip on every train/dev/test doc for every input mode
              (each sentence covered, markers at the right positions, order and count
              recovered); (b) the submission validator of rr_predict accepts a dummy all-"None"
              prediction built from the real test ids and rejects 5 corrupted variants.

Input representations (section 4 of the code-generation prompt):
  doc    sentences packed into windows of <= context tokens; every sentence is followed by the
         EOS token (id 0) = its MARKER; the sentence vector is the hidden state at the marker.
         Long documents use overlapping windows (train stride 0.75, inference stride 0.5 of the
         budget, in tokens, snapped to sentence boundaries). Each sentence is predicted from
         the window where it has the most context (bidir: centre-most = max(min(left, right));
         causal: most left context), or from the mean over its windows (`window_combine`).
  sent   one sentence per window (context ablation baseline).
  ctx_k  target sentence + k neighbours on each side; only the target's marker is labelled.
A domain prefix ("[CL]" / "[IT]", tokenised as normal text: the tokenizer has no spare
special tokens) starts every window. Sentences are truncated to `max_sent_tokens` (256).
"""
import argparse
import os
import pickle
import sys
import time
import urllib.error
import urllib.request
from collections import Counter
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

from rr_common import (SPEC, GateFailure, default_input_roots, file_md5, find_files, read_json,
                       write_json)

IGNORE = -100
CACHE_VERSION = 3


# =============================================================================================
# 1. locating / downloading the official files
# =============================================================================================
def _hf_read_token() -> Optional[str]:
    """Optional READ token for downloading the official data (never used for writing)."""
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient          # only exists on Kaggle
        return UserSecretsClient().get_secret("HF")
    except Exception:
        return None


def _download(rel_path: str, dest: str, token: Optional[str]) -> None:
    url = f"https://huggingface.co/datasets/{SPEC['hf_repo']}/resolve/main/{rel_path}"
    req = urllib.request.Request(url)
    if token:
        req.add_header("Authorization", f"Bearer {token}")
    with urllib.request.urlopen(req, timeout=120) as r, open(dest + ".part", "wb") as f:
        while True:
            b = r.read(1 << 20)
            if not b:
                break
            f.write(b)
    os.replace(dest + ".part", dest)


def locate_raw_files(raw_dir: str, roots: Optional[List[str]] = None, allow_download: bool = True,
                     log=print) -> Dict[str, str]:
    """{key: path} for every file in SPEC["files"], found under the input roots (attached Kaggle
    dataset) or downloaded once, read-only, from the Hugging Face dataset repo."""
    roots = roots or default_input_roots()
    found = {}
    for key, rel in SPEC["files"].items():
        name = os.path.basename(rel)
        hits = [h for h in find_files(name, roots + [raw_dir]) if "phase3_out" not in h]
        if hits:
            sub = os.path.dirname(rel)                         # "Train" / "Test"
            hits.sort(key=lambda p: (SPEC["hf_folder"] not in p, sub not in p, len(p)))
            found[key] = hits[0]
    missing = [k for k in SPEC["files"] if k not in found]
    if missing and allow_download:
        os.makedirs(raw_dir, exist_ok=True)
        token = _hf_read_token()
        for key in missing:
            rel = f"{SPEC['hf_folder']}/{SPEC['files'][key]}"
            dest = os.path.join(raw_dir, key + ".json")
            try:
                log(f"   downloading {rel} ...")
                _download(rel, dest, token)
                found[key] = dest
            except (urllib.error.URLError, OSError) as e:
                log(f"   could not download {rel}: {e}")
    missing = [SPEC["files"][k] for k in SPEC["files"] if k not in found]
    if missing:
        raise FileNotFoundError(
            "Phase-3 files not found: " + ", ".join(missing) + f". Attach the official "
            f"{SPEC['hf_folder']} folder as a Kaggle dataset (or add a read-only HF token as the "
            f"Kaggle secret 'HF' and enable internet).")
    return found


# =============================================================================================
# 2. tolerant loaders
# =============================================================================================
def load_split_file(path: str, has_labels: bool) -> List[Dict]:
    """[{"id": str, "text": [str], "labels": [str] | None}] from either
    [{"id", "text", "labels"?}]  or  {id: [sentences]}  or  {id: {"text", "labels"?}}."""
    obj = read_json(path)
    docs = []
    if isinstance(obj, list):
        for i, rec in enumerate(obj):
            if not isinstance(rec, dict) or "id" not in rec or "text" not in rec:
                raise GateFailure(f"G0: {path}: element {i} lacks 'id'/'text': "
                                  f"{str(rec)[:200]}")
            docs.append({"id": str(rec["id"]), "text": rec["text"],
                         "labels": rec.get("labels")})
    elif isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, list):
                docs.append({"id": str(k), "text": v, "labels": None})
            elif isinstance(v, dict) and "text" in v:
                docs.append({"id": str(k), "text": v["text"], "labels": v.get("labels")})
            else:
                raise GateFailure(f"G0: {path}: document {k!r} has unexpected value "
                                  f"{str(v)[:200]}")
    else:
        raise GateFailure(f"G0: {path}: top level must be a list or an object")
    for d in docs:
        if not isinstance(d["text"], list) or not all(isinstance(s, str) for s in d["text"]):
            raise GateFailure(f"G0: {path}: document {d['id']!r}: 'text' must be a list of "
                              f"strings")
        if has_labels and d["labels"] is None:
            raise GateFailure(f"G0: {path}: document {d['id']!r} has no labels")
        if not has_labels:
            d["labels"] = None
    return docs


def load_label_vocab(path: str) -> List[str]:
    """label_vocab.json {label: index} -> labels in index order; must equal SPEC["labels"]."""
    obj = read_json(path)
    if isinstance(obj, dict):
        labels = [k for k, _ in sorted(obj.items(), key=lambda kv: int(kv[1]))]
        if sorted(int(v) for v in obj.values()) != list(range(len(obj))):
            raise GateFailure(f"G0: label_vocab indices are not 0..{len(obj) - 1}: {obj}")
    elif isinstance(obj, list):
        labels = [str(x) for x in obj]
    else:
        raise GateFailure(f"G0: label_vocab.json has an unexpected layout: {str(obj)[:200]}")
    if labels != SPEC["labels"]:
        raise GateFailure(f"G0: label_vocab.json {labels} differs from SPEC['labels'] "
                          f"{SPEC['labels']} (spelling/order). The brief may have changed: "
                          f"update rr_common.SPEC and regenerate the notebooks.")
    return labels


# =============================================================================================
# 3. window packing (pure functions, unit-tested in selftest.py and gate G2)
# =============================================================================================
def sentence_piece(ids: np.ndarray, max_sent_tokens: int, eos_id: int) -> np.ndarray:
    """Sentence ids truncated to max_sent_tokens, followed by the EOS marker."""
    return np.concatenate([ids[:max_sent_tokens], [eos_id]]).astype(np.int32)


def doc_ranges(lengths: Sequence[int], budget: int, stride: int) -> List[Tuple[int, int]]:
    """Sentence ranges [a, b) whose pieces (lengths incl. marker) fit in `budget` tokens.
    Consecutive ranges overlap: the next range starts at the first sentence at least `stride`
    tokens after the current start (never after the current end, so nothing is skipped)."""
    n = len(lengths)
    if n == 0:
        return []
    assert max(lengths) <= budget, f"a sentence piece ({max(lengths)}) exceeds the budget {budget}"
    off = np.concatenate([[0], np.cumsum(lengths)])
    out, a = [], 0
    while True:
        b = a + 1
        while b < n and off[b + 1] - off[a] <= budget:
            b += 1
        out.append((a, b))
        if b >= n:
            break
        a2 = a + 1
        while a2 < b and off[a2] - off[a] < stride:
            a2 += 1
        a = a2
    return out


def parse_mode(mode: str) -> Tuple[str, int]:
    if mode in ("doc", "sent"):
        return mode, 0
    if mode.startswith("ctx_"):
        k = int(mode.split("_", 1)[1])
        assert k >= 1
        return "ctx", k
    raise ValueError(f"unknown input mode {mode!r}")


def window_budget(cfg: Dict, meta: Dict, domain: str) -> Tuple[List[int], int]:
    prefix = list(meta["domain_prefix_ids"][domain]) if cfg.get("domain_prefix", True) else []
    budget = int(cfg.get("max_len", SPEC["context"])) - int(cfg.get("n_virtual", 0)) - len(prefix)
    return prefix, budget


def build_doc_windows(doc: Dict, cfg: Dict, meta: Dict, purpose: str) -> List[Dict]:
    """Windows of one document. Each window: tokens (int32), markers / starts / sent_idx /
    labels (int64, one per sentence slot), score (float32 context score per slot)."""
    mode, k = parse_mode(cfg.get("input_mode", "doc"))
    eos = int(meta["eos_id"])
    max_sent = int(cfg.get("max_sent_tokens", 256))
    prefix, budget = window_budget(cfg, meta, doc["domain"])
    pieces = [sentence_piece(s, max_sent, eos) for s in doc["ids"]]
    lens = [len(p) for p in pieces]
    labels = doc["labels"]
    causal = cfg.get("attn_mode", "bidir") == "causal"
    wins = []

    def make(sent_ids: List[int], labelled: List[bool]) -> Dict:
        toks = list(prefix)
        markers, starts = [], []
        for i in sent_ids:
            starts.append(len(toks))
            toks.extend(pieces[i].tolist())
            markers.append(len(toks) - 1)
        toks = np.asarray(toks, dtype=np.int32)
        markers = np.asarray(markers, dtype=np.int64)
        L = len(toks)
        score = (markers.astype(np.float32) if causal else
                 np.minimum(markers, L - 1 - markers).astype(np.float32))
        lab = np.full(len(sent_ids), IGNORE, dtype=np.int64)
        if labels is not None:
            for j, i in enumerate(sent_ids):
                if labelled[j]:
                    lab[j] = int(labels[i])
        keep = [j for j in range(len(sent_ids)) if labelled[j]]
        return {"key": doc["key"], "tokens": toks, "markers": markers[keep],
                "starts": np.asarray(starts, dtype=np.int64)[keep],
                "sent_idx": np.asarray(sent_ids, dtype=np.int64)[keep], "labels": lab[keep],
                "score": score[keep]}

    if mode == "doc":
        frac = float(cfg.get("train_stride_frac" if purpose == "train" else "infer_stride_frac",
                             0.75 if purpose == "train" else 0.5))
        stride = max(1, int(frac * budget))
        for a, b in doc_ranges(lens, budget, stride):
            ids = list(range(a, b))
            wins.append(make(ids, [True] * len(ids)))
    elif mode == "sent":
        for i in range(len(pieces)):
            wins.append(make([i], [True]))
    else:
        n = len(pieces)
        for i in range(n):
            left = list(range(max(0, i - k), i))
            right = list(range(i + 1, min(n, i + 1 + k)))
            while sum(lens[j] for j in left + [i] + right) > budget and (left or right):
                if len(left) >= len(right):            # drop the farthest neighbour first
                    left = left[1:]
                else:
                    right = right[:-1]
            ids = left + [i] + right
            wins.append(make(ids, [j == i for j in ids]))
    return wins


def build_windows(docs: Dict[str, Dict], keys: Sequence[str], cfg: Dict, meta: Dict,
                  purpose: str) -> List[Dict]:
    out = []
    for key in keys:
        for w in build_doc_windows(docs[key], cfg, meta, purpose):
            out.append(w)
    return out


def assign_sentences(windows: Sequence[Dict], n_sent: Dict[str, int]) -> Dict[str, List[List]]:
    """{doc_key: [[(window_idx, slot_idx, score), ...] per sentence]}. Raises if a sentence is
    not covered by any window (this is the 'unpack' of the packing round trip)."""
    occ = {k: [[] for _ in range(n)] for k, n in n_sent.items()}
    for wi, w in enumerate(windows):
        for j, si in enumerate(w["sent_idx"].tolist()):
            occ[w["key"]][si].append((wi, j, float(w["score"][j])))
    for k, lst in occ.items():
        miss = [i for i, o in enumerate(lst) if not o]
        if miss:
            raise AssertionError(f"{k}: sentences {miss[:5]} are not covered by any window")
    return occ


def combine_slot_scores(windows: Sequence[Dict], slot_scores: Sequence[np.ndarray],
                        n_sent: Dict[str, int], how: str = "center") -> Dict[str, np.ndarray]:
    """Per-slot scores (one (n_slots, C) array per window) -> {doc_key: (n_sent, C)}.
    center: the slot with the highest context score (ties: earliest window); mean: average."""
    occ = assign_sentences(windows, n_sent)
    C = slot_scores[0].shape[1] if len(slot_scores) else len(SPEC["labels"])
    out = {}
    for key, lst in occ.items():
        arr = np.zeros((len(lst), C), dtype=np.float32)
        for i, o in enumerate(lst):
            if how == "mean":
                arr[i] = np.mean([slot_scores[wi][j] for wi, j, _ in o], axis=0)
            else:
                wi, j, _ = max(o, key=lambda t: (t[2], -t[0]))
                arr[i] = slot_scores[wi][j]
        out[key] = arr
    return out


def check_packing(doc: Dict, cfg: Dict, meta: Dict, purpose: str) -> Dict:
    """Round-trip check of one document's windows (gate G2a, selftest #2)."""
    wins = build_doc_windows(doc, cfg, meta, purpose)
    eos = int(meta["eos_id"])
    max_sent = int(cfg.get("max_sent_tokens", 256))
    ctx = int(cfg.get("max_len", SPEC["context"])) - int(cfg.get("n_virtual", 0))
    for w in wins:
        assert len(w["tokens"]) <= ctx, f"window of {len(w['tokens'])} tokens > {ctx}"
        assert np.all(np.diff(w["sent_idx"]) > 0) or len(w["sent_idx"]) <= 1, "order broken"
        for j, si in enumerate(w["sent_idx"].tolist()):
            m, s = int(w["markers"][j]), int(w["starts"][j])
            assert w["tokens"][m] == eos, "marker is not EOS"
            want = doc["ids"][si][:max_sent]
            assert np.array_equal(w["tokens"][s:m], want), "marker maps to the wrong sentence"
    occ = assign_sentences(wins, {doc["key"]: len(doc["ids"])})[doc["key"]]
    chosen = [max(o, key=lambda t: (t[2], -t[0])) for o in occ]
    recovered = [int(wins[wi]["sent_idx"][j]) for wi, j, _ in chosen]
    assert recovered == list(range(len(doc["ids"]))), "unpack(pack(doc)) != sentence order"
    return {"windows": len(wins), "max_window_tokens": max(len(w["tokens"]) for w in wins),
            "max_occurrences": max(len(o) for o in occ)}


# =============================================================================================
# 4. class statistics
# =============================================================================================
def class_counts(docs: Dict[str, Dict], keys: Sequence[str], n_labels: int) -> np.ndarray:
    labs = [docs[k]["labels"] for k in keys if docs[k]["labels"] is not None]
    if not labs:
        return np.zeros(n_labels, dtype=np.int64)
    return np.bincount(np.concatenate(labs).astype(np.int64), minlength=n_labels)


def class_weights(counts: np.ndarray, kind: str = "inv_sqrt") -> np.ndarray:
    """Inverse-sqrt-frequency weights, normalised to mean 1 over the classes seen in train;
    unseen classes get the largest weight (they cannot be learned, but must not be 0)."""
    c = counts.astype(np.float64)
    w = np.zeros_like(c)
    seen = c > 0
    w[seen] = 1.0 / np.sqrt(c[seen])
    if seen.any():
        w[seen] /= w[seen].mean()
        w[~seen] = w[seen].max()
    else:
        w[:] = 1.0
    return w.astype(np.float32)


# =============================================================================================
# 5. cache build (gates G0 / G1 / G2)
# =============================================================================================
def load_tokenizer(roots: Optional[List[str]] = None, explicit: Optional[str] = None):
    """bpe_model.json from the Phase-1 final/ dirs (all copies must be identical)."""
    from bpe_tokenizer import BPETokenizer
    paths = [explicit] if explicit else [p for p in find_files("bpe_model.json",
                                                               roots or default_input_roots())
                                         if "phase3_out" not in p]
    if not paths:
        raise FileNotFoundError("bpe_model.json not found; attach the Phase-1 output datasets")
    hashes = {file_md5(p) for p in paths}
    if len(hashes) != 1:
        raise GateFailure(f"found {len(hashes)} DIFFERENT bpe_model.json files: {paths}")
    return BPETokenizer.load(paths[0]), paths[0], hashes.pop()


def _pct(a: Sequence[int]) -> Dict[str, float]:
    a = np.asarray(a, dtype=np.float64)
    if a.size == 0:
        return {}
    q = np.percentile(a, [50, 95, 99])
    return {"mean": float(a.mean()), "p50": float(q[0]), "p95": float(q[1]), "p99": float(q[2]),
            "max": float(a.max())}


def build_cache(out_dir: str, roots: Optional[List[str]] = None,
                tokenizer_path: Optional[str] = None, max_sent_tokens: int = 256,
                log=print) -> Dict:
    t0 = time.time()
    os.makedirs(out_dir, exist_ok=True)
    gates: Dict = {"started": time.strftime("%Y-%m-%d %H:%M:%S")}
    paths = locate_raw_files(os.path.join(out_dir, "raw"), roots, log=log)
    gates["raw_files"] = paths
    gates["raw_md5"] = {k: file_md5(v) for k, v in paths.items()}

    # ---- G0 ---------------------------------------------------------------------------------
    labels = load_label_vocab(paths["label_vocab"])
    lab2i = {l: i for i, l in enumerate(labels)}
    raw: Dict[str, List[Dict]] = {}
    for dom in SPEC["domains"]:
        for split in ("train", "dev", "test"):
            raw[f"{dom}_{split}"] = load_split_file(paths[f"{dom}_{split}"], split != "test")
    g0 = {"docs": {}, "sentences": {}, "class_hist": {}, "warnings": []}
    for name, docs in raw.items():
        ids = [d["id"] for d in docs]
        dup = [i for i, c in Counter(ids).items() if c > 1]
        if dup:
            raise GateFailure(f"G0: duplicate document ids in {name}: {dup[:5]}")
        for d in docs:
            if not d["text"]:
                raise GateFailure(f"G0: {name}: document {d['id']!r} has no sentences")
            if d["labels"] is not None:
                if len(d["labels"]) != len(d["text"]):
                    raise GateFailure(f"G0: {name}: document {d['id']!r} has "
                                      f"{len(d['text'])} sentences but {len(d['labels'])} labels")
                bad = sorted({l for l in d["labels"] if l not in lab2i})
                if bad:
                    raise GateFailure(f"G0: {name}: document {d['id']!r} uses labels outside "
                                      f"label_vocab.json: {bad}")
        g0["docs"][name] = len(docs)
        g0["sentences"][name] = int(sum(len(d["text"]) for d in docs))
        if not name.endswith("test"):
            hist = Counter(l for d in docs for l in d["labels"])
            g0["class_hist"][name] = {l: int(hist.get(l, 0)) for l in labels}
    ids_of = {n: {d["id"] for d in docs} for n, docs in raw.items()}
    clash = ids_of["CL_test"] & ids_of["IT_test"]
    g0["id_overlap"] = {}
    for a in ids_of:
        for b in ids_of:
            if a < b and ids_of[a] & ids_of[b]:
                g0["id_overlap"][f"{a}&{b}"] = sorted(ids_of[a] & ids_of[b])[:10]
    for pair, ex in g0["id_overlap"].items():
        if "test" in pair:
            msg = f"!!! LOUD WARNING: document ids shared by {pair}: {ex[:5]}"
            g0["warnings"].append(msg)
            log(msg)
    if clash:
        write_json(os.path.join(out_dir, "GATES.json"), {**gates, "G0": g0, "status": "FAILED"})
        raise GateFailure(
            f"G0: CL_test and IT_test share {len(clash)} document ids (e.g. {sorted(clash)[:3]}). "
            f"A single prediction.json dict would be ambiguous. STOP and ask the TAs on Discord; "
            f"do not guess a prefix.")
    log(f"   G0 passed: docs {g0['docs']} | sentences {g0['sentences']}")
    for n, h in g0["class_hist"].items():
        log(f"   class histogram {n}: {h}")

    # ---- G1: tokenisation -------------------------------------------------------------------
    tok, tok_path, tok_md5 = load_tokenizer(roots, tokenizer_path)
    gates["tokenizer"] = {"path": tok_path, "md5": tok_md5, "vocab": len(tok)}
    eos, pad, unk = int(tok.eos_token_id), int(tok.pad_token_id), int(tok.unk_token_id)
    if (eos, pad, unk) != (SPEC["eos_id"], SPEC["pad_id"], SPEC["unk_id"]):
        raise GateFailure(f"G1: tokenizer specials (eos, pad, unk) = {(eos, pad, unk)} differ "
                          f"from the spec")
    docs: Dict[str, Dict] = {}
    split_keys: Dict[str, List[str]] = {"train": [], "dev": [], "test": []}
    test_ids: Dict[str, List[str]] = {}
    sent_len, doc_len = [], []
    n_unk = n_trunc = 0
    for dom in SPEC["domains"]:
        for split in ("train", "dev", "test"):
            for d in raw[f"{dom}_{split}"]:
                key = f"{dom}/{d['id']}"
                ids = []
                for s in d["text"]:
                    e = tok.encode(s.strip())
                    if not e:
                        e = [unk]
                        n_unk += 1
                    if len(e) > max_sent_tokens:
                        n_trunc += 1
                    ids.append(np.asarray(e, dtype=np.int32))
                    sent_len.append(len(e))
                doc_len.append(int(sum(len(x) + 1 for x in ids)))
                docs[key] = {"key": key, "id": d["id"], "domain": dom, "split": split,
                             "text": d["text"], "ids": ids,
                             "labels": (np.asarray([lab2i[l] for l in d["labels"]], dtype=np.int64)
                                        if d["labels"] is not None else None)}
                split_keys[split].append(key)
                if split == "test":
                    test_ids.setdefault(dom, []).append(d["id"])
    assert all(len(x) > 0 for d in docs.values() for x in d["ids"]), "zero-token sentence"
    prefix_ids = {dom: tok.encode(f"[{dom}]") for dom in SPEC["domains"]}
    g1 = {"tokens_per_sentence": _pct(sent_len), "tokens_per_doc": _pct(doc_len),
          "empty_sentences_unk_fallback": n_unk, "max_sent_tokens": max_sent_tokens,
          "sentences_truncated": n_trunc, "total_tokens": int(sum(sent_len)),
          "domain_prefix_ids": prefix_ids}
    log(f"   G1 passed: tokens/sentence {g1['tokens_per_sentence']} | tokens/doc "
        f"{g1['tokens_per_doc']} | unk fallbacks {n_unk} | truncated at {max_sent_tokens}: "
        f"{n_trunc}")

    meta = {"version": CACHE_VERSION, "labels": labels, "split": split_keys,
            "test_ids": test_ids, "eos_id": eos, "pad_id": pad, "unk_id": unk,
            "vocab_size": len(tok), "tokenizer_md5": tok_md5, "tokenizer_path": tok_path,
            "domain_prefix_ids": prefix_ids, "raw_md5": gates["raw_md5"],
            "built": time.strftime("%Y-%m-%d %H:%M:%S")}

    # ---- G2a: packing round trip ------------------------------------------------------------
    g2 = {"packing": {}}
    for mode in ("doc", "sent", "ctx_1"):
        for attn in ("bidir", "causal"):
            cfg = {"input_mode": mode, "attn_mode": attn, "max_len": SPEC["context"],
                   "max_sent_tokens": max_sent_tokens, "n_virtual": 20}
            st = Counter()
            for key, d in docs.items():
                for purpose in ("train", "infer"):
                    r = check_packing(d, cfg, meta, purpose)
                    st["windows_" + purpose] += r["windows"]
                    st["max_window_tokens"] = max(st["max_window_tokens"], r["max_window_tokens"])
            g2["packing"][f"{mode}/{attn}"] = dict(st)
    log(f"   G2a passed: window packing round trip for doc/sent/ctx_1 x bidir/causal "
        f"({g2['packing']['doc/bidir']})")

    # ---- G2b: submission validator round trip -------------------------------------------------
    from rr_predict import g2_validator_roundtrip
    n_sent = {d["id"]: len(d["text"]) for d in docs.values() if d["split"] == "test"}
    g2["validator"] = g2_validator_roundtrip(n_sent, labels, out_dir)
    log(f"   G2b passed: validator accepts the dummy all-'None' prediction and rejects "
        f"{len(g2['validator']['rejected'])} corrupted variants")

    with open(os.path.join(out_dir, "docs.pkl.tmp"), "wb") as f:
        pickle.dump(docs, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(os.path.join(out_dir, "docs.pkl.tmp"), os.path.join(out_dir, "docs.pkl"))
    write_json(os.path.join(out_dir, "meta.json"), meta)
    gates.update({"G0": g0, "G1": g1, "G2": g2, "status": "PASSED",
                  "seconds": round(time.time() - t0, 1)})
    write_json(os.path.join(out_dir, "GATES.json"), gates)
    log(f"   data cache written to {out_dir} in {gates['seconds']} s")
    return meta


def load_cache(data_dir: str) -> Tuple[Dict, Dict[str, Dict]]:
    meta = read_json(os.path.join(data_dir, "meta.json"))
    if meta is None:
        raise FileNotFoundError(f"no data cache in {data_dir}")
    with open(os.path.join(data_dir, "docs.pkl"), "rb") as f:
        docs = pickle.load(f)
    return meta, docs


def find_or_build_cache(data_dir: str, roots: Optional[List[str]] = None, log=print,
                        **kw) -> str:
    """Reuse a cache (here or attached) built with the SAME tokenizer and raw files, else build."""
    roots = roots or default_input_roots()
    try:
        _tok, _p, tok_md5 = load_tokenizer(roots)
    except FileNotFoundError:
        tok_md5 = None

    def ok(d):
        meta = read_json(os.path.join(d, "meta.json"))
        return (isinstance(meta, dict) and meta.get("version") == CACHE_VERSION
                and os.path.exists(os.path.join(d, "docs.pkl"))
                and (tok_md5 is None or meta.get("tokenizer_md5") == tok_md5)
                and read_json(os.path.join(d, "GATES.json"), {}).get("status") == "PASSED")
    if ok(data_dir):
        return data_dir
    for m in find_files("meta.json", roots):
        d = os.path.dirname(m)
        if ok(d):
            log(f"   using the data cache attached at {d}")
            return d
    build_cache(data_dir, roots=roots, log=log, **kw)
    return data_dir


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["build"])
    ap.add_argument("--out", default="/kaggle/working/phase3_data")
    ap.add_argument("--tokenizer", default=None, help="explicit path to bpe_model.json")
    ap.add_argument("--max-sent-tokens", type=int, default=256)
    ap.add_argument("--force", action="store_true", help="rebuild even if a valid cache exists")
    a = ap.parse_args()
    try:
        if a.force or a.tokenizer:
            build_cache(a.out, tokenizer_path=a.tokenizer, max_sent_tokens=a.max_sent_tokens)
        else:
            d = find_or_build_cache(a.out, max_sent_tokens=a.max_sent_tokens)
            g = read_json(os.path.join(d, "GATES.json"), {})
            print(f"   data cache: {d} | gates {g.get('status')} | G0 docs "
                  f"{(g.get('G0') or {}).get('docs')} | G1 tokens/sentence "
                  f"{(g.get('G1') or {}).get('tokens_per_sentence')}")
            for w in (g.get("G0") or {}).get("warnings", []):
                print(w)
    except GateFailure as e:
        print("\nSTOP AND ASK:", e, file=sys.stderr)
        sys.exit(3)
    except FileNotFoundError as e:
        print("\nMISSING INPUT:", e, file=sys.stderr)
        sys.exit(2)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase3/rr_eval.py
"""
rr_eval.py -- Macro-Precision / Recall / F1 from scratch (numpy only), reports, bias tuning.

Macro rule (documented, unit-tested in selftest.py): a class that has ZERO gold support AND
ZERO predictions in the evaluated set is skipped (excluded from the macro average) and listed
in `classes_skipped`. Every other class is included; a class that is predicted but absent
from gold, or present in gold but never predicted, scores P = R = F1 = 0 (0/0 -> 0). This is
what sklearn's f1_score(average="macro") does with labels=None (labels = union of gold and
predicted), which is the most likely behaviour of the official scorer. Macro-F1 is the mean
of per-class F1 (not the harmonic mean of macro-P and macro-R).

Pooling: the brief does not say whether CL and IT are pooled or averaged per domain, so we
report `pooled` (all dev sentences together; the selection metric), `per_domain` and
`domain_avg_macro_f1`.

Post-hoc bias tuning (section 8): an additive per-class logit bias found by coordinate ascent
on a bounded grid. Because dev is also used for early stopping, the real gain is estimated by
2-fold cross-fitting over a 50/50 document split (per domain): fit on one half, measure on the
other, both ways. Biases fitted on the FULL dev are applied to test only if the mean held-out
gain is positive; the decision is recorded.

Inputs : gold / predicted label indices (per document), logits (per document)
Outputs: metric dicts, confusion matrix, bias decision dict
"""
import csv
import os
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

BIAS_GRID = np.round(np.arange(-3.0, 3.0001, 0.25), 4)


# =============================================================================================
# metrics
# =============================================================================================
def confusion(gold: Sequence[int], pred: Sequence[int], n_classes: int) -> np.ndarray:
    """cm[g, p] = number of sentences with gold g predicted as p."""
    g = np.asarray(gold, dtype=np.int64)
    p = np.asarray(pred, dtype=np.int64)
    assert g.shape == p.shape, f"gold {g.shape} vs pred {p.shape}"
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    np.add.at(cm, (g, p), 1)
    return cm


def scores_from_confusion(cm: np.ndarray, labels: Sequence[str]) -> Dict:
    tp = np.diag(cm).astype(np.float64)
    support = cm.sum(1).astype(np.float64)          # gold count per class
    n_pred = cm.sum(0).astype(np.float64)           # predicted count per class
    fp, fn = n_pred - tp, support - tp
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    rec = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    used = (support > 0) | (n_pred > 0)
    total = cm.sum()
    per = {labels[c]: {"precision": float(prec[c]), "recall": float(rec[c]), "f1": float(f1[c]),
                       "support": int(support[c]), "predicted": int(n_pred[c]),
                       "in_macro": bool(used[c])} for c in range(len(labels))}
    if not used.any():
        return {"macro_f1": 0.0, "macro_precision": 0.0, "macro_recall": 0.0, "accuracy": 0.0,
                "n": 0, "per_class": per, "classes_used": [], "classes_skipped": list(labels)}
    return {"macro_f1": float(f1[used].mean()), "macro_precision": float(prec[used].mean()),
            "macro_recall": float(rec[used].mean()),
            "accuracy": float(tp.sum() / total) if total else 0.0, "n": int(total),
            "per_class": per,
            "classes_used": [labels[c] for c in range(len(labels)) if used[c]],
            "classes_skipped": [labels[c] for c in range(len(labels)) if not used[c]]}


def macro_scores(gold: Sequence[int], pred: Sequence[int], labels: Sequence[str]) -> Dict:
    return scores_from_confusion(confusion(gold, pred, len(labels)), labels)


def macro_f1(gold: Sequence[int], pred: Sequence[int], n_classes: int) -> float:
    """Fast path (used inside bias tuning): same rule as macro_scores."""
    cm = confusion(gold, pred, n_classes)
    tp = np.diag(cm).astype(np.float64)
    sup, npred = cm.sum(1), cm.sum(0)
    denom = sup + npred
    used = denom > 0
    if not used.any():
        return 0.0
    f1 = np.divide(2 * tp, denom, out=np.zeros_like(tp), where=denom > 0)   # = 2PR/(P+R)
    return float(f1[used].mean())


def full_report(gold_by_doc: Dict[str, np.ndarray], pred_by_doc: Dict[str, np.ndarray],
                domain_of: Dict[str, str], labels: Sequence[str]) -> Dict:
    """Pooled + per-domain macro scores and the pooled confusion matrix."""
    keys = sorted(gold_by_doc)
    for k in keys:
        assert len(gold_by_doc[k]) == len(pred_by_doc[k]), f"{k}: length mismatch"
    g = np.concatenate([gold_by_doc[k] for k in keys])
    p = np.concatenate([pred_by_doc[k] for k in keys])
    cm = confusion(g, p, len(labels))
    rep = {"pooled": scores_from_confusion(cm, labels), "per_domain": {},
           "confusion": cm.tolist(), "labels": list(labels)}
    for dom in sorted(set(domain_of[k] for k in keys)):
        ks = [k for k in keys if domain_of[k] == dom]
        rep["per_domain"][dom] = macro_scores(np.concatenate([gold_by_doc[k] for k in ks]),
                                              np.concatenate([pred_by_doc[k] for k in ks]),
                                              labels)
    doms = rep["per_domain"]
    rep["domain_avg_macro_f1"] = float(np.mean([v["macro_f1"] for v in doms.values()])) \
        if doms else 0.0
    return rep


def headline(rep: Dict) -> float:
    """The selection metric: pooled Macro-F1."""
    return float(rep["pooled"]["macro_f1"])


def write_confusion_csv(path: str, cm: Sequence[Sequence[int]], labels: Sequence[str]) -> None:
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["gold\\pred"] + list(labels))
        for lab, row in zip(labels, cm):
            w.writerow([lab] + [int(x) for x in row])


# =============================================================================================
# post-hoc bias tuning
# =============================================================================================
def tune_bias(logits: np.ndarray, gold: np.ndarray, grid: np.ndarray = BIAS_GRID,
              rounds: int = 3) -> Tuple[np.ndarray, float]:
    """Coordinate ascent over an additive per-class bias (bounded grid). Returns (bias, F1).
    Ties keep the value closest to 0, so an uninformative class keeps bias 0."""
    C = logits.shape[1]
    b = np.zeros(C, dtype=np.float64)
    best = macro_f1(gold, (logits + b).argmax(1), C)
    order = np.argsort(np.abs(grid), kind="stable")
    for _ in range(rounds):
        changed = False
        for c in range(C):
            cur_v, cur_f = b[c], best
            for v in grid[order]:
                if v == b[c]:
                    continue
                b2 = b.copy()
                b2[c] = v
                f = macro_f1(gold, (logits + b2).argmax(1), C)
                if f > cur_f + 1e-12:
                    cur_v, cur_f = v, f
            if cur_v != b[c]:
                b[c], best, changed = cur_v, cur_f, True
        if not changed:
            break
    return b.astype(np.float32), float(best)


def halves(keys: Sequence[str], domain_of: Dict[str, str]) -> Tuple[List[str], List[str]]:
    """Deterministic 50/50 document split, alternating within each domain."""
    a, b = [], []
    for dom in sorted(set(domain_of[k] for k in keys)):
        ks = sorted(k for k in keys if domain_of[k] == dom)
        a += ks[0::2]
        b += ks[1::2]
    return a, b


def bias_decision(logits_by_doc: Dict[str, np.ndarray], gold_by_doc: Dict[str, np.ndarray],
                  domain_of: Dict[str, str], n_classes: int) -> Dict:
    keys = sorted(gold_by_doc)

    def cat(ks):
        return (np.concatenate([logits_by_doc[k] for k in ks]).astype(np.float64),
                np.concatenate([gold_by_doc[k] for k in ks]))
    A, B = halves(keys, domain_of)
    folds = []
    if A and B:
        for fit, ev in ((A, B), (B, A)):
            lf, gf = cat(fit)
            le, ge = cat(ev)
            bias, _ = tune_bias(lf, gf)
            before = macro_f1(ge, le.argmax(1), n_classes)
            after = macro_f1(ge, (le + bias).argmax(1), n_classes)
            folds.append({"fit_docs": len(fit), "eval_docs": len(ev), "before": before,
                          "after": after, "gain": after - before})
    lf, gf = cat(keys)
    bias_full, f_full = tune_bias(lf, gf)
    before_full = macro_f1(gf, lf.argmax(1), n_classes)
    gain = float(np.mean([f["gain"] for f in folds])) if folds else 0.0
    apply = bool(folds) and gain > 0.0
    return {"apply": apply, "heldout_gain_mean": gain, "folds": folds,
            "bias": bias_full.tolist() if apply else [0.0] * n_classes,
            "bias_fitted_full_dev": bias_full.tolist(), "dev_before": before_full,
            "dev_after_in_sample": f_full,
            "rule": "apply the full-dev biases only if the mean 2-fold held-out gain > 0"}


def labels_from_logits(logits_by_doc: Dict[str, np.ndarray],
                       bias: Optional[Sequence[float]] = None) -> Dict[str, np.ndarray]:
    b = np.zeros(1) if bias is None else np.asarray(bias, dtype=np.float64)
    return {k: (v.astype(np.float64) + b).argmax(1).astype(np.int64)
            for k, v in logits_by_doc.items()}


In [ ]:
%%writefile /kaggle/working/phase3/rr_models.py
"""
rr_models.py -- RRModel = Phase-1 trunk + sentence head (pure PyTorch).

`RRTrunk` subclasses the Phase-2 `ner_models.Trunk` (byte-identical, reused): with no virtual
tokens it calls Trunk.forward unchanged (causal/bidir masks, RoPE, ALiBi, sliding window, GQA
as verified in Phase 2). Two optional additions serve Prompt and Prefix Tuning:

  prompt  (P, d) soft embeddings prepended to the input embeddings (Lester et al., 2021).
          Position ids / masks are built for P + T tokens; prompt columns are visible to every
          query (also outside M3's sliding window); outputs are returned for the T real tokens.
  prefix  per-layer key/value prefixes (Li & Liang, 2021), passed through the Phase-1 blocks'
          own `past` argument, i.e. concatenated AFTER RoPE (prefix keys are not rotated), at
          the KV-head count (GQA), with mask columns that are True for every query (also under
          the sliding window and causal masks) and a zero ALiBi bias.

Sentence vectors are gathered at the EOS marker positions (or mean-pooled over the sentence's
tokens, `pool="mean"`). Heads: `linear` = LayerNorm -> Dropout -> Linear(d, 13); `seqctx` = a
1-layer BiGRU over the ordered sentence vectors of a window (residual) before the linear head,
to model label transitions (Fact -> Issue -> Argument -> Ratio -> Ruling).

Inputs : input_ids / attention_mask (B, T), markers / starts / marker_mask (B, M)
Outputs: logits (B, M, 13) float32, sentence vectors (B, M, d)
"""
import os
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn

from ner_models import MAX_POSITIONS, Trunk, load_phase1_lm, torch_load
from phase1_models import causal_allowed, positions_from_mask

STATE_BUFFER_TAGS = (".ada_mask", ".ada_ipt", ".ada_unc")   # non-parameter state to checkpoint


# =============================================================================================
# trunk with virtual tokens
# =============================================================================================
class RRTrunk(Trunk):
    def __init__(self, lm, attn_mode: str = "bidir", layer_ids: Optional[List[int]] = None):
        super().__init__(lm, attn_mode, layer_ids)
        self.prompt: Optional[nn.Parameter] = None   # set by rr_peft (prompt tuning)
        self.prefix: Optional[nn.Module] = None      # set by rr_peft (prefix tuning)
        self.force_virtual = False                   # test hook: use the virtual path even if empty
        if self.kind == "gpt2":
            self.n_kv, self.head_dim = lm.config.n_head, lm.config.n_embd // lm.config.n_head
        elif self.kind == "llama":
            self.n_kv = lm.config.num_key_value_heads
            self.head_dim = lm.config.hidden_size // lm.config.num_attention_heads
        else:
            self.n_kv, self.head_dim = lm.config.n_head, lm.config.hidden_size // lm.config.n_head

    @property
    def n_virtual(self) -> int:
        """Positions consumed by the soft prompt (prefix tuning consumes none)."""
        return 0 if self.prompt is None else int(self.prompt.shape[0])

    def forward(self, input_ids, attention_mask=None):
        if self.prompt is None and self.prefix is None and not self.force_virtual:
            return super().forward(input_ids, attention_mask)
        return self._forward_virtual(input_ids, attention_mask)

    # ------------------------------------------------------------------ helpers
    @staticmethod
    def _work_dtype(x: torch.Tensor) -> torch.dtype:
        if x.is_cuda and torch.is_autocast_enabled():
            try:
                return torch.get_autocast_dtype("cuda")
            except (AttributeError, TypeError):
                return torch.get_autocast_gpu_dtype()
        return x.dtype

    @staticmethod
    def _bool_mask(mask, is_causal, T, dev):
        """(mask, is_causal) from Trunk -> explicit bool (B|1, 1, T, T) or None (= all True)."""
        if mask is None:
            return causal_allowed(T, 0, None, None, dev) if is_causal else None
        return mask

    def _forward_virtual(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        dev = input_ids.device
        P = self.n_virtual
        T2 = P + T
        assert T2 <= MAX_POSITIONS, f"{P} virtual + {T} tokens > {MAX_POSITIONS}"
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        am = attention_mask
        if P:
            am = torch.cat([torch.ones(B, P, dtype=attention_mask.dtype, device=dev),
                            attention_mask], 1)
        pos = positions_from_mask(am, 0, T2, B, dev)
        x = self.embed(input_ids)
        if P:
            x = torch.cat([self.prompt.to(x.dtype).unsqueeze(0).expand(B, P, -1), x], 1)
        past = self.prefix(B, self._work_dtype(x), dev) if self.prefix is not None else None
        K = past[0][0].shape[2] if past is not None else 0

        if self.kind in ("gpt2", "llama"):
            mask, is_causal = self._sdpa_mask(T2, am, dev)
            if P or K:
                mask = self._bool_mask(mask, is_causal, T2, dev)
                is_causal = False
                if mask is not None and P:
                    mask = mask.clone()
                    mask[..., :P] = True                    # soft prompt visible to every query
                if mask is not None and K:
                    mask = torch.cat([torch.ones(*mask.shape[:-1], K, dtype=torch.bool,
                                                 device=dev), mask], -1)
            if self.kind == "gpt2":
                x = x + self.wpe(pos.clamp(max=self.n_positions - 1))
                for i, blk in enumerate(self.blocks):
                    x, _ = blk(x, mask, is_causal, past[i] if past is not None else None)
            else:
                cos, sin = self.rotary(pos)
                for i, blk in enumerate(self.blocks):
                    x, _ = blk(x, cos, sin, mask, is_causal, past[i] if past is not None else None)
        else:
            bias = self._alibi(T2, am, dev)                  # (B, H, T2, T2), -inf = masked
            if P:
                bias = bias.clone()
                bias[..., :P] = torch.where(torch.isinf(bias[..., :P]),
                                            torch.zeros_like(bias[..., :P]), bias[..., :P])
            if K:
                bias = torch.cat([torch.zeros(*bias.shape[:-1], K, dtype=bias.dtype, device=dev),
                                  bias], -1)              # ALiBi bias 0 for prefix keys
            for i, blk in enumerate(self.blocks):
                x, _ = blk(x, bias, past[i] if past is not None else None)
        return self.norm(x)[:, P:]


# =============================================================================================
# heads
# =============================================================================================
class SentenceHead(nn.Module):
    def __init__(self, d: int, n_labels: int, head: str = "linear", dropout: float = 0.1):
        super().__init__()
        assert head in ("linear", "seqctx")
        self.kind = head
        self.ctx = None
        if head == "seqctx":
            self.ctx = nn.GRU(d, d // 2, num_layers=1, batch_first=True, bidirectional=True)
            self.ctx_norm = nn.LayerNorm(d)
        self.norm = nn.LayerNorm(d)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d, n_labels)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, v: torch.Tensor, marker_mask: torch.Tensor) -> torch.Tensor:
        if self.ctx is not None:
            lens = marker_mask.sum(1).clamp(min=1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(self.ctx_norm(v.float()), lens,
                                                       batch_first=True, enforce_sorted=False)
            out, _ = self.ctx(packed)
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True,
                                                      total_length=v.shape[1])
            v = v.float() + out
        return self.classifier(self.dropout(self.norm(v.float()))).float()


class RRModel(nn.Module):
    """Trunk -> sentence vectors (marker / mean pooling) -> head."""

    def __init__(self, trunk: RRTrunk, n_labels: int, head: str = "linear",
                 dropout: float = 0.1, pool: str = "marker"):
        super().__init__()
        assert pool in ("marker", "mean")
        self.trunk = trunk
        self.n_labels = n_labels
        self.pool = pool
        self.head = SentenceHead(trunk.hidden_size, n_labels, head, dropout)
        self.kd_proj: Optional[nn.Module] = None     # optional, KD intermediate term

    def sentence_vectors(self, input_ids, attention_mask, markers, starts):
        h = self.trunk(input_ids, attention_mask)                   # (B, T, d)
        d = h.shape[-1]
        if self.pool == "marker":
            return torch.gather(h, 1, markers.unsqueeze(-1).expand(-1, -1, d))
        cs = torch.cumsum(h.float(), 1)
        cs = torch.cat([torch.zeros_like(cs[:, :1]), cs], 1)        # cs[t] = sum h[:t]
        hi = torch.gather(cs, 1, (markers + 1).unsqueeze(-1).expand(-1, -1, d))
        lo = torch.gather(cs, 1, starts.unsqueeze(-1).expand(-1, -1, d))
        n = (markers - starts + 1).clamp(min=1).unsqueeze(-1).float()
        return ((hi - lo) / n).to(h.dtype)

    def forward(self, input_ids, attention_mask, markers, starts, marker_mask):
        v = self.sentence_vectors(input_ids, attention_mask, markers, starts)
        return self.head(v, marker_mask), v


def collate(windows, idx, pad_id: int, device=None, labels_override=None) -> Dict[str, torch.Tensor]:
    """Right-padded batch of windows. Padded marker slots point at position 0 with
    marker_mask False and label -100. `labels_override` = {window_idx: labels array}."""
    B = len(idx)
    L = max(len(windows[i]["tokens"]) for i in idx)
    M = max(len(windows[i]["markers"]) for i in idx)
    x = torch.full((B, L), pad_id, dtype=torch.long)
    am = torch.zeros((B, L), dtype=torch.long)
    mk = torch.zeros((B, M), dtype=torch.long)
    st = torch.zeros((B, M), dtype=torch.long)
    mm = torch.zeros((B, M), dtype=torch.bool)
    y = torch.full((B, M), -100, dtype=torch.long)
    for r, i in enumerate(idx):
        w = windows[i]
        n, m = len(w["tokens"]), len(w["markers"])
        x[r, :n] = torch.from_numpy(w["tokens"].astype("int64"))
        am[r, :n] = 1
        mk[r, :m] = torch.from_numpy(w["markers"])
        st[r, :m] = torch.from_numpy(w["starts"])
        mm[r, :m] = True
        lab = labels_override.get(i, w["labels"]) if labels_override else w["labels"]
        y[r, :m] = torch.from_numpy(lab.astype("int64"))
    out = {"input_ids": x, "attention_mask": am, "markers": mk, "starts": st,
           "marker_mask": mm, "labels": y}
    if device is not None:
        out = {k: v.to(device) for k, v in out.items()}
    return out


def build_rr_model(phase1_dir: str, n_labels: int, attn_mode: str = "bidir",
                   head: str = "linear", layer_ids: Optional[List[int]] = None,
                   dropout: float = 0.1, pool: str = "marker") -> Tuple[RRModel, Dict]:
    lm, cfg = load_phase1_lm(phase1_dir)
    trunk = RRTrunk(lm, attn_mode, layer_ids)
    del lm                                          # unused blocks / lm_head are freed here
    return RRModel(trunk, n_labels, head, dropout, pool), cfg


# =============================================================================================
# checkpoint state: trainable parameters + PEFT state buffers (AdaLoRA masks / importance)
# =============================================================================================
def _is_state_buffer(name: str) -> bool:
    return any(t in name for t in STATE_BUFFER_TAGS)


def trainable_state_dict(model: nn.Module, half: bool = False) -> Dict[str, torch.Tensor]:
    sd = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
    sd.update({n: b.detach().cpu() for n, b in model.named_buffers() if _is_state_buffer(n)})
    if half:
        sd = {n: (t.half() if t.is_floating_point() and not _is_state_buffer(n) else t)
              for n, t in sd.items()}
    return sd


def load_trainable_state(model: nn.Module, sd: Dict[str, torch.Tensor]) -> None:
    """Every key must exist; every trainable parameter / state buffer must be present."""
    params = dict(model.named_parameters())
    bufs = {n: b for n, b in model.named_buffers() if _is_state_buffer(n)}
    unknown = [n for n in sd if n not in params and n not in bufs]
    if unknown:
        raise KeyError(f"checkpoint has unknown tensors: {unknown[:5]}")
    with torch.no_grad():
        for n, t in sd.items():
            dst = params.get(n, bufs.get(n))
            dst.copy_(t.to(dst.dtype))
    expected = {n for n, p in params.items() if p.requires_grad} | set(bufs)
    absent = expected - set(sd)
    if absent:
        raise KeyError(f"checkpoint lacks trainable tensors: {sorted(absent)[:5]}")


def rebuild_trained_model(run_config: Dict, best_pt: str, phase1_dirs: Dict[str, str],
                          device="cpu") -> RRModel:
    """Recreate the exact module structure of a finished run and load its best.pt."""
    from rr_peft import get_technique
    mid = run_config["model"]
    p1 = phase1_dirs.get(mid) or run_config.get("phase1_dir")
    if not p1 or not os.path.exists(os.path.join(p1, "model.pt")):
        raise FileNotFoundError(f"Phase-1 weights for {mid} not found")
    model, _ = build_rr_model(p1, run_config["n_labels"], run_config["attn_mode"],
                              run_config["head"], run_config.get("layer_ids"),
                              run_config.get("dropout", 0.1), run_config.get("pool", "marker"))
    tech = get_technique(run_config["technique"])
    tech.apply(model, run_config, log=lambda *a: None)
    if run_config.get("kd_hidden_weight", 0) and run_config["technique"] == "kd":
        from rr_kd import attach_projection
        attach_projection(model, run_config)
    load_trainable_state(model, torch_load(best_pt))
    return model.to(device).eval()


def model_size(model: nn.Module) -> Dict[str, float]:
    """Parameter counts; quantised (QLoRA) base weights count as parameters of the model."""
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    n_q = sum(int(getattr(m, "n_quantized", 0)) for m in model.modules())
    total = n_all + n_q
    return {"total_params": int(total), "trainable_params": int(n_tr),
            "trainable_pct": 100.0 * n_tr / max(1, total), "quantized_params": int(n_q)}


In [ ]:
%%writefile /kaggle/working/phase3/rr_peft.py
"""
rr_peft.py -- fine-tuning / PEFT techniques for Phase 3, pure PyTorch (no `peft` library).

Every technique has the same interface (class `Technique`):
    apply(model, cfg, log) -> optimizer param groups (the trainable parameters, grouped)
    extra_loss(model)      -> regulariser added to the loss (AdaLoRA) or None
    after_backward(model, step, total_steps)   (AdaLoRA importance + budget; grads unscaled)
    merge(model) -> bool   folds the update into plain layers where meaningful
    state_dict_trainable(model), count(model), final_report(model)
`apply` freezes everything, inserts the modules, unfreezes exactly what the method trains and
ASSERTS that the set of trainable parameter names equals the expected set.

  lorma    LoRMA+  y = (I + s*B*A) W0 x     -- Phase-2 code (ner_peft), compulsory
           Bihany, Patel & Modi (2025), "LoRMA: Low-Rank Multiplicative Adaptation for LLMs",
           arXiv:2506.07621.
  lora     LoRA    y = W0 x + (a/r) B A x  -- Phase-2 code (ner_peft)
           Hu et al. (2021), "LoRA: Low-Rank Adaptation of Large Language Models",
           arXiv:2106.09685.
  rslora   LoRA with the rank-stabilised scale a/sqrt(r) (shares the LoRA code; run as its own
           experiment with a higher rank, where the scale matters).
           Kalajdzievski (2023), "A Rank Stabilization Scaling Factor for Fine-Tuning with LoRA",
           arXiv:2312.03732.
  adalora  dW = P diag(lambda) Q, orthogonality regulariser ||P^T P - I||^2 + ||Q Q^T - I||^2,
           sensitivity importance |w * dL/dw| smoothed by EMA (+ uncertainty), triplet score
           s(lambda_i) + mean s(P_:i) + mean s(Q_i:), pruning to a cubic budget schedule
           (warm-up, then decay to the target budget, then a final phase with a fixed mask).
           Zhang et al. (2023), "Adaptive Budget Allocation for Parameter-Efficient
           Fine-Tuning", ICLR 2023, arXiv:2303.10512.
  qlora    frozen base linears stored as block-wise NF4 (block 64, packed uint8 + absmax
           scales, optional double quantisation of the scales: int8, block 256, mean-centred),
           de-quantised on the fly in the forward pass; trainable LoRA on top. Simulated
           without bitsandbytes. Dettmers et al. (2023), "QLoRA: Efficient Finetuning of
           Quantized LLMs", NeurIPS 2023, arXiv:2305.14314.
  ia3      learned scaling vectors (init 1) on keys, values and the FFN intermediate activation
           (input of the down projection, i.e. the SwiGLU product for gated FFNs).
           Liu et al. (2022), "Few-Shot Parameter-Efficient Fine-Tuning is Better and Cheaper
           than In-Context Learning", NeurIPS 2022, arXiv:2205.05638.
  prefix   per-layer key/value prefixes (length 20), via a reparameterisation MLP that is
           dropped after training (`merge`). Li & Liang (2021), "Prefix-Tuning: Optimizing
           Continuous Prompts for Generation", ACL 2021, arXiv:2101.00190.
  prompt   20 soft input embeddings, initialised from random vocabulary embeddings.
           Lester, Al-Rfou & Constant (2021), "The Power of Scale for Parameter-Efficient Prompt
           Tuning", EMNLP 2021, arXiv:2104.08691.
  full_ft  every trunk parameter (reference only; also the KD teacher)
  kd       every parameter of the layer-pruned student (see rr_kd.py)
The sentence head (head.*) is always trainable.
"""
import math
from typing import Dict, List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

from ner_peft import TARGETS, LoRALinear, LoRMALinear, _get, _groups, _set
from ner_peft import apply_technique as phase2_apply_technique
from rr_models import model_size, trainable_state_dict

# IA3 sites per architecture family: (path, kind); kind "kv_fused" scales the k and v slices of
# a fused QKV output, "out" scales a whole projection output, "in" scales a projection input.
IA3_SITES = {
    "gpt2": [("attn.qkv", "kv_fused"), ("mlp.down_proj", "in")],
    "llama": [("attn.k_proj", "out"), ("attn.v_proj", "out"), ("mlp.down_proj", "in")],
    "bloom": [("self_attention.query_key_value", "kv_fused"), ("mlp.down_proj", "in")],
}

# NF4 code book (QLoRA, Appendix E / bitsandbytes `create_normal_map`), checked in selftest
NF4_CODE = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
            -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
            0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
            0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0]


def nf4_codebook_from_normal(offset: float = 0.9677083) -> torch.Tensor:
    """Re-derive the NF4 code book from normal quantiles (8 positive, 7 negative, exact 0)."""
    def ppf(p):
        return torch.special.ndtri(torch.as_tensor(p, dtype=torch.float64))
    pos = ppf(torch.linspace(offset, 0.5, 9, dtype=torch.float64)[:-1])
    neg = -ppf(torch.linspace(offset, 0.5, 8, dtype=torch.float64)[:-1])
    v = torch.cat([pos, torch.zeros(1, dtype=torch.float64), neg]).sort().values
    return (v / v.abs().max()).float()


def work_dtype(x: torch.Tensor) -> torch.dtype:
    if x.is_cuda and torch.is_autocast_enabled():
        try:
            return torch.get_autocast_dtype("cuda")
        except (AttributeError, TypeError):
            return torch.get_autocast_gpu_dtype()
    return x.dtype


# =============================================================================================
# modules
# =============================================================================================
class RSLoRALinear(LoRALinear):
    """LoRA with the rank-stabilised scale alpha / sqrt(r)."""

    def __init__(self, base: nn.Linear, r: int = 32, alpha: float = 16, dropout: float = 0.05):
        super().__init__(base, r, alpha, dropout)
        self.scale = alpha / math.sqrt(r)


def quantize_nf4(w: torch.Tensor, block: int = 64):
    """Flat fp tensor -> (packed uint8 (ceil(N/2) padded), absmax float32 per block, N)."""
    code = torch.tensor(NF4_CODE, dtype=torch.float32, device=w.device)
    n = w.numel()
    flat = w.detach().float().flatten()
    pad = (-n) % block
    if pad:
        flat = F.pad(flat, (0, pad))
    blocks = flat.view(-1, block)
    absmax = blocks.abs().amax(1).clamp(min=1e-12)
    normed = (blocks / absmax[:, None]).flatten()
    mids = (code[1:] + code[:-1]) / 2
    idx = torch.bucketize(normed, mids).to(torch.uint8)       # nearest code
    if idx.numel() % 2:
        idx = F.pad(idx, (0, 1))
    packed = idx[0::2] | (idx[1::2] << 4)
    return packed, absmax, n


def dequantize_nf4(packed: torch.Tensor, absmax: torch.Tensor, n: int, block: int,
                   code: torch.Tensor, dtype=torch.float32) -> torch.Tensor:
    idx = torch.stack([packed & 15, packed >> 4], -1).flatten()[: absmax.numel() * block]
    vals = code.to(dtype)[idx.long()].view(-1, block) * absmax.to(dtype)[:, None]
    return vals.flatten()[:n]


def quantize_absmax_int8(x: torch.Tensor, block: int = 256):
    """Double quantisation of the NF4 scales: mean-centred, int8, per-block absmax."""
    mean = x.mean()
    c = x - mean
    pad = (-c.numel()) % block
    cp = F.pad(c, (0, pad)).view(-1, block)
    s = (cp.abs().amax(1) / 127).clamp(min=1e-12)
    q = torch.round(cp / s[:, None]).clamp(-127, 127).to(torch.int8)
    return q.flatten()[: x.numel()], s, mean


def dequantize_absmax_int8(q, s, mean, block: int = 256):
    pad = (-q.numel()) % block
    qp = F.pad(q.float(), (0, pad)).view(-1, block)
    return (qp * s[:, None]).flatten()[: q.numel()] + mean


class NF4Linear(nn.Module):
    """Frozen linear whose weight is stored as block-wise NF4 and de-quantised per forward."""

    def __init__(self, base: nn.Linear, block: int = 64, double_quant: bool = True,
                 dq_block: int = 256):
        super().__init__()
        self.in_features, self.out_features = base.in_features, base.out_features
        self.block, self.double_quant, self.dq_block = block, double_quant, dq_block
        packed, absmax, n = quantize_nf4(base.weight, block)
        self.n_quantized = n
        self.register_buffer("qweight", packed)
        self.register_buffer("code", torch.tensor(NF4_CODE, dtype=torch.float32))
        if double_quant:
            q, s, mean = quantize_absmax_int8(absmax, dq_block)
            self.register_buffer("qabs", q)
            self.register_buffer("qabs_scale", s)
            self.register_buffer("qabs_mean", mean.reshape(1))
        else:
            self.register_buffer("absmax", absmax)
        self.bias = None
        if base.bias is not None:
            self.bias = nn.Parameter(base.bias.detach().clone(), requires_grad=False)

    def scales(self) -> torch.Tensor:
        if self.double_quant:
            return dequantize_absmax_int8(self.qabs, self.qabs_scale, self.qabs_mean[0],
                                          self.dq_block)
        return self.absmax

    def dequantize(self, dtype=torch.float32) -> torch.Tensor:
        return dequantize_nf4(self.qweight, self.scales(), self.n_quantized, self.block,
                              self.code, dtype).view(self.out_features, self.in_features)

    def forward(self, x):
        dt = work_dtype(x)
        b = self.bias.to(dt) if self.bias is not None else None
        return F.linear(x.to(dt), self.dequantize(dt), b)

    def storage_bytes(self) -> int:
        n = self.qweight.numel()
        if self.double_quant:
            n += self.qabs.numel() + 4 * (self.qabs_scale.numel() + 1)
        else:
            n += 4 * self.absmax.numel()
        return int(n)


class IA3Out(nn.Module):
    """y = base(x) * l, where l scales chosen output slices (others stay 1)."""

    def __init__(self, base: nn.Linear, slices):
        super().__init__()
        self.base = base
        self.slices = list(slices)
        self.ia3_l = nn.ParameterList([nn.Parameter(torch.ones(e - s)) for s, e in self.slices])

    def scale_vector(self, dtype, device) -> torch.Tensor:
        v = torch.ones(self.base.out_features, dtype=dtype, device=device)
        for (s, e), p in zip(self.slices, self.ia3_l):
            v = torch.cat([v[:s], p.to(dtype), v[e:]])
        return v

    def forward(self, x):
        y = self.base(x)
        return y * self.scale_vector(y.dtype, y.device)

    @torch.no_grad()
    def merged(self) -> nn.Linear:
        l = self.scale_vector(torch.float32, self.base.weight.device)
        lin = _clone_linear(self.base)
        lin.weight.mul_(l[:, None].to(lin.weight.dtype))
        if lin.bias is not None:
            lin.bias.mul_(l.to(lin.bias.dtype))
        return lin


class IA3In(nn.Module):
    """y = base(x * l): scales the FFN intermediate activation before the down projection."""

    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        self.ia3_l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.ia3_l.to(x.dtype))

    @torch.no_grad()
    def merged(self) -> nn.Linear:
        lin = _clone_linear(self.base)
        lin.weight.mul_(self.ia3_l[None, :].to(lin.weight.dtype))
        return lin


class AdaLoRALinear(nn.Module):
    """y = W0 x + b + (alpha/r) * P diag(lambda * mask) Q dropout(x); lambda = 0 at init."""

    def __init__(self, base: nn.Linear, r: int = 12, alpha: float = 32, dropout: float = 0.05):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        self.ada_P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)   # left vectors
        self.ada_E = nn.Parameter(torch.zeros(r))                             # singular values
        self.ada_Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)    # right vectors
        self.register_buffer("ada_mask", torch.ones(r))
        for n in ("P", "E", "Q"):
            self.register_buffer(f"ada_ipt_{n}", torch.zeros_like(getattr(self, f"ada_{n}")))
            self.register_buffer(f"ada_unc_{n}", torch.zeros_like(getattr(self, f"ada_{n}")))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        y = self.base(x)
        lam = (self.ada_E * self.ada_mask).to(x.dtype)
        u = F.linear(self.drop(x), self.ada_Q.to(x.dtype)) * lam
        return y + self.scale * F.linear(u, self.ada_P.to(x.dtype))

    @torch.no_grad()
    def merged_weight(self):
        d = self.ada_P @ torch.diag(self.ada_E * self.ada_mask) @ self.ada_Q
        return self.base.weight + self.scale * d.to(self.base.weight.dtype)

    def orth_reg(self) -> torch.Tensor:
        P, Q = self.ada_P.float(), self.ada_Q.float()
        I = torch.eye(self.r, device=P.device)
        return ((P.t() @ P - I) ** 2).sum() + ((Q @ Q.t() - I) ** 2).sum()


class PrefixEncoder(nn.Module):
    """Per-layer key/value prefixes (n_layers, 2, n_kv, length, head_dim)."""

    def __init__(self, n_layers: int, n_kv: int, head_dim: int, d_model: int, length: int = 20,
                 reparam: bool = True, hidden: int = 256, init_emb: Optional[torch.Tensor] = None):
        super().__init__()
        self.n_layers, self.n_kv, self.head_dim, self.length = n_layers, n_kv, head_dim, length
        self.reparam = reparam
        out = n_layers * 2 * n_kv * head_dim
        if reparam:
            emb = init_emb if init_emb is not None else torch.randn(length, d_model) * 0.02
            self.prefix_emb = nn.Parameter(emb.detach().clone().float())
            self.prefix_mlp = nn.Sequential(nn.Linear(d_model, hidden), nn.Tanh(),
                                            nn.Linear(hidden, out))
            self.prefix_kv = None
        else:
            self.prefix_kv = nn.Parameter(torch.randn(length, out) * 0.02)

    def table(self) -> torch.Tensor:
        return self.prefix_mlp(self.prefix_emb) if self.reparam else self.prefix_kv

    def forward(self, B: int, dtype, device):
        t = self.table().view(self.length, self.n_layers, 2, self.n_kv, self.head_dim)
        t = t.permute(1, 2, 3, 0, 4).to(dtype)          # (L, 2, n_kv, length, hd)
        return [(t[i, 0].unsqueeze(0).expand(B, -1, -1, -1),
                 t[i, 1].unsqueeze(0).expand(B, -1, -1, -1)) for i in range(self.n_layers)]

    @torch.no_grad()
    def materialize(self) -> None:
        """Drop the reparameterisation MLP: keep only the computed prefix table."""
        if not self.reparam:
            return
        kv = self.table().detach().clone()
        del self.prefix_mlp
        del self.prefix_emb
        self.reparam = False
        self.prefix_kv = nn.Parameter(kv, requires_grad=False)


def _clone_linear(base: nn.Linear) -> nn.Linear:
    lin = nn.Linear(base.in_features, base.out_features, bias=base.bias is not None)
    lin = lin.to(device=base.weight.device, dtype=base.weight.dtype)
    with torch.no_grad():
        lin.weight.copy_(base.weight)
        if base.bias is not None:
            lin.bias.copy_(base.bias)
    lin.weight.requires_grad = False
    if lin.bias is not None:
        lin.bias.requires_grad = False
    return lin


def _replace_modules(model: nn.Module, cls, make) -> int:
    """Replace every submodule of type `cls` by make(module). Returns the count."""
    hits = [(n, m) for n, m in model.named_modules() if isinstance(m, cls)]
    for n, m in hits:
        _set(model, n, make(m))
    return len(hits)


# =============================================================================================
# techniques
# =============================================================================================
class Technique:
    name = "base"

    def apply(self, model: nn.Module, cfg: Dict, log=print) -> List[Dict]:
        raise NotImplementedError

    def extra_loss(self, model: nn.Module) -> Optional[torch.Tensor]:
        return None

    def after_backward(self, model: nn.Module, step: int, total_steps: int) -> None:
        return None

    def merge(self, model: nn.Module) -> bool:
        return False

    def state_dict_trainable(self, model: nn.Module) -> Dict[str, torch.Tensor]:
        return trainable_state_dict(model)

    def count(self, model: nn.Module) -> Dict[str, float]:
        return model_size(model)

    def final_report(self, model: nn.Module) -> Dict:
        return {}

    def selectable(self, step: int, total_steps: int) -> bool:
        """May a checkpoint taken after `step` completed steps become best.pt? (AdaLoRA: only
        once the final budget is in place.) Early stopping is also held back until then."""
        return True

    # ---------------------------------------------------------------- shared tail of apply()
    @staticmethod
    def _freeze(model):
        for p in model.parameters():
            p.requires_grad = False

    def _finish(self, model, groups, expected: set, cfg, log) -> List[Dict]:
        head = [(n, p) for n, p in model.named_parameters() if n.startswith("head.")
                or n.startswith("kd_proj.")]
        for _, p in head:
            p.requires_grad = True
        expected = set(expected) | {n for n, _ in head}
        groups = list(groups) + _groups(head, float(cfg.get("head_lr", 1e-3)),
                                        float(cfg.get("weight_decay", 0.01)), "head")
        actual = {n for n, p in model.named_parameters() if p.requires_grad}
        if actual != expected:
            raise AssertionError(f"[{self.name}] trainable-parameter mismatch: unexpected "
                                 f"{sorted(actual - expected)[:5]}, missing "
                                 f"{sorted(expected - actual)[:5]}")
        c = self.count(model)
        log(f"   [{self.name}] trainable {c['trainable_params']:,} / {c['total_params']:,} "
            f"params ({c['trainable_pct']:.3f}%) in {len(actual)} tensors")
        return groups


class FullFT(Technique):
    name = "full_ft"

    def apply(self, model, cfg, log=print):
        groups = phase2_apply_technique(model, "full_ft", cfg, log=lambda *a: None)
        expected = {n for n, p in model.named_parameters() if p.requires_grad}
        return self._finish(model, groups, expected, cfg, log)


class KDStudent(FullFT):
    name = "kd"


class Phase2LowRank(Technique):
    """LoRA / LoRMA+: the Phase-2 implementation (ner_peft.apply_technique), wrapped."""

    def __init__(self, which: str):
        self.name = which

    def apply(self, model, cfg, log=print):
        groups = phase2_apply_technique(model, self.name, cfg, log=lambda *a: None)
        expected = {n for n, p in model.named_parameters() if p.requires_grad}
        assert expected and all((".lora_" in n) or (".lorma_" in n) for n in expected)
        return self._finish(model, groups, expected, cfg, log)

    def merge(self, model):
        cls = LoRALinear if self.name == "lora" else LoRMALinear

        def make(m):
            lin = _clone_linear(m.base)
            with torch.no_grad():
                lin.weight.copy_(m.merged_weight())
            return lin
        return _replace_modules(model, cls, make) > 0


class RSLoRA(Technique):
    name = "rslora"

    def apply(self, model, cfg, log=print):
        self._freeze(model)
        trunk = model.trunk
        r, alpha = int(cfg.get("rslora_r", 32)), float(cfg.get("rslora_alpha", 16))
        for blk in trunk.blocks:
            for path in TARGETS[trunk.kind]:
                _set(blk, path, RSLoRALinear(_get(blk, path), r, alpha,
                                             float(cfg.get("lora_dropout", 0.05))))
        named = [(n, p) for n, p in model.named_parameters() if ".lora_" in n]
        for _, p in named:
            p.requires_grad = True
        groups = _groups(named, float(cfg["lr"]), float(cfg.get("weight_decay", 0.01)), "rslora")
        return self._finish(model, groups, {n for n, _ in named}, cfg, log)

    def merge(self, model):
        def make(m):
            lin = _clone_linear(m.base)
            with torch.no_grad():
                lin.weight.copy_(m.merged_weight())
            return lin
        return _replace_modules(model, RSLoRALinear, make) > 0


class QLoRA(Technique):
    name = "qlora"

    def apply(self, model, cfg, log=print):
        self._freeze(model)
        trunk = model.trunk
        r, alpha = int(cfg.get("qlora_r", 16)), float(cfg.get("qlora_alpha", 32))
        block, dq = int(cfg.get("qlora_block", 64)), bool(cfg.get("qlora_double_quant", True))
        fp16_bytes = nf4_bytes = 0
        for blk in trunk.blocks:
            for path in TARGETS[trunk.kind]:
                base = _get(blk, path)
                q = NF4Linear(base, block, dq)
                fp16_bytes += 2 * base.weight.numel()
                nf4_bytes += q.storage_bytes()
                _set(blk, path, LoRALinear(q, r, alpha, float(cfg.get("lora_dropout", 0.05))))
        self.memory = {"base_linear_fp16_mb": fp16_bytes / 2**20, "base_linear_nf4_mb":
                       nf4_bytes / 2**20, "ratio": fp16_bytes / max(1, nf4_bytes),
                       "block": block, "double_quant": dq}
        log(f"   [qlora] base linears: {self.memory['base_linear_fp16_mb']:.1f} MB fp16 -> "
            f"{self.memory['base_linear_nf4_mb']:.1f} MB NF4 ({self.memory['ratio']:.2f}x)")
        named = [(n, p) for n, p in model.named_parameters() if ".lora_" in n]
        for _, p in named:
            p.requires_grad = True
        groups = _groups(named, float(cfg["lr"]), float(cfg.get("weight_decay", 0.01)), "qlora")
        return self._finish(model, groups, {n for n, _ in named}, cfg, log)

    def final_report(self, model):
        return {"qlora_memory": getattr(self, "memory", {})}


class IA3(Technique):
    name = "ia3"

    def apply(self, model, cfg, log=print):
        self._freeze(model)
        trunk = model.trunk
        d = trunk.hidden_size
        for blk in trunk.blocks:
            for path, kind in IA3_SITES[trunk.kind]:
                base = _get(blk, path)
                if kind == "kv_fused":
                    assert base.out_features == 3 * d
                    new = IA3Out(base, [(d, 2 * d), (2 * d, 3 * d)])
                elif kind == "out":
                    new = IA3Out(base, [(0, base.out_features)])
                else:
                    new = IA3In(base)
                _set(blk, path, new)
        named = [(n, p) for n, p in model.named_parameters() if ".ia3_l" in n]
        for _, p in named:
            p.requires_grad = True
        groups = _groups(named, float(cfg["lr"]), 0.0, "ia3")
        return self._finish(model, groups, {n for n, _ in named}, cfg, log)

    def merge(self, model):
        n = _replace_modules(model, IA3Out, lambda m: m.merged())
        return (n + _replace_modules(model, IA3In, lambda m: m.merged())) > 0


class AdaLoRA(Technique):
    name = "adalora"

    def apply(self, model, cfg, log=print):
        self._freeze(model)
        trunk = model.trunk
        self.r = int(cfg.get("adalora_init_r", 12))
        self.target_r = int(cfg.get("adalora_target_r", 8))
        alpha = float(cfg.get("adalora_alpha", 32))
        self.tinit_frac = float(cfg.get("adalora_tinit_frac", 0.1))
        self.tfinal_frac = float(cfg.get("adalora_tfinal_frac", 0.2))
        self.delta_t = int(cfg.get("adalora_delta_t", 10))
        self.beta1 = float(cfg.get("adalora_beta1", 0.85))
        self.beta2 = float(cfg.get("adalora_beta2", 0.85))
        self.orth_w = float(cfg.get("adalora_orth", 0.1))
        for blk in trunk.blocks:
            for path in TARGETS[trunk.kind]:
                _set(blk, path, AdaLoRALinear(_get(blk, path), self.r, alpha,
                                              float(cfg.get("lora_dropout", 0.05))))
        self.modules_ = [(n, m) for n, m in model.named_modules() if isinstance(m, AdaLoRALinear)]
        self.b0 = self.r * len(self.modules_)
        self.bT = self.target_r * len(self.modules_)
        named = [(n, p) for n, p in model.named_parameters() if ".ada_" in n]
        for _, p in named:
            p.requires_grad = True
        groups = _groups(named, float(cfg["lr"]), float(cfg.get("weight_decay", 0.01)), "adalora")
        log(f"   [adalora] {len(self.modules_)} matrices, rank {self.r} -> target "
            f"{self.target_r} (budget {self.b0} -> {self.bT} singular values)")
        return self._finish(model, groups, {n for n, _ in named}, cfg, log)

    # ---------------------------------------------------------------- regulariser
    def extra_loss(self, model):
        if not self.modules_ or self.orth_w <= 0:
            return None
        reg = sum(m.orth_reg() for _, m in self.modules_) / len(self.modules_)
        return self.orth_w * reg

    # ---------------------------------------------------------------- budget schedule
    def phases(self, total: int):
        """(t_init, t_final_start): no pruning before t_init; the final budget is set at
        t_final_start (always <= the last step, so short runs reach it too)."""
        tf_start = max(0, min(total - int(self.tfinal_frac * total), total - 1))
        return min(int(self.tinit_frac * total), tf_start), tf_start

    def budget(self, step: int, total: int) -> int:
        ti, tf_start = self.phases(total)
        if step < ti:
            return self.b0
        if step >= tf_start:
            return self.bT
        mul = 1.0 - (step - ti) / max(1, tf_start - ti)
        return int(self.bT + (self.b0 - self.bT) * mul ** 3)

    def selectable(self, step, total_steps):
        return step > self.phases(total_steps)[1]

    @torch.no_grad()
    def _update_importance(self):
        for _, m in self.modules_:
            for n in ("P", "E", "Q"):
                p = getattr(m, f"ada_{n}")
                if p.grad is None:
                    continue
                ipt = (p * p.grad).abs()
                ema = getattr(m, f"ada_ipt_{n}")
                unc = getattr(m, f"ada_unc_{n}")
                ema.mul_(self.beta1).add_((1 - self.beta1) * ipt)
                unc.mul_(self.beta2).add_((1 - self.beta2) * (ipt - ema).abs())

    @torch.no_grad()
    def _scores(self) -> List[torch.Tensor]:
        out = []
        for _, m in self.modules_:
            s = lambda n: getattr(m, f"ada_ipt_{n}") * getattr(m, f"ada_unc_{n}")  # noqa: E731
            out.append(s("E") + s("P").mean(0) + s("Q").mean(1))
        return out

    @torch.no_grad()
    def after_backward(self, model, step, total_steps):
        grads_ok = all(p.grad is None or torch.isfinite(p.grad).all()
                       for _, m in self.modules_ for p in (m.ada_P, m.ada_E, m.ada_Q))
        if not grads_ok:
            return
        self._update_importance()
        ti, tf_start = self.phases(total_steps)
        if step < ti or step > tf_start:
            return                                      # warm-up / final phase: mask fixed
        if step != tf_start and (step - ti) % self.delta_t != 0:
            return
        b = self.budget(step, total_steps)
        scores = self._scores()
        flat = torch.cat(scores)
        if b >= flat.numel():
            return
        thr = torch.kthvalue(flat.float().cpu(), flat.numel() - b).values.item()
        for (_, m), s in zip(self.modules_, scores):
            keep = (s > thr).float()
            m.ada_mask.copy_(keep)
            m.ada_E.mul_(keep)

    def merge(self, model):
        def make(m):
            lin = _clone_linear(m.base)
            with torch.no_grad():
                lin.weight.copy_(m.merged_weight())
            return lin
        return _replace_modules(model, AdaLoRALinear, make) > 0

    def final_report(self, model):
        return {**adalora_ranks(model), "adalora_budget_initial": self.b0,
                "adalora_budget_target": self.bT}


def adalora_ranks(model: nn.Module) -> Dict:
    """Rank kept per adapted matrix and per layer (from the saved masks)."""
    ranks = {n.replace("trunk.blocks.", "L"): int(m.ada_mask.sum().item())
             for n, m in model.named_modules() if isinstance(m, AdaLoRALinear)}
    per_layer: Dict[str, int] = {}
    for n, r in ranks.items():
        layer = n.split(".")[0]
        per_layer[layer] = per_layer.get(layer, 0) + r
    return {"adalora_rank_per_matrix": ranks, "adalora_rank_per_layer": per_layer,
            "adalora_budget_final": int(sum(ranks.values()))}


class PrefixTuning(Technique):
    name = "prefix"

    def apply(self, model, cfg, log=print):
        self._freeze(model)
        trunk = model.trunk
        P = int(cfg.get("prefix_len", 20))
        g = torch.Generator().manual_seed(int(cfg.get("seed", 42)))
        idx = torch.randint(259, trunk.embed.weight.shape[0], (P,), generator=g)
        trunk.prefix = PrefixEncoder(len(trunk.blocks), trunk.n_kv, trunk.head_dim,
                                     trunk.hidden_size, P, bool(cfg.get("prefix_reparam", True)),
                                     int(cfg.get("prefix_hidden", 256)),
                                     trunk.embed.weight[idx].detach().float())
        named = [(n, p) for n, p in model.named_parameters() if n.startswith("trunk.prefix.")]
        for _, p in named:
            p.requires_grad = True
        groups = _groups(named, float(cfg["lr"]), 0.0, "prefix")
        return self._finish(model, groups, {n for n, _ in named}, cfg, log)

    def merge(self, model):
        if model.trunk.prefix is None or not model.trunk.prefix.reparam:
            return False
        model.trunk.prefix.materialize()
        return True


class PromptTuning(Technique):
    name = "prompt"

    def apply(self, model, cfg, log=print):
        self._freeze(model)
        trunk = model.trunk
        P = int(cfg.get("prompt_len", 20))
        g = torch.Generator().manual_seed(int(cfg.get("seed", 42)))
        idx = torch.randint(259, trunk.embed.weight.shape[0], (P,), generator=g)
        trunk.prompt = nn.Parameter(trunk.embed.weight[idx].detach().clone().float())
        named = [(n, p) for n, p in model.named_parameters() if n == "trunk.prompt"]
        for _, p in named:
            p.requires_grad = True
        groups = _groups(named, float(cfg["lr"]), 0.0, "prompt")
        return self._finish(model, groups, {n for n, _ in named}, cfg, log)


def get_technique(name: str) -> Technique:
    if name in ("lora", "lorma"):
        return Phase2LowRank(name)
    table = {"full_ft": FullFT, "kd": KDStudent, "rslora": RSLoRA, "qlora": QLoRA, "ia3": IA3,
             "adalora": AdaLoRA, "prefix": PrefixTuning, "prompt": PromptTuning}
    if name not in table:
        raise ValueError(f"unknown technique {name!r}")
    return table[name]()


def n_virtual_tokens(cfg: Dict) -> int:
    """Context positions a technique consumes (prompt tuning only)."""
    return int(cfg.get("prompt_len", 20)) if cfg.get("technique") == "prompt" else 0


In [ ]:
%%writefile /kaggle/working/phase3/rr_kd.py
"""
rr_kd.py -- knowledge distillation for Phase 3 (experiment E6).

Teacher: by default SELF-CONTAINED: the E6 notebook first trains a full fine-tuned teacher per
Phase-1 model (same recipe as the E0 reference, job `teacher_<M>`), then distils it.
`--teacher-dir` instead loads existing teachers: either a folder with per-model run folders
(<dir>/M?_*/checkpoints/best.pt, e.g. phase3_out/E0_full_ft) -> the student of model M learns
from teacher M; or one BEST / run folder (best.pt + run_config.json) -> every student learns
from that teacher. Missing teacher -> exit code 2 (missing dependency).

Student: the same Phase-1 model pruned to every second layer (M1/M4: layers 0,2,..,12;
M2/M3: 0,2,..,10), same width, initialised from those layers' Phase-1 weights, all parameters
+ a fresh head trained. Asserted: student params < teacher params (both printed). A pruned
student is a Phase-3 artifact, so the 50M lower bound of the brief does not apply; its size is
reported honestly.

Loss (Hinton, Vinyals & Dean (2015), "Distilling the Knowledge in a Neural Network",
arXiv:1503.02531):
    L = alpha * CE(student, gold) + (1 - alpha) * T^2 * KL(teacher_T || student_T)
alpha 0.5, T 2.0. Optional (off by default, `kd_hidden_weight` > 0): MSE between the student's
and the teacher's sentence vectors (through a learned projection when the widths differ).
Teacher outputs are computed once per training window and cached on disk (fp16).

Inputs : teacher run(s), student run_config
Outputs: teacher_cache.pt, KD loss, the teacher/student comparison (rr_predict writes it)
"""
import os
from typing import Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from rr_common import MODELS, read_json
from rr_models import rebuild_trained_model


def resolve_teacher(teacher_dir: str, mid: str) -> Optional[Tuple[str, str]]:
    """(run_config.json, best.pt) of the teacher for model `mid`, or None."""
    if not teacher_dir or not os.path.isdir(teacher_dir):
        return None
    for cfg_rel, pt_rel in (("logs/run_config.json", "checkpoints/best.pt"),
                            ("run_config.json", "best.pt")):
        sub = os.path.join(teacher_dir, MODELS[mid]["folder"])
        for base in (sub, teacher_dir):
            c, p = os.path.join(base, cfg_rel), os.path.join(base, pt_rel)
            if os.path.exists(c) and os.path.exists(p):
                return c, p
    return None


def load_teacher(cfg: Dict, device, phase1_dirs: Dict[str, str]) -> Tuple[nn.Module, Dict]:
    found = resolve_teacher(cfg["teacher_dir"], cfg["model"])
    if not found:
        raise FileNotFoundError(f"KD teacher for {cfg['model']} not found in {cfg['teacher_dir']}")
    tcfg = read_json(found[0])
    p1 = dict(phase1_dirs)
    if tcfg.get("phase1_dir"):
        p1.setdefault(tcfg["model"], tcfg["phase1_dir"])
    teacher = rebuild_trained_model(tcfg, found[1], p1, device)
    for p in teacher.parameters():
        p.requires_grad = False
    return teacher.eval(), tcfg


def attach_projection(model: nn.Module, cfg: Dict) -> None:
    """Learned projection student-width -> teacher-width (only when the widths differ)."""
    d_s = model.trunk.hidden_size
    d_t = int(cfg.get("teacher_hidden", d_s))
    if d_t != d_s and model.kd_proj is None:
        model.kd_proj = nn.Linear(d_s, d_t)


def kd_loss(s_logits: torch.Tensor, t_logits: torch.Tensor, labels: torch.Tensor, hard_fn,
            alpha: float = 0.5, T: float = 2.0, s_vec: Optional[torch.Tensor] = None,
            t_vec: Optional[torch.Tensor] = None, hidden_w: float = 0.0,
            proj: Optional[nn.Module] = None) -> torch.Tensor:
    """s_logits / t_logits (N, C) for valid sentences, labels (N,)."""
    s, t = s_logits.float(), t_logits.float()
    hard = hard_fn(s, labels)
    soft = F.kl_div(F.log_softmax(s / T, -1), F.log_softmax(t / T, -1), log_target=True,
                    reduction="batchmean") * (T * T)
    loss = alpha * hard + (1 - alpha) * soft
    if hidden_w > 0 and s_vec is not None and t_vec is not None:
        sv = proj(s_vec.float()) if proj is not None else s_vec.float()
        loss = loss + hidden_w * F.mse_loss(sv, t_vec.float())
    return loss


@torch.no_grad()
def build_teacher_cache(teacher: nn.Module, windows, cfg: Dict, meta: Dict, device, amp_ctx,
                        path: str, with_vectors: bool = False, log=print) -> Dict:
    """Teacher logits (and optionally sentence vectors) for every training window, fp16."""
    from rr_predict import window_outputs
    if os.path.exists(path):
        cache = torch.load(path, map_location="cpu")
        if cache.get("n_windows") == len(windows) and (cache.get("vectors") is not None
                                                       or not with_vectors):
            log(f"   [kd] teacher cache reused: {path}")
            return cache
    logits, vecs = window_outputs(teacher, windows, meta, device, amp_ctx,
                                  int(cfg.get("eval_batch", 16)), return_vectors=with_vectors)
    cache = {"n_windows": len(windows), "logits": [l.half() for l in logits],
             "vectors": [v.half() for v in vecs] if with_vectors else None}
    torch.save(cache, path + ".tmp")
    os.replace(path + ".tmp", path)
    log(f"   [kd] teacher outputs cached for {len(windows)} windows -> {path}")
    return cache


In [ ]:
%%writefile /kaggle/working/phase3/rr_train.py
"""
rr_train.py -- one training run for one job (experiment x model [x variant]).

    python rr_train.py --run-dir RUN --mode train                       # 1 GPU / CPU
    python -m torch.distributed.run --nproc_per_node=2 rr_train.py --run-dir RUN --mode train

Same structure as the Phase-2 ner_train.py (DDP via torchrun, fp16 autocast + GradScaler,
30-minute checkpoints, resume, NaN / OOM handling, exit codes), adapted to sentence windows.
Reads RUN/logs/run_config.json. Modes:
  smoke    selftest.job_checks on the REAL model (zero-init == base logits, merge, trainable %,
           prefix visibility, output lengths) + ~30 optimizer steps; writes logs/smoke.json
           (tokens/s, memory, projected minutes)
  overfit  32 labelled sentences, <= 300 steps: train accuracy must exceed 95%
  probe    lr x {1/3, 1, 3} for ~1 epoch each; writes logs/lr_probe.json
  train    the real run: cosine schedule with warm-up, AdamW, clip 1.0, dev pooled Macro-F1 each
           epoch, early stopping with patience, best checkpoint, time guard, resume

Exit codes: 0 ok | 2 time budget reached (checkpoint saved) | 3 out of memory in DDP |
            4 NaN abort | 5 early-fail watchdog | 6 smoke self-check failed | 1 other error
"""
import argparse
import datetime
import json
import math
import os
import random
import sys
import time
import traceback
from typing import Dict, List

import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F

from rr_common import default_input_roots, find_phase1_models, read_json, write_json
from rr_data import IGNORE, build_windows, class_counts, class_weights, load_cache
from rr_models import (build_rr_model, collate, load_trainable_state, model_size, torch_load,
                       trainable_state_dict)
from rr_peft import get_technique
from rr_predict import amp_setup, eval_keys, quick_dev_f1

EXIT_TIME, EXIT_OOM, EXIT_NAN, EXIT_EARLYFAIL, EXIT_SELFCHECK = 2, 3, 4, 5, 6
FULL_PARAM = ("full_ft", "kd")


def is_oom(e: BaseException) -> bool:
    oom_cls = getattr(torch.cuda, "OutOfMemoryError", None)
    return (oom_cls is not None and isinstance(e, oom_cls)) or "out of memory" in str(e).lower()


class Ctx:
    """Distributed / device context (Phase-2 code)."""

    def __init__(self):
        self.world = int(os.environ.get("WORLD_SIZE", "1"))
        self.rank = int(os.environ.get("RANK", "0"))
        self.local = int(os.environ.get("LOCAL_RANK", "0"))
        cuda = torch.cuda.is_available()
        if self.world > 1:
            if cuda:
                torch.cuda.set_device(self.local)
            dist.init_process_group("nccl" if cuda else "gloo",
                                    timeout=datetime.timedelta(minutes=10))
        self.device = torch.device("cuda", self.local) if cuda else torch.device("cpu")
        self.main = self.rank == 0

    def barrier(self):
        if self.world > 1:
            dist.barrier()

    def bcast(self, values):
        t = torch.tensor(values, dtype=torch.float64, device=self.device)
        if self.world > 1:
            dist.broadcast(t, 0)
        return t.tolist()

    def all_min(self, v):
        t = torch.tensor([float(v)], device=self.device)
        if self.world > 1:
            dist.all_reduce(t, op=dist.ReduceOp.MIN)
        return t.item()

    def close(self):
        if self.world > 1 and dist.is_initialized():
            dist.destroy_process_group()


def log(ctx, run_dir, msg, record=None):
    if not ctx.main:
        return
    print(msg, flush=True)
    if record is not None:
        with open(os.path.join(run_dir, "logs", "train_log.jsonl"), "a") as f:
            f.write(json.dumps(record) + "\n")


class EpochOrder:
    """Deterministic permutation per epoch; rank r takes every world-th item after skipping the
    first `skip` (already consumed; resume). Pads by wrap-around so all ranks step equally."""

    def __init__(self, n, seed):
        self.n, self.seed = n, seed

    def indices(self, epoch, rank, world, skip=0):
        perm = np.random.default_rng(self.seed + 1000 * epoch).permutation(self.n)
        rest = perm[skip:]
        if len(rest) == 0:
            return []
        per_rank = math.ceil(len(rest) / world)
        if per_rank * world > len(rest):
            rest = np.concatenate([rest, np.resize(perm, per_rank * world - len(rest))])
        return rest[rank::world].tolist()


class _null:
    def __enter__(self):
        return self

    def __exit__(self, *a):
        return False


# =============================================================================================
# model / optimizer / schedule / loss
# =============================================================================================
def make_model(cfg: Dict, ctx: Ctx, lr_mult: float = 1.0):
    model, _ = build_rr_model(cfg["phase1_dir"], cfg["n_labels"], cfg["attn_mode"], cfg["head"],
                              cfg.get("layer_ids"), cfg.get("dropout", 0.1),
                              cfg.get("pool", "marker"))
    c = dict(cfg)
    c["lr"] = cfg["lr"] * lr_mult
    c["head_lr"] = cfg["head_lr"] * (lr_mult if cfg["technique"] in FULL_PARAM else 1.0)
    tech = get_technique(cfg["technique"])
    if cfg["technique"] == "kd" and cfg.get("kd_hidden_weight", 0):
        from rr_kd import attach_projection
        attach_projection(model, cfg)
    groups = tech.apply(model, c, log=(lambda m: print(m, flush=True)) if ctx.main
                        else (lambda m: None))
    model.to(ctx.device)
    for g in groups:
        g["initial_lr"] = g["lr"]
    opt = torch.optim.AdamW([{k: v for k, v in g.items() if k != "name"} for g in groups],
                            betas=(0.9, 0.98), eps=1e-8)
    return model, opt, tech


def make_scheduler(opt, total_steps: int, warmup_frac: float, kind: str = "cosine"):
    warm = max(1, int(round(warmup_frac * total_steps)))

    def f(step):
        if step < warm:
            return (step + 1) / warm
        prog = min(1.0, (step - warm) / max(1, total_steps - warm))
        if kind == "linear":
            return max(0.0, 1.0 - prog)
        return 0.5 * (1.0 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)


def make_scaler(enabled: bool):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def wrap_ddp(model, ctx):
    if ctx.world == 1:
        return model
    from torch.nn.parallel import DistributedDataParallel as DDP
    kw = {"device_ids": [ctx.local]} if ctx.device.type == "cuda" else {}
    return DDP(model, find_unused_parameters=False, **kw)


def make_loss(cfg: Dict, counts: np.ndarray, device):
    """ce | weighted_ce (inverse-sqrt frequency) | focal (gamma 2) | logit_adjust (Menon et al.
    2021, tau * log prior added to the training logits only)."""
    kind = cfg.get("loss", "weighted_ce")
    w = torch.tensor(class_weights(counts), device=device) if kind == "weighted_ce" else None
    prior = torch.tensor((counts + 1e-3) / (counts.sum() + 1e-3 * len(counts)),
                         dtype=torch.float32, device=device)
    gamma = float(cfg.get("focal_gamma", 2.0))
    tau = float(cfg.get("logit_adjust_tau", 1.0))

    def fn(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        logits = logits.float()
        if kind in ("ce", "weighted_ce"):
            return F.cross_entropy(logits, y, weight=w)
        if kind == "focal":
            logp = F.log_softmax(logits, -1).gather(1, y[:, None]).squeeze(1)
            return (-(1 - logp.exp()) ** gamma * logp).mean()
        if kind == "logit_adjust":
            return F.cross_entropy(logits + tau * torch.log(prior), y)
        raise ValueError(f"unknown loss {kind!r}")
    return fn


def train_windows(cfg: Dict, meta: Dict, docs: Dict) -> List[Dict]:
    keys = [k for k in meta["split"]["train"] + (meta["split"]["dev"] if cfg.get("refit") else [])
            if docs[k]["domain"] in set(cfg.get("domains") or [])]
    return build_windows(docs, keys, cfg, meta, "train")


# =============================================================================================
# the training loop (shared by every mode)
# =============================================================================================
def train_loop(cfg, ctx, run_dir, meta, docs, windows, mode, lr_mult=1.0, max_steps=None,
               epochs=None, resume=True, eval_each_epoch=True, labels_override=None,
               kill_at_step=None):
    """Returns (summary, raw_model, technique). Raises SystemExit with the module's codes."""
    torch.manual_seed(cfg["seed"] + ctx.rank)
    random.seed(cfg["seed"] + ctx.rank)
    np.random.seed(cfg["seed"] + ctx.rank)
    model, opt, tech = make_model(cfg, ctx, lr_mult)
    raw_model = model
    amp_dtype, amp_ctx = amp_setup(ctx.device)
    scaler = make_scaler(amp_dtype == torch.float16)
    tr_keys = [k for k in meta["split"]["train"] if docs[k]["domain"] in set(cfg["domains"])]
    counts = class_counts(docs, tr_keys, cfg["n_labels"])
    loss_fn = make_loss(cfg, counts, ctx.device)
    ckpt_dir = os.path.join(run_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)
    half_best = cfg["technique"] in FULL_PARAM

    # ---- KD: teacher outputs, cached once per training window ---------------------------------
    t_cache = None
    if cfg["technique"] == "kd" and mode != "overfit":
        from rr_kd import build_teacher_cache, load_teacher
        teacher, _tcfg = load_teacher(cfg, ctx.device, find_phase1_models(default_input_roots()))
        n_s, n_t = model_size(raw_model)["total_params"], model_size(teacher)["total_params"]
        log(ctx, run_dir, f"   [kd] student {n_s / 1e6:.1f}M params < teacher {n_t / 1e6:.1f}M "
                          f"({n_s / n_t:.2f}x)")
        assert n_s < n_t, "the KD student must be smaller than the teacher"
        if ctx.main:
            t_cache = build_teacher_cache(teacher, windows, cfg, meta, ctx.device, amp_ctx,
                                          os.path.join(run_dir, "teacher_cache.pt"),
                                          float(cfg.get("kd_hidden_weight", 0)) > 0,
                                          log=lambda m: log(ctx, run_dir, m))
        ctx.barrier()
        if not ctx.main:
            t_cache = torch.load(os.path.join(run_dir, "teacher_cache.pt"), map_location="cpu")
        del teacher
        if ctx.device.type == "cuda":
            torch.cuda.empty_cache()

    # ---- micro-batch: memory probe on the longest window --------------------------------------
    mb = int(cfg.get("micro_batch", 4))
    if ctx.device.type == "cuda":
        longest = max(range(len(windows)), key=lambda i: len(windows[i]["tokens"]))
        while True:
            try:
                b = collate(windows, [longest] * mb, meta["pad_id"], ctx.device)
                with amp_ctx():
                    logits, _v = raw_model(b["input_ids"], b["attention_mask"], b["markers"],
                                           b["starts"], b["marker_mask"])
                    loss = logits.float().mean()
                scaler.scale(loss).backward()
                raw_model.zero_grad(set_to_none=True)
                del b, logits, loss, _v
                torch.cuda.empty_cache()
                break
            except RuntimeError as e:
                if not is_oom(e) or mb == 1:
                    raise
                raw_model.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                mb //= 2
        mb = int(ctx.all_min(mb))
        torch.cuda.reset_peak_memory_stats(ctx.device)
    eff = int(cfg.get("effective_batch", 8))
    accum = max(1, math.ceil(eff / (mb * ctx.world)))
    per_step = mb * ctx.world * accum
    n = len(windows)
    steps_per_epoch = max(1, math.ceil(n / per_step))
    epochs = epochs or cfg["epochs"]
    total_steps = max_steps or steps_per_epoch * epochs
    sched = make_scheduler(opt, total_steps, cfg.get("warmup_frac", 0.06),
                           cfg.get("schedule", "cosine"))
    model = wrap_ddp(raw_model, ctx)
    order = EpochOrder(n, cfg["seed"])

    # ---- resume ---------------------------------------------------------------------------------
    st = {"epoch": 0, "consumed": 0, "global_step": 0, "best_f1": -1.0, "best_epoch": -1,
          "bad_epochs": 0, "history": [], "skipped": 0, "train_seconds": 0.0, "resumed_from": None,
          "max_f1_seen": -1.0}
    last_pt = os.path.join(ckpt_dir, "last.pt")
    if mode == "train" and resume and os.path.exists(last_pt):
        ck = torch_load(last_pt)
        load_trainable_state(raw_model, ck["trainable"])
        opt.load_state_dict(ck["optimizer"])
        sched.load_state_dict(ck["scheduler"])
        scaler.load_state_dict(ck["scaler"])
        st.update(ck["state"])
        st["resumed_from"] = st["global_step"]
        if ck.get("total_steps") != total_steps:
            log(ctx, run_dir, f"   [resume] total steps changed {ck.get('total_steps')} -> "
                              f"{total_steps} (micro-batch/world change)")
        log(ctx, run_dir, f"   [resume] epoch {st['epoch']} consumed {st['consumed']} "
                          f"step {st['global_step']} best F1 {st['best_f1']:.4f}")
        if ctx.main:
            write_json(os.path.join(run_dir, "logs", "resume.json"),
                       {"resumed_at_step": st["global_step"], "epoch": st["epoch"],
                        "time": time.strftime("%Y-%m-%d %H:%M:%S")})

    def save_last():
        if not ctx.main:
            return
        tmp = last_pt + ".tmp"
        torch.save({"trainable": trainable_state_dict(raw_model), "optimizer": opt.state_dict(),
                    "scheduler": sched.state_dict(), "scaler": scaler.state_dict(),
                    "state": st, "total_steps": total_steps, "micro_batch": mb,
                    "world": ctx.world}, tmp)
        os.replace(tmp, last_pt)

    def save_best():
        if not ctx.main:
            return
        p = os.path.join(ckpt_dir, "best.pt")
        torch.save(trainable_state_dict(raw_model, half=half_best), p + ".tmp")
        os.replace(p + ".tmp", p)

    deadline = float(cfg.get("deadline_unix", 0) or 0)
    margin = float(cfg.get("deadline_margin_s", 300))
    ckpt_every = float(cfg.get("ckpt_every_min", 30)) * 60
    last_ckpt_t = time.time()
    t_start = time.time()
    clip = float(cfg.get("grad_clip", 1.0))
    log_every = int(cfg.get("log_every", 25))
    alpha, T = float(cfg.get("kd_alpha", 0.5)), float(cfg.get("kd_T", 2.0))
    hidden_w = float(cfg.get("kd_hidden_weight", 0.0))
    losses: List[float] = []
    first_loss = None
    stop_reason = None
    tok_count, tok_t0 = 0, time.time()
    min_steps_for_watchdog = min(150, max(1, total_steps // 3))
    watchdog_done = st["global_step"] >= min_steps_for_watchdog

    model.train()
    while st["epoch"] < epochs and stop_reason is None:
        idx = order.indices(st["epoch"], ctx.rank, ctx.world, st["consumed"])
        pos = 0
        while pos < len(idx):
            if max_steps and st["global_step"] >= max_steps:
                stop_reason = "max_steps"
                break
            step_loss, micro = 0.0, 0
            for a in range(accum):
                chunk = idx[pos:pos + mb]
                if not chunk:
                    break
                pos += len(chunk)
                micro += 1
                b = collate(windows, chunk, meta["pad_id"], ctx.device, labels_override)
                sync = (a == accum - 1) or pos >= len(idx)
                cm = model.no_sync() if (ctx.world > 1 and not sync) else _null()
                try:
                    with cm:
                        with amp_ctx():
                            logits, vec = model(b["input_ids"], b["attention_mask"], b["markers"],
                                                b["starts"], b["marker_mask"])
                        valid = b["labels"] != IGNORE
                        y = b["labels"][valid]
                        s_logits = logits[valid].float()
                        if t_cache is not None:
                            t_l = torch.cat([t_cache["logits"][i] for i in chunk]).to(ctx.device)
                            t_v = (torch.cat([t_cache["vectors"][i] for i in chunk]).to(ctx.device)
                                   if hidden_w > 0 else None)
                            from rr_kd import kd_loss
                            loss = kd_loss(s_logits, t_l, y, loss_fn, alpha, T,
                                           vec[valid] if hidden_w > 0 else None, t_v, hidden_w,
                                           raw_model.kd_proj)
                        else:
                            loss = loss_fn(s_logits, y) if y.numel() else logits.sum() * 0.0
                        extra = tech.extra_loss(raw_model)
                        if extra is not None:
                            loss = loss + extra
                        scaler.scale(loss / accum).backward()
                except RuntimeError as e:
                    if not is_oom(e):
                        raise
                    if ctx.world > 1:
                        log(ctx, run_dir, "   [oom] out of memory under DDP -> exit for retry")
                        save_last()
                        raise SystemExit(EXIT_OOM)
                    opt.zero_grad(set_to_none=True)
                    torch.cuda.empty_cache()
                    if mb == 1:
                        raise
                    mb //= 2
                    accum *= 2
                    log(ctx, run_dir, f"   [oom] micro-batch -> {mb}, accumulation -> {accum}")
                    pos -= len(chunk)
                    step_loss, micro = 0.0, 0
                    break
                step_loss += float(loss.detach()) / accum
                tok_count += int(b["attention_mask"].sum())
            if micro == 0:
                continue
            # ---------------- step (NaN-safe; identical decision on all ranks) ----------------
            scaler.unscale_(opt)
            params = [p for p in raw_model.parameters() if p.requires_grad]
            gnorm = torch.nn.utils.clip_grad_norm_(params, clip)
            finite = bool(torch.isfinite(gnorm))
            if finite:
                tech.after_backward(raw_model, st["global_step"], total_steps)
            if finite or amp_dtype == torch.float16:
                scaler.step(opt)
            if not finite and st["global_step"] >= 30:
                st["skipped"] += 1
            scaler.update()
            opt.zero_grad(set_to_none=True)
            sched.step()
            st["global_step"] += 1
            st["consumed"] = min(n, st["consumed"] + per_step)
            if first_loss is None and math.isfinite(step_loss):
                first_loss = step_loss
            losses.append(step_loss)
            collapsed = amp_dtype == torch.float16 and scaler.get_scale() < 1e-4
            if collapsed or (st["skipped"] >= 5 and
                             st["skipped"] > 0.05 * max(1, st["global_step"] - 30)):
                save_last()
                log(ctx, run_dir, f"   [nan] {st['skipped']} steps with non-finite gradients "
                                  f"(loss scale {scaler.get_scale():.3g}) -> abort")
                raise SystemExit(EXIT_NAN)
            if st["global_step"] % log_every == 0 or st["global_step"] == 1:
                el = time.time() - tok_t0
                rec = {"step": st["global_step"], "epoch": st["epoch"],
                       "loss": float(np.mean(losses[-log_every:])),
                       "lr": opt.param_groups[0]["lr"], "grad_norm": float(gnorm),
                       "tokens_per_s": tok_count * ctx.world / max(el, 1e-9),
                       "peak_mem_gb": (torch.cuda.max_memory_allocated(ctx.device) / 1e9
                                       if ctx.device.type == "cuda" else 0.0),
                       "skipped": st["skipped"]}
                log(ctx, run_dir, f"   step {rec['step']:>5}/{total_steps} | ep {st['epoch']} | "
                                  f"loss {rec['loss']:.4f} | lr {rec['lr']:.2e} | gnorm "
                                  f"{rec['grad_norm']:.2f} | {rec['tokens_per_s']:,.0f} tok/s | "
                                  f"mem {rec['peak_mem_gb']:.1f} GB", rec if mode == "train" else None)
            if mode == "train":
                if kill_at_step is not None and st["global_step"] == kill_at_step:
                    st["train_seconds"] += time.time() - t_start
                    save_last()
                    log(ctx, run_dir, f"   [test] simulated kill after step {kill_at_step}")
                    sys.stdout.flush()
                    os._exit(137)
                over = [1.0 if (deadline and time.time() > deadline - margin) else 0.0]
                if ctx.bcast(over)[0]:
                    st["train_seconds"] += time.time() - t_start
                    save_last()
                    log(ctx, run_dir, "   [time] job time budget reached -> checkpoint saved")
                    raise SystemExit(EXIT_TIME)
                if time.time() - last_ckpt_t > ckpt_every:
                    save_last()
                    last_ckpt_t = time.time()
                    log(ctx, run_dir, f"   [ckpt] last.pt written at step {st['global_step']}")
        if stop_reason is not None:
            break
        # ---------------- end of epoch ----------------
        st["epoch"] += 1
        st["consumed"] = 0
        if not eval_each_epoch:
            continue
        f1 = -1.0
        if ctx.main:
            f1, _ = quick_dev_f1(raw_model, meta, docs, cfg, ctx.device, amp_ctx)
            raw_model.train()
        f1 = ctx.bcast([f1])[0]
        st["max_f1_seen"] = max(st.get("max_f1_seen", -1.0), f1)
        selectable = tech.selectable(st["global_step"], total_steps)
        improved = selectable and f1 > st["best_f1"]
        if improved:
            st["best_f1"], st["best_epoch"], st["bad_epochs"] = f1, st["epoch"], 0
            save_best()
        elif selectable:
            st["bad_epochs"] += 1
        st["history"].append({"epoch": st["epoch"], "dev_macro_f1": f1, "selectable": selectable,
                              "train_loss": float(np.mean(losses[-steps_per_epoch:]))
                              if losses else None})
        log(ctx, run_dir, f"   == epoch {st['epoch']}/{epochs} | dev pooled Macro-F1 {f1:.4f} "
                          f"{'(best)' if improved else ''}"
                          f"{'' if selectable else ' (not selectable yet)'} | train loss "
                          f"{st['history'][-1]['train_loss'] or float('nan'):.4f}",
            {"epoch_end": st["epoch"], "dev_macro_f1": f1} if mode == "train" else None)
        if mode == "train":
            if not watchdog_done and st["global_step"] >= min_steps_for_watchdog:
                watchdog_done = True
                if st["max_f1_seen"] < float(cfg.get("watchdog_min_f1", 0.01)):
                    save_last()
                    log(ctx, run_dir, f"   [watchdog] dev F1 {st['max_f1_seen']:.4f} after "
                                      f"{st['global_step']} steps -> stop (check GATES.json, "
                                      f"the learning rate and logs/overfit.json)")
                    raise SystemExit(EXIT_EARLYFAIL)
            if st["bad_epochs"] >= int(cfg.get("patience", 4)):
                stop_reason = "early_stop"
            if time.time() - last_ckpt_t > 15 * 60:
                save_last()
                last_ckpt_t = time.time()
    st["train_seconds"] += time.time() - t_start
    el = time.time() - tok_t0
    summary = {"status": stop_reason or "done", "epochs_run": st["epoch"],
               "global_step": st["global_step"], "steps_per_epoch": steps_per_epoch,
               "total_steps": total_steps, "micro_batch": mb, "accum": accum, "world": ctx.world,
               "windows": n, "best_dev_macro_f1": st["best_f1"], "best_epoch": st["best_epoch"],
               "history": st["history"], "skipped_steps": st["skipped"],
               "first_loss": first_loss,
               "last_loss": float(np.mean(losses[-10:])) if losses else None,
               "tokens_per_s": tok_count * ctx.world / max(el, 1e-9),
               "seconds_per_step": el / max(1, len(losses)),
               "peak_mem_gb": (torch.cuda.max_memory_allocated(ctx.device) / 1e9
                               if ctx.device.type == "cuda" else 0.0),
               **model_size(raw_model), "train_seconds": st["train_seconds"],
               "resumed_from_step": st["resumed_from"], "lr_mult": lr_mult,
               "peft_report": tech.final_report(raw_model)}
    return summary, raw_model, tech


@torch.no_grad()
def subset_accuracy(model, windows, labels_override, meta, device, amp_ctx) -> float:
    model.eval()
    idx = sorted(labels_override)
    b = collate(windows, idx, meta["pad_id"], device, labels_override)
    with amp_ctx():
        logits, _ = model(b["input_ids"], b["attention_mask"], b["markers"], b["starts"],
                          b["marker_mask"])
    valid = b["labels"] != IGNORE
    return float((logits[valid].argmax(-1) == b["labels"][valid]).float().mean())


def overfit_subset(windows, n_sent: int = 32):
    """The first windows holding n_sent labelled sentences; other slots ignored."""
    over, left = {}, n_sent
    for i, w in enumerate(windows):
        if left <= 0:
            break
        lab = w["labels"].copy()
        keep = np.nonzero(lab != IGNORE)[0][:left]
        mask = np.full_like(lab, IGNORE)
        mask[keep] = lab[keep]
        if len(keep):
            over[i] = mask
            left -= len(keep)
    return over


# =============================================================================================
# modes
# =============================================================================================
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--run-dir", required=True)
    ap.add_argument("--mode", choices=["train", "smoke", "overfit", "probe"], default="train")
    ap.add_argument("--max-steps", type=int, default=None)
    ap.add_argument("--tag", default="")
    ap.add_argument("--kill-at-step", type=int, default=None, help="test hook (selftest #7)")
    a = ap.parse_args()
    run_dir = a.run_dir
    cfg = read_json(os.path.join(run_dir, "logs", "run_config.json"))
    ctx = Ctx()
    code = 0
    try:
        meta, docs = load_cache(cfg["data_dir"])
        windows = train_windows(cfg, meta, docs)
        if ctx.main:
            n_lab = sum(int((w["labels"] != IGNORE).sum()) for w in windows)
            print(f"   train windows: {len(windows)} ({cfg['input_mode']}) | labelled sentence "
                  f"slots {n_lab} | max tokens {max(len(w['tokens']) for w in windows)}",
                  flush=True)
        # first-batch assertions
        b = collate(windows, list(range(min(4, len(windows)))), meta["pad_id"])
        assert int(b["input_ids"].max()) < meta["vocab_size"] and int(b["input_ids"].min()) >= 0
        y = b["labels"]
        assert ((y == IGNORE) | ((y >= 0) & (y < cfg["n_labels"]))).all(), "bad label"
        assert (y[~b["marker_mask"]] == IGNORE).all(), "labels on padded marker slots"
        mk = b["markers"][b["marker_mask"]]
        assert (b["input_ids"].gather(1, b["markers"])[b["marker_mask"]] == meta["eos_id"]).all(), \
            "a marker does not point at EOS"
        assert b["input_ids"].shape[1] + int(cfg.get("n_virtual", 0)) <= 1024 and mk.numel() > 0

        if a.mode == "smoke":
            if ctx.main and not cfg.get("skip_job_checks"):
                import selftest
                res = selftest.job_checks(cfg, meta, docs, ctx.device)
                write_json(os.path.join(run_dir, "logs", "job_checks.json"), res)
                if not res["ok"]:
                    print(f"   job self-checks FAILED: {res['failed']}", flush=True)
                    raise SystemExit(EXIT_SELFCHECK)
                print(f"   job self-checks passed: {sorted(res['checks'])}", flush=True)
            steps = a.max_steps or 30
            s, _m, _t = train_loop(cfg, ctx, run_dir, meta, docs, windows, "smoke",
                                   max_steps=steps, resume=False, eval_each_epoch=False)
            if ctx.main:
                epochs = cfg["epochs"]
                train_min = s["seconds_per_step"] * s["steps_per_epoch"] * epochs / 60
                n_dev_tok = sum(len(x) + 1 for k in eval_keys(meta, docs, "dev", cfg)
                                for x in docs[k]["ids"]) * 2
                eval_min = epochs * n_dev_tok / max(1.0, 3 * s["tokens_per_s"]) / 60
                s["projected_train_minutes"] = train_min + eval_min
                s["loss_decreased"] = (s["last_loss"] is not None and s["first_loss"] is not None
                                       and s["last_loss"] < s["first_loss"])
                write_json(os.path.join(run_dir, "logs", f"smoke{a.tag}.json"), s)
                print(f"   smoke ok: {s['tokens_per_s']:,.0f} tok/s | mem {s['peak_mem_gb']:.1f} GB"
                      f" | micro-batch {s['micro_batch']} x accum {s['accum']} x world "
                      f"{s['world']} | projected {s['projected_train_minutes']:.1f} min", flush=True)
        elif a.mode == "overfit":
            over = overfit_subset(windows, 32)
            sub = sorted(over)
            sub_w = [windows[i] for i in sub]
            sub_over = {j: over[i] for j, i in enumerate(sub)}
            c2 = dict(cfg)
            c2["dropout"] = 0.0
            c2["warmup_frac"] = 0.05
            c2["effective_batch"] = len(sub_w)
            c2["loss"] = "ce"
            c2["head_lr"] = max(float(cfg["head_lr"]), 1e-2)
            mult = 10.0 if cfg["technique"] in FULL_PARAM else 3.0
            s, model, _t = train_loop(c2, ctx, run_dir, meta, docs, sub_w, "overfit",
                                      lr_mult=mult, max_steps=a.max_steps or 300, epochs=10 ** 6,
                                      resume=False, eval_each_epoch=False,
                                      labels_override=sub_over)
            if ctx.main:
                _amp, amp_ctx = amp_setup(ctx.device)
                acc = subset_accuracy(model, sub_w, sub_over, meta, ctx.device, amp_ctx)
                s["train_accuracy"] = acc
                s["n_sentences"] = int(sum((v != IGNORE).sum() for v in sub_over.values()))
                s["passed"] = bool(acc > 0.95)
                s["fatal"] = bool(acc < 0.5)
                write_json(os.path.join(run_dir, "logs", "overfit.json"), s)
                print(f"   overfit-32: train accuracy {acc:.3f} after {s['global_step']} steps "
                      f"(loss {s['first_loss']:.3f} -> {s['last_loss']:.4f}) "
                      f"{'PASS' if s['passed'] else 'FAIL'}", flush=True)
        elif a.mode == "probe":
            results = []
            for mult in cfg.get("probe_mults", [1 / 3, 1.0, 3.0]):
                budget = a.max_steps or cfg.get("probe_steps")
                if budget is None:
                    eff = cfg.get("effective_batch", 8)
                    budget = min(200, max(30, math.ceil(len(windows) / eff)))
                s, model, _t = train_loop(cfg, ctx, run_dir, meta, docs, windows, "probe",
                                          lr_mult=mult, max_steps=budget, epochs=10 ** 6,
                                          resume=False, eval_each_epoch=False)
                f1 = -1.0
                if ctx.main:
                    _amp, amp_ctx = amp_setup(ctx.device)
                    f1, _ = quick_dev_f1(model, meta, docs, cfg, ctx.device, amp_ctx)
                f1 = ctx.bcast([f1])[0]
                results.append({"lr_mult": mult, "lr": cfg["lr"] * mult, "steps": budget,
                                "dev_macro_f1": f1, "last_loss": s["last_loss"]})
                if ctx.main:
                    print(f"   probe lr x{mult:.3g} ({cfg['lr'] * mult:.2e}): dev F1 {f1:.4f} "
                          f"loss {s['last_loss']:.4f}", flush=True)
                del model
                if ctx.device.type == "cuda":
                    torch.cuda.empty_cache()
            best = max(results, key=lambda r: (r["dev_macro_f1"], -(r["last_loss"] or 1e9)))
            if ctx.main:
                write_json(os.path.join(run_dir, "logs", "lr_probe.json"),
                           {"best_lr_mult": best["lr_mult"], "best_lr": best["lr"],
                            "results": results})
        else:
            s, _m, _t = train_loop(cfg, ctx, run_dir, meta, docs, windows, "train",
                                   kill_at_step=a.kill_at_step)
            if ctx.main:
                write_json(os.path.join(run_dir, "logs", "train_summary.json"), s)
                if s["best_epoch"] < 0:
                    raise RuntimeError("training produced no best checkpoint")
                print(f"   training {s['status']}: best dev Macro-F1 {s['best_dev_macro_f1']:.4f} "
                      f"at epoch {s['best_epoch']} ({s['train_seconds'] / 60:.1f} min)", flush=True)
    except SystemExit as e:
        code = int(e.code or 0)
    except Exception:
        traceback.print_exc()
        code = 1
    finally:
        try:
            ctx.close()
        except Exception:
            pass
    sys.exit(code)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase3/rr_predict.py
"""
rr_predict.py -- window inference, dev report, bias decision, test prediction, submission,
validator, raw logits, DONE.json.

    python rr_predict.py --run-dir OUT/E1_lorma/M2_llamastyle        # after training
    python rr_predict.py --validate PATH/submission.zip              # check any zip

Inference: each document is packed into overlapping windows exactly as in training
(rr_data.build_windows, inference stride); per-sentence logits are taken from the window where
the sentence has the most context ("center") or averaged over windows ("mean"). Exactly one
label per sentence, in order, mapped back through label_vocab with the official spelling.

Submission: ONE prediction.json {doc_id: [label, ...]} covering every CL and IT test id, zipped
as submission.zip with prediction.json at the ROOT. `validate_submission` re-opens the zip and
checks: the only entry is exactly "prediction.json"; valid JSON without NaN/Infinity; a dict of
lists of strings; id set == union of the official test ids; per-document length == number of
test sentences; every label in the vocabulary. It runs after every write; failure aborts.

The validator part is torch-free (used by the data gates); torch is imported lazily.

Inputs : run dir (logs/run_config.json, checkpoints/best.pt), data cache
Outputs: dev/dev_report.json, dev/confusion.csv, logits/{dev,test}_logits.npy (+ index),
         submission/{prediction.json, submission.zip, validation_report.json}, DONE.json
"""
import argparse
import contextlib
import json
import os
import sys
import time
import zipfile
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

from rr_common import (EXIT, SPEC, default_input_roots, find_phase1_models, read_json,
                       write_json)
from rr_data import build_windows, combine_slot_scores, load_cache
from rr_eval import (bias_decision, full_report, headline, labels_from_logits,
                     write_confusion_csv)


# =============================================================================================
# submission writing + validation (torch-free)
# =============================================================================================
def write_submission(sub_dir: str, pred: Dict[str, List[str]]) -> str:
    os.makedirs(sub_dir, exist_ok=True)
    pj = os.path.join(sub_dir, SPEC["submission_entry"])
    with open(pj + ".tmp", "w", encoding="utf-8") as f:
        json.dump(pred, f, ensure_ascii=False, allow_nan=False)
    os.replace(pj + ".tmp", pj)
    zp = os.path.join(sub_dir, SPEC["submission_zip"])
    with zipfile.ZipFile(zp + ".tmp", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname=SPEC["submission_entry"])           # at the ROOT of the archive
    os.replace(zp + ".tmp", zp)
    return zp


def _no_nan(x):
    raise ValueError(f"non-finite number {x} in JSON")


def validate_submission(zip_path: str, test_n_sent: Dict[str, int],
                        labels: Sequence[str]) -> Dict:
    """test_n_sent = {official test id: number of sentences} for CL and IT together."""
    errors: List[str] = []
    try:
        with zipfile.ZipFile(zip_path) as z:
            names = z.namelist()
            if names != [SPEC["submission_entry"]]:
                errors.append(f"zip must contain exactly ['{SPEC['submission_entry']}'] at the "
                              f"root, got {names}")
            if SPEC["submission_entry"] not in names:
                return {"ok": False, "errors": errors}
            raw = z.read(SPEC["submission_entry"]).decode("utf-8")
    except (zipfile.BadZipFile, OSError) as e:
        return {"ok": False, "errors": [f"cannot open zip: {e}"]}
    try:
        pred = json.loads(raw, parse_constant=_no_nan)
    except ValueError as e:
        return {"ok": False, "errors": errors + [f"invalid JSON: {e}"]}
    if not isinstance(pred, dict):
        return {"ok": False, "errors": errors + ["prediction.json must be an object {id: [..]}"]}
    want, got = set(test_n_sent), set(pred)
    if want - got:
        errors.append(f"{len(want - got)} test ids missing, e.g. {sorted(want - got)[:3]}")
    if got - want:
        errors.append(f"{len(got - want)} unknown ids, e.g. {sorted(got - want)[:3]}")
    labset = set(labels)
    n_sent = 0
    for did, labs in pred.items():
        if not isinstance(labs, list):
            errors.append(f"{did}: value must be a list of labels")
            continue
        if did in test_n_sent and len(labs) != test_n_sent[did]:
            errors.append(f"{did}: {len(labs)} labels for {test_n_sent[did]} sentences")
        bad = [l for l in labs if not isinstance(l, str) or l not in labset]
        if bad:
            errors.append(f"{did}: labels outside the vocabulary: {bad[:3]}")
        n_sent += len(labs)
    return {"ok": not errors, "errors": errors[:20], "n_docs": len(pred), "n_labels": n_sent}


def g2_validator_roundtrip(test_n_sent: Dict[str, int], labels: Sequence[str],
                           work_dir: str) -> Dict:
    """Gate G2b: accept a dummy all-'None' prediction, reject 5 corrupted variants."""
    d = os.path.join(work_dir, "g2_validator")
    good = {k: ["None"] * n for k, n in test_n_sent.items()}
    zp = write_submission(os.path.join(d, "good"), good)
    r = validate_submission(zp, test_n_sent, labels)
    if not r["ok"]:
        from rr_common import GateFailure
        raise GateFailure(f"G2: validator rejects the dummy prediction: {r['errors']}")
    k0 = sorted(test_n_sent)[0]
    variants = {
        "missing_id": {k: v for k, v in good.items() if k != k0},
        "extra_id": {**good, "__not_a_test_id__": ["None"]},
        "short_list": {**good, k0: good[k0][:-1]},
        "long_list": {**good, k0: good[k0] + ["None"]},
        "bad_label": {**good, k0: ["NotALabel"] + good[k0][1:]},
    }
    rejected = {}
    for name, p in variants.items():
        zp = write_submission(os.path.join(d, name), p)
        rv = validate_submission(zp, test_n_sent, labels)
        if rv["ok"]:
            from rr_common import GateFailure
            raise GateFailure(f"G2: validator ACCEPTS the corrupted variant '{name}'")
        rejected[name] = rv["errors"][:2]
    # a zip with a folder layer must be rejected too
    nested = os.path.join(d, "nested.zip")
    with zipfile.ZipFile(nested, "w") as z:
        z.writestr("folder/prediction.json", json.dumps(good))
    if validate_submission(nested, test_n_sent, labels)["ok"]:
        from rr_common import GateFailure
        raise GateFailure("G2: validator accepts prediction.json inside a folder")
    rejected["folder_layer"] = ["rejected"]
    return {"accepted_dummy": True, "rejected": rejected}


def official_test_counts(meta: Dict, docs: Dict[str, Dict]) -> Dict[str, int]:
    return {docs[k]["id"]: len(docs[k]["text"]) for k in meta["split"]["test"]}


# =============================================================================================
# inference (torch)
# =============================================================================================
def amp_setup(device):
    """The Phase-1 notebooks' AMP pattern: bf16 on Ampere+, fp16 on T4/P100, none on CPU."""
    import torch
    if device.type == "cuda" and torch.cuda.get_device_capability(device)[0] >= 8:
        amp_dtype = torch.bfloat16
    elif device.type == "cuda":
        amp_dtype = torch.float16
    else:
        amp_dtype = None

    def amp_ctx():
        if amp_dtype is None:
            return contextlib.nullcontext()
        return torch.autocast(device_type="cuda", dtype=amp_dtype)
    return amp_dtype, amp_ctx


def token_batches(windows, max_tokens: int) -> List[List[int]]:
    """Windows sorted by length, grouped so that B * max_len <= max_tokens."""
    order = sorted(range(len(windows)), key=lambda i: len(windows[i]["tokens"]))
    batches, cur, cur_max = [], [], 0
    for i in order:
        n = len(windows[i]["tokens"])
        if cur and max(cur_max, n) * (len(cur) + 1) > max_tokens:
            batches.append(cur)
            cur, cur_max = [], 0
        cur.append(i)
        cur_max = max(cur_max, n)
    if cur:
        batches.append(cur)
    return batches


def window_outputs(model, windows, meta: Dict, device, amp_ctx, eval_batch: int = 16,
                   return_vectors: bool = False):
    """[(n_slots, C) float32 CPU tensor per window], [(n_slots, d)] or None."""
    import torch
    from rr_models import collate
    was = model.training
    model.eval()
    out: List = [None] * len(windows)
    vecs: Optional[List] = [None] * len(windows) if return_vectors else None
    with torch.no_grad():
        for idx in token_batches(windows, int(eval_batch) * SPEC["context"]):
            b = collate(windows, idx, meta["pad_id"], device)
            with amp_ctx():
                logits, v = model(b["input_ids"], b["attention_mask"], b["markers"], b["starts"],
                                  b["marker_mask"])
            logits = logits.float().cpu()
            if return_vectors:
                v = v.float().cpu()
            for r, i in enumerate(idx):
                m = len(windows[i]["markers"])
                out[i] = logits[r, :m]
                if return_vectors:
                    vecs[i] = v[r, :m]
    model.train(was)
    return out, vecs


def eval_keys(meta: Dict, docs: Dict, split: str, cfg: Dict) -> List[str]:
    doms = set(cfg.get("domains") or SPEC["domains"])
    return [k for k in meta["split"][split] if docs[k]["domain"] in doms]


def doc_logits(models, docs: Dict, keys: Sequence[str], cfg: Dict, meta: Dict, device,
               amp_ctx) -> Tuple[Dict[str, np.ndarray], float]:
    """{key: (n_sent, C)} averaged over `models` (log-softmax space when > 1), tokens/s."""
    import torch
    wins = build_windows(docs, keys, cfg, meta, "infer")
    n_sent = {k: len(docs[k]["ids"]) for k in keys}
    t0 = time.time()
    acc = None
    for m in models:
        slot, _ = window_outputs(m, wins, meta, device, amp_ctx, cfg.get("eval_batch", 16))
        if len(models) > 1:
            slot = [torch.log_softmax(s, -1) for s in slot]
        per = combine_slot_scores(wins, [s.numpy() for s in slot], n_sent,
                                  cfg.get("window_combine", "center"))
        acc = per if acc is None else {k: acc[k] + per[k] for k in per}
    acc = {k: v / len(models) for k, v in acc.items()}
    n_tok = sum(len(w["tokens"]) for w in wins) * len(models)
    return acc, n_tok / max(1e-9, time.time() - t0)


def quick_dev_f1(model, meta: Dict, docs: Dict, cfg: Dict, device, amp_ctx) -> Tuple[float, Dict]:
    """Pooled dev Macro-F1 (argmax, no bias) -- used for early stopping and the LR probe."""
    keys = eval_keys(meta, docs, "dev", cfg)
    lg, _ = doc_logits([model], docs, keys, cfg, meta, device, amp_ctx)
    rep = full_report({k: docs[k]["labels"] for k in keys}, labels_from_logits(lg),
                      {k: docs[k]["domain"] for k in keys}, meta["labels"])
    return headline(rep), rep


# =============================================================================================
# finalisation from logits (shared by single runs, domain combination and ensembles)
# =============================================================================================
def save_logits(path_dir: str, split: str, logits: Dict[str, np.ndarray]) -> None:
    os.makedirs(path_dir, exist_ok=True)
    keys = sorted(logits)
    arr = np.concatenate([logits[k] for k in keys]).astype(np.float16) if keys else \
        np.zeros((0, len(SPEC["labels"])), np.float16)
    np.save(os.path.join(path_dir, f"{split}_logits.npy"), arr)
    off, idx = 0, {}
    for k in keys:
        idx[k] = [off, off + len(logits[k])]
        off += len(logits[k])
    write_json(os.path.join(path_dir, f"{split}_index.json"), idx)


def load_logits(path_dir: str, split: str) -> Optional[Dict[str, np.ndarray]]:
    p = os.path.join(path_dir, f"{split}_logits.npy")
    idx = read_json(os.path.join(path_dir, f"{split}_index.json"))
    if not os.path.exists(p) or idx is None:
        return None
    arr = np.load(p).astype(np.float32)
    return {k: arr[a:b] for k, (a, b) in idx.items()}


def finalize_from_logits(dev_lg: Dict[str, np.ndarray], test_lg: Dict[str, np.ndarray],
                         meta: Dict, docs: Dict, out_dir: str, log=print,
                         tune_bias: bool = True) -> Dict:
    """Dev report + bias decision + (if all domains are covered) a validated submission."""
    labels = meta["labels"]
    dkeys = sorted(dev_lg)
    gold = {k: docs[k]["labels"] for k in dkeys}
    dom = {k: docs[k]["domain"] for k in dkeys}
    rep = full_report(gold, labels_from_logits(dev_lg), dom, labels)
    dec = bias_decision(dev_lg, gold, dom, len(labels)) if tune_bias else \
        {"apply": False, "bias": [0.0] * len(labels), "rule": "disabled"}
    rep_b = full_report(gold, labels_from_logits(dev_lg, dec["bias"]), dom, labels)
    dev_dir = os.path.join(out_dir, "dev")
    write_confusion_csv(os.path.join(dev_dir, "confusion.csv"), rep["confusion"], labels)
    report = {"metrics": rep, "metrics_with_bias_in_sample": rep_b, "bias": dec}
    write_json(os.path.join(dev_dir, "dev_report.json"), report)
    save_logits(os.path.join(out_dir, "logits"), "dev", dev_lg)
    save_logits(os.path.join(out_dir, "logits"), "test", test_lg)
    log(f"   dev Macro-F1 pooled {headline(rep):.4f} | CL "
        f"{rep['per_domain'].get('CL', {}).get('macro_f1', float('nan')):.4f} | IT "
        f"{rep['per_domain'].get('IT', {}).get('macro_f1', float('nan')):.4f} | bias "
        f"{'APPLIED' if dec['apply'] else 'not applied'} (held-out gain "
        f"{dec.get('heldout_gain_mean', 0.0):+.4f})")
    out = {"report": report, "submission": None, "validation": None}
    all_test = set(meta["split"]["test"])
    if set(test_lg) == all_test:
        pred_idx = labels_from_logits(test_lg, dec["bias"])
        pred = {docs[k]["id"]: [labels[i] for i in pred_idx[k].tolist()] for k in sorted(test_lg)}
        zp = write_submission(os.path.join(out_dir, "submission"), pred)
        vr = validate_submission(zp, official_test_counts(meta, docs), labels)
        write_json(os.path.join(out_dir, "submission", "validation_report.json"), vr)
        if not vr["ok"]:
            raise SubmissionInvalid(f"submission failed validation: {vr['errors'][:5]}")
        hist = {}
        for v in pred.values():
            for l in v:
                hist[l] = hist.get(l, 0) + 1
        write_json(os.path.join(out_dir, "submission", "test_label_histogram.json"), hist)
        log(f"   submission written and validated: {zp} ({vr['n_docs']} docs, "
            f"{vr['n_labels']} labels)")
        out.update(submission=zp, validation=vr)
    else:
        log("   partial-domain run: no submission written (combined later)")
    return out


class SubmissionInvalid(RuntimeError):
    pass


def done_record(cfg: Dict, fin: Dict, extra: Dict) -> Dict:
    rep = fin["report"]["metrics"]
    pooled = rep["pooled"]
    return {"exp": cfg.get("exp"), "job": cfg.get("job"), "model": cfg.get("model"),
            "arch": cfg.get("arch"), "technique": cfg.get("technique"),
            "input_mode": cfg.get("input_mode"), "head": cfg.get("head"),
            "attn_mode": cfg.get("attn_mode"), "loss": cfg.get("loss"),
            "domains": cfg.get("domains"), "variant": cfg.get("variant"),
            "dev_macro_f1": pooled["macro_f1"], "dev_macro_precision": pooled["macro_precision"],
            "dev_macro_recall": pooled["macro_recall"],
            "dev_domain_avg_macro_f1": rep["domain_avg_macro_f1"],
            "dev_per_domain_macro_f1": {d: v["macro_f1"] for d, v in rep["per_domain"].items()},
            "dev_macro_f1_bias_in_sample": fin["report"]["metrics_with_bias_in_sample"]
            ["pooled"]["macro_f1"],
            "bias_applied": fin["report"]["bias"]["apply"],
            "bias_heldout_gain": fin["report"]["bias"].get("heldout_gain_mean"),
            "submission": fin["submission"], "submission_valid": bool(fin["validation"] and
                                                                      fin["validation"]["ok"]),
            **extra, "finished": time.strftime("%Y-%m-%d %H:%M:%S")}


def run_single(run_dir: str, data_dir: Optional[str] = None, log=print) -> Dict:
    import torch
    from rr_models import model_size, rebuild_trained_model
    cfg = read_json(os.path.join(run_dir, "logs", "run_config.json"))
    meta, docs = load_cache(data_dir or cfg["data_dir"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _, amp_ctx = amp_setup(device)
    best_pt = os.path.join(run_dir, "checkpoints", "best.pt")
    if not os.path.exists(best_pt):
        raise FileNotFoundError(f"{best_pt} missing -- training did not produce a best model")
    p1 = find_phase1_models(default_input_roots())
    if cfg.get("phase1_dir"):
        p1.setdefault(cfg["model"], cfg["phase1_dir"])
    model = rebuild_trained_model(cfg, best_pt, p1, device)
    size = model_size(model)                      # as trained (before any merge)
    if cfg["technique"] == "prefix":              # drop the reparameterisation MLP
        from rr_peft import get_technique
        get_technique("prefix").merge(model)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    dkeys, tkeys = eval_keys(meta, docs, "dev", cfg), eval_keys(meta, docs, "test", cfg)
    dev_lg, dev_tps = doc_logits([model], docs, dkeys, cfg, meta, device, amp_ctx)
    test_lg, _ = doc_logits([model], docs, tkeys, cfg, meta, device, amp_ctx)
    infer_mem = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else 0.0
    fin = finalize_from_logits(dev_lg, test_lg, meta, docs, run_dir, log,
                               tune_bias=cfg.get("bias_tuning", True))
    train = read_json(os.path.join(run_dir, "logs", "train_summary.json"), {})
    eff = {"inference_params": model_size(model)["total_params"],"total_params": size["total_params"], "trainable_params": size["trainable_params"],
           "trainable_pct": size["trainable_pct"], "train_peak_mem_gb": train.get("peak_mem_gb"),
           "train_tokens_per_s": train.get("tokens_per_s"),
           "train_minutes": (train.get("train_seconds") or 0) / 60,
           "infer_tokens_per_s": dev_tps, "infer_peak_mem_gb": infer_mem}
    peft_report = dict(train.get("peft_report") or {})
    if cfg["technique"] == "adalora":             # ranks of the submitted (best) checkpoint
        from rr_peft import adalora_ranks
        peft_report.update(adalora_ranks(model))
    extra = {"efficiency": eff, "train": train, "peft_report": peft_report}
    if cfg["technique"] == "kd":
        extra["kd_report"] = kd_comparison(cfg, model, size, dev_tps, infer_mem, meta, docs,
                                           device, amp_ctx, p1, headline(fin["report"]["metrics"]))
    done = done_record(cfg, fin, extra)
    write_json(os.path.join(run_dir, "DONE.json"), done)
    return done


def kd_comparison(cfg, student, s_size, s_tps, s_mem, meta, docs, device, amp_ctx, p1,
                  s_f1) -> Dict:
    """Teacher vs student: dev Macro-F1, size, inference speed and memory (same windows)."""
    import torch
    from rr_kd import load_teacher, resolve_teacher
    from rr_models import model_size
    teacher, tcfg = load_teacher(cfg, device, p1)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    keys = eval_keys(meta, docs, "dev", cfg)
    t_lg, t_tps = doc_logits([teacher], docs, keys, tcfg, meta, device, amp_ctx)
    t_mem = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else 0.0
    t_rep = full_report({k: docs[k]["labels"] for k in keys}, labels_from_logits(t_lg),
                        {k: docs[k]["domain"] for k in keys}, meta["labels"])
    t_size = model_size(teacher)
    rc, _ = resolve_teacher(cfg["teacher_dir"], cfg["model"])
    return {"teacher": rc, "teacher_model": tcfg["model"], "teacher_dev_macro_f1": headline(t_rep),
            "student_dev_macro_f1": s_f1, "f1_retained": s_f1 / max(1e-9, headline(t_rep)),
            "teacher_params": t_size["total_params"], "student_params": s_size["total_params"],
            "size_ratio": s_size["total_params"] / t_size["total_params"],
            "teacher_infer_tokens_per_s": t_tps, "student_infer_tokens_per_s": s_tps,
            "speedup": s_tps / max(1e-9, t_tps), "teacher_infer_peak_mem_gb": t_mem,
            "student_infer_peak_mem_gb": s_mem,
            "memory_ratio": (s_mem / t_mem) if t_mem else None,
            "student_layers": cfg.get("layer_ids")}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--run-dir")
    ap.add_argument("--data-dir", default=None)
    ap.add_argument("--validate", default=None, help="validate a submission.zip and exit")
    a = ap.parse_args()
    if a.validate:
        from rr_common import DATA_DIR
        meta, docs = load_cache(a.data_dir or DATA_DIR)
        r = validate_submission(a.validate, official_test_counts(meta, docs), meta["labels"])
        print(json.dumps(r, indent=1))
        sys.exit(EXIT["ok"] if r["ok"] else EXIT["validation_failure"])
    try:
        done = run_single(a.run_dir, a.data_dir)
    except SubmissionInvalid as e:
        print("VALIDATION FAILED:", e, file=sys.stderr)
        sys.exit(EXIT["validation_failure"])
    except FileNotFoundError as e:
        print("MISSING:", e, file=sys.stderr)
        sys.exit(EXIT["missing_dependency"])
    print(f"DONE {done['exp']} {done['job']}: dev Macro-F1 {done['dev_macro_f1']:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase3/rr_select.py
"""
rr_select.py -- best-of-4 per experiment, and the rule / model-coverage check.

    python rr_select.py --exp E1                 # (re)build OUT/E1_lorma/BEST + summary
    python rr_select.py --coverage               # winners, rule compliance, extra uploads

Selection rule: highest dev POOLED Macro-F1 (argmax, before post-hoc bias: the same unbiased
number for every run); ties -> fewer trainable parameters. Only runs with a validated
submission covering CL + IT are eligible (KD teachers and partial-domain ablations are not).
BEST/ receives the winner's submission.zip, best.pt, run_config.json, DONE.json, dev report,
confusion.csv and raw logits, plus SELECTION.json.

Coverage: the brief requires all four Phase-1 models to be used and >= 6 experiments with
>= 5 PEFT techniques (LoRMA compulsory) and >= 1 KD. If the winners do not include every
model, the best submission of each missing model is listed as an extra upload.

Inputs : OUT/<exp>/<job>/DONE.json (+ attached phase3_out folders for --coverage)
Outputs: BEST/, SELECTION.json, summary.csv/json, coverage.json
"""
import argparse
import csv
import os
import shutil
from typing import Dict, List, Optional

from rr_common import (EXPERIMENTS, MODELS, OUT_ROOT, SPEC, TECHNIQUES, default_input_roots,
                       iter_finished_runs, read_json, write_json)


def _rows(exp_dir: str) -> List[Dict]:
    rows = []
    if not os.path.isdir(exp_dir):
        return rows
    for job in sorted(os.listdir(exp_dir)):
        d = os.path.join(exp_dir, job)
        done = read_json(os.path.join(d, "DONE.json"))
        if job == "BEST" or not done:
            continue
        eff = done.get("efficiency") or {}
        rows.append({"job": job, "model": done.get("model"), "technique": done.get("technique"),
                     "variant": done.get("variant"), "dir": d,
                     "dev_macro_f1": done["dev_macro_f1"],
                     "dev_cl_f1": (done.get("dev_per_domain_macro_f1") or {}).get("CL"),
                     "dev_it_f1": (done.get("dev_per_domain_macro_f1") or {}).get("IT"),
                     "dev_macro_precision": done.get("dev_macro_precision"),
                     "dev_macro_recall": done.get("dev_macro_recall"),
                     "trainable_params": eff.get("trainable_params"),
                     "total_params": eff.get("total_params"),
                     "train_minutes": eff.get("train_minutes"),
                     "submission_valid": bool(done.get("submission_valid")),
                     "eligible": bool(done.get("submission_valid")) and not job.startswith(
                         "teacher_")})
    return rows


def _key(r: Dict):
    return (r["dev_macro_f1"], -(r["trainable_params"] or 0))


def select(exp_dir: str, log=print) -> Optional[Dict]:
    rows = _rows(exp_dir)
    if rows:
        with open(os.path.join(exp_dir, "summary.csv"), "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(rows)
        write_json(os.path.join(exp_dir, "summary.json"), rows)
    elig = [r for r in rows if r["eligible"]]
    if not elig:
        log("   no finished run with a valid submission yet -- nothing to select")
        return None
    win = max(elig, key=_key)
    best = os.path.join(exp_dir, "BEST")
    if os.path.isdir(best):
        shutil.rmtree(best)
    os.makedirs(best)
    src = win["dir"]
    for rel, dst in (("submission/submission.zip", "submission.zip"),
                     ("submission/prediction.json", "prediction.json"),
                     ("checkpoints/best.pt", "best.pt"), ("logs/run_config.json", "run_config.json"),
                     ("logs/train_summary.json", "train_summary.json"), ("DONE.json", "DONE.json"),
                     ("dev/dev_report.json", "dev_report.json"), ("dev/confusion.csv",
                                                                  "confusion.csv")):
        p = os.path.join(src, rel)
        if os.path.exists(p):
            shutil.copy2(p, os.path.join(best, dst))
    if os.path.isdir(os.path.join(src, "logits")):
        shutil.copytree(os.path.join(src, "logits"), os.path.join(best, "logits"))
    sel = {"winner": win["job"], "model": win["model"], "technique": win["technique"],
           "dev_macro_f1": win["dev_macro_f1"], "from": src, "n_eligible": len(elig),
           "rule": "max dev pooled Macro-F1 (no bias), ties -> fewer trainable params",
           "table": sorted(({k: r[k] for k in ("job", "model", "dev_macro_f1", "dev_cl_f1",
                                               "dev_it_f1", "trainable_params")} for r in elig),
                           key=lambda r: -r["dev_macro_f1"])}
    write_json(os.path.join(best, "SELECTION.json"), sel)
    log(f"   BEST of {len(elig)} -> {win['job']} ({win['technique']}), dev Macro-F1 "
        f"{win['dev_macro_f1']:.4f}")
    return sel


def coverage(out_root: str, input_roots, log=print) -> Dict:
    folders = {v["folder"]: k for k, v in EXPERIMENTS.items()}
    found: Dict[str, set] = {}
    for exp_f, _job, run_dir in iter_finished_runs(out_root, input_roots):
        found.setdefault(exp_f, set()).add(os.path.dirname(run_dir))
    winners, per_model = {}, {}
    for folder, exp in folders.items():
        best = None
        for d in sorted(found.get(folder, [])):
            for r in _rows(d):
                if not r["eligible"]:
                    continue
                r["exp"] = exp
                r["zip"] = os.path.join(r["dir"], "submission", "submission.zip")
                if best is None or _key(r) > _key(best):
                    best = r
                pm = per_model.get(r["model"])
                if pm is None or _key(r) > _key(pm):
                    per_model[r["model"]] = r
        if best:
            winners[exp] = best
    log("Winners (one upload per notebook):")
    for exp, spec in EXPERIMENTS.items():
        w = winners.get(exp)
        log(f"   {exp:>4} {spec['title'][:40]:<40} " +
            (f"{w['model']} dev F1 {w['dev_macro_f1']:.4f} -> {w['zip']}" if w else "-"))
    techs = {winners[e]["technique"] for e in winners if EXPERIMENTS[e]["counts"]}
    peft = sorted(t for t in techs if TECHNIQUES.get(t, {}).get("peft"))
    rules = SPEC["rules"]
    n_exp = sum(1 for e in winners if EXPERIMENTS[e]["counts"])
    compliance = {
        "experiments_counted": n_exp, "min_experiments": rules["min_experiments"],
        "peft_techniques": peft, "min_peft_techniques": rules["min_peft_techniques"],
        "lorma_done": "lorma" in techs, "kd_done": "kd" in techs,
        "ok": (n_exp >= rules["min_experiments"] and len(peft) >= rules["min_peft_techniques"]
               and "lorma" in techs and "kd" in techs)}
    log(f"\nRule check: {compliance}")
    used = {w["model"] for w in winners.values()}
    extra = []
    missing = [m for m in MODELS if m not in used]
    if missing:
        log(f"Models not represented among the winners: {missing}. Also upload:")
        for m in missing:
            r = per_model.get(m)
            if r:
                log(f"   {m}: best run {r['exp']}/{r['job']} (dev F1 {r['dev_macro_f1']:.4f}) -> "
                    f"{r['zip']}")
                extra.append({"model": m, "exp": r["exp"], "zip": r["zip"]})
            else:
                log(f"   {m}: no finished run anywhere yet")
    n = len(winners) + len(extra)
    log(f"\nPlanned uploads: {n} (max {SPEC['submissions']['max_per_day']} per day, "
        f"{SPEC['submissions']['max']} per phase; scoring takes 15-30 min each)")
    return {"winners": winners, "extra_uploads": extra, "compliance": compliance}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--exp", default=None)
    ap.add_argument("--out", default=OUT_ROOT)
    ap.add_argument("--coverage", action="store_true")
    a = ap.parse_args()
    if a.coverage:
        write_json(os.path.join(a.out, "coverage.json"), coverage(a.out, default_input_roots()))
        return
    folder = EXPERIMENTS.get(a.exp, {}).get("folder", a.exp)
    select(os.path.join(a.out, folder))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase3/rr_aggregate.py
"""
rr_aggregate.py -- the report tables and README_PHASE3.md, generated from the run outputs.

    python rr_aggregate.py [--out /kaggle/working/phase3_out] [--codabench scores.json]

Reads every phase3_out/<exp>/<job>/DONE.json (here and in attached inputs) and writes:
  RESULTS_ALL.json / results_all.csv   one row per finished job
  REPORT_TABLES.md   technique x model Macro-F1 (pooled, CL, IT); per-class F1 + confusion of
                     the best model; context / head / loss / domain ablations (E10); PEFT vs
                     full-FT gap; KD teacher vs student; efficiency (params, memory, speed)
  README_PHASE3.md   architecture, methodology, how to run, results, rule compliance, citations
`scores.json` (optional, filled by hand): {"E1/M4_bloom_alibi": 0.61, ...} Codabench scores.
"""
import argparse
import csv
import os
from typing import Dict, List, Optional

from rr_common import (EXPERIMENTS, MODELS, OUT_ROOT, SPEC, TECHNIQUES, default_input_roots,
                       iter_finished_runs, read_json, write_json)

FOLDER_TO_EXP = {v["folder"]: k for k, v in EXPERIMENTS.items()}
MODEL_FOLDERS = {v["folder"]: k for k, v in MODELS.items()}

CITATIONS = [
    "Hu et al. (2021). LoRA: Low-Rank Adaptation of Large Language Models. arXiv:2106.09685.",
    "Dettmers et al. (2023). QLoRA: Efficient Finetuning of Quantized LLMs. NeurIPS. "
    "arXiv:2305.14314.",
    "Zhang et al. (2023). Adaptive Budget Allocation for Parameter-Efficient Fine-Tuning "
    "(AdaLoRA). ICLR. arXiv:2303.10512.",
    "Kalajdzievski (2023). A Rank Stabilization Scaling Factor for Fine-Tuning with LoRA "
    "(rsLoRA). arXiv:2312.03732.",
    "Bihany, Patel & Modi (2025). LoRMA: Low-Rank Multiplicative Adaptation for LLMs. "
    "arXiv:2506.07621.",
    "Li & Liang (2021). Prefix-Tuning: Optimizing Continuous Prompts for Generation. ACL. "
    "arXiv:2101.00190.",
    "Lester, Al-Rfou & Constant (2021). The Power of Scale for Parameter-Efficient Prompt "
    "Tuning. EMNLP. arXiv:2104.08691.",
    "Liu et al. (2022). Few-Shot Parameter-Efficient Fine-Tuning is Better and Cheaper than "
    "In-Context Learning (IA3). NeurIPS. arXiv:2205.05638.",
    "Hinton, Vinyals & Dean (2015). Distilling the Knowledge in a Neural Network. "
    "arXiv:1503.02531.",
    "Lin et al. (2017). Focal Loss for Dense Object Detection. ICCV (focal loss ablation).",
    "Menon et al. (2021). Long-tail learning via logit adjustment. ICLR (logit-adjust ablation).",
]


def abbrev(label: str) -> str:
    """Unique short column names: capitals of CamelCase labels (PrecedentNotReliedUpon -> PNRU)."""
    caps = "".join(c for c in label if c.isupper())
    return caps if len(caps) > 1 else label[:5]


def fmt(v) -> str:
    if isinstance(v, bool):
        return "yes" if v else "no"
    if isinstance(v, float):
        return f"{v:.4f}" if abs(v) < 10 else f"{v:,.1f}"
    if isinstance(v, int):
        return f"{v:,}"
    return "" if v is None else str(v)


def table(cols: List[str], rows: List[List]) -> List[str]:
    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r in rows:
        out.append("| " + " | ".join(fmt(x) for x in r) + " |")
    return out


def collect(out_root: str, roots: List[str]) -> List[Dict]:
    rows = []
    for exp_f, job_f, run_dir in iter_finished_runs(out_root, roots):
        d = read_json(os.path.join(run_dir, "DONE.json"))
        e = d.get("efficiency") or {}
        pd = d.get("dev_per_domain_macro_f1") or {}
        rows.append({"exp": FOLDER_TO_EXP.get(exp_f, exp_f), "job": job_f, "dir": run_dir,
                     "technique": d.get("technique"), "model": d.get("model"),
                     "variant": d.get("variant"), "dev_macro_f1": d.get("dev_macro_f1"),
                     "dev_cl": pd.get("CL"), "dev_it": pd.get("IT"),
                     "dev_domain_avg": d.get("dev_domain_avg_macro_f1"),
                     "dev_p": d.get("dev_macro_precision"), "dev_r": d.get("dev_macro_recall"),
                     "bias_applied": d.get("bias_applied"),
                     "submission_valid": d.get("submission_valid"),
                     "total_params": e.get("total_params"),
                     "trainable_params": e.get("trainable_params"),
                     "trainable_pct": e.get("trainable_pct"),
                     "train_minutes": e.get("train_minutes"),
                     "train_peak_mem_gb": e.get("train_peak_mem_gb"),
                     "train_tok_s": e.get("train_tokens_per_s"),
                     "infer_tok_s": e.get("infer_tokens_per_s"),
                     "infer_peak_mem_gb": e.get("infer_peak_mem_gb"),
                     "kd_report": d.get("kd_report"), "peft_report": d.get("peft_report")})
    rows.sort(key=lambda r: (r["exp"], r["job"]))
    return rows


def build_report(out_root: str, roots: List[str], codabench: Optional[str] = None,
                 log=print) -> Dict:
    rows = collect(out_root, roots)
    if not rows:
        log("   no finished runs found -- nothing to aggregate")
        return {}
    cb = read_json(codabench, {}) if codabench else {}
    for r in rows:
        r["codabench"] = cb.get(f"{r['exp']}/{r['job']}")
    write_json(os.path.join(out_root, "RESULTS_ALL.json"), rows)
    flat = [{k: v for k, v in r.items() if k not in ("kd_report", "peft_report")} for r in rows]
    with open(os.path.join(out_root, "results_all.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(flat[0].keys()))
        w.writeheader()
        w.writerows(flat)

    main = [r for r in rows if r["job"] in MODEL_FOLDERS and r["exp"] in EXPERIMENTS
            and EXPERIMENTS[r["exp"]]["technique"] not in ("ablation", "ensemble")]
    L: List[str] = ["# Phase-3 report tables (generated by rr_aggregate.py)", ""]
    # 1. technique x model
    L += ["## Dev Macro-F1 per technique and model (pooled / CL / IT)", ""]
    techs = [e for e in EXPERIMENTS if EXPERIMENTS[e]["technique"] not in ("ablation",
                                                                           "ensemble")]
    trs = []
    for e in techs:
        cells = [e, EXPERIMENTS[e]["title"]]
        for mid in MODELS:
            r = next((x for x in main if x["exp"] == e and x["model"] == mid), None)
            cells.append(f"{r['dev_macro_f1']:.4f} / {r['dev_cl'] or 0:.3f} / "
                         f"{r['dev_it'] or 0:.3f}" if r else "")
        trs.append(cells)
    L += table(["exp", "technique"] + [f"{m} ({MODELS[m]['arch']})" for m in MODELS], trs) + [""]
    # 2. best model: per-class F1 + confusion
    elig = [r for r in main if r["submission_valid"] and EXPERIMENTS[r["exp"]]["counts"]]
    best = max(elig, key=lambda r: r["dev_macro_f1"]) if elig else None
    if best:
        rep = read_json(os.path.join(best["dir"], "dev", "dev_report.json"), {})
        m = (rep.get("metrics") or {})
        per = (m.get("pooled") or {}).get("per_class", {})
        L += [f"## Best counted model: {best['exp']} / {best['job']} (dev pooled Macro-F1 "
              f"{best['dev_macro_f1']:.4f})", "", "Per-class dev scores:", ""]
        L += table(["label", "P", "R", "F1", "support", "predicted"],
                   [[k, v["precision"], v["recall"], v["f1"], v["support"], v["predicted"]]
                    for k, v in per.items()]) + [""]
        if m.get("confusion"):
            labs = m["labels"]
            L += ["Confusion matrix (rows gold, columns predicted):", ""]
            L += table(["gold \\ pred"] + [abbrev(l) for l in labs],
                       [[labs[i]] + list(row) for i, row in enumerate(m["confusion"])]) + [""]
    # 3. ablations
    abl = [r for r in rows if r["exp"] == "E10"]
    if abl:
        L += ["## Ablations (E10)", ""]
        L += table(["job", "technique", "model", "variant", "pooled F1", "CL", "IT"],
                   [[r["job"], r["technique"], r["model"], r["variant"], r["dev_macro_f1"],
                     r["dev_cl"], r["dev_it"]] for r in abl]) + [""]
    # 4. PEFT vs full FT
    ft = {r["model"]: r["dev_macro_f1"] for r in main if r["technique"] == "full_ft"}
    if ft:
        L += ["## PEFT vs full fine-tuning gap (dev pooled Macro-F1, technique - full FT)", ""]
        gap_rows = []
        for e in techs:
            if EXPERIMENTS[e]["technique"] in ("full_ft",):
                continue
            cells = [e, EXPERIMENTS[e]["technique"]]
            for mid in MODELS:
                r = next((x for x in main if x["exp"] == e and x["model"] == mid), None)
                cells.append(f"{r['dev_macro_f1'] - ft[mid]:+.4f}" if r and mid in ft else "")
            gap_rows.append(cells)
        L += table(["exp", "technique"] + list(MODELS), gap_rows) + [""]
    # 5. KD
    kd = [r for r in main if r["kd_report"]]
    if kd:
        L += ["## Knowledge distillation: teacher (full FT) vs pruned student", ""]
        L += table(["model", "teacher F1", "student F1", "F1 retained", "teacher params",
                    "student params", "size ratio", "speed-up", "memory ratio"],
                   [[r["model"], r["kd_report"]["teacher_dev_macro_f1"],
                     r["kd_report"]["student_dev_macro_f1"], r["kd_report"]["f1_retained"],
                     r["kd_report"]["teacher_params"], r["kd_report"]["student_params"],
                     r["kd_report"]["size_ratio"], r["kd_report"]["speedup"],
                     r["kd_report"].get("memory_ratio")] for r in kd]) + [""]
    # 6. efficiency
    L += ["## Efficiency", ""]
    L += table(["exp", "job", "technique", "total params", "trainable", "trainable %",
                "train min", "train peak GB", "train tok/s", "infer tok/s"],
               [[r["exp"], r["job"], r["technique"], r["total_params"], r["trainable_params"],
                 r["trainable_pct"], r["train_minutes"], r["train_peak_mem_gb"], r["train_tok_s"],
                 r["infer_tok_s"]] for r in main]) + [""]
    # 7. AdaLoRA ranks
    ada = [r for r in main if (r["peft_report"] or {}).get("adalora_rank_per_layer")]
    for r in ada:
        L += [f"AdaLoRA final rank per layer ({r['model']}): "
              f"{r['peft_report']['adalora_rank_per_layer']}", ""]
    # 8. compliance
    from rr_select import coverage
    cov = coverage(out_root, roots, log=lambda *x: None)
    L += ["## Rule compliance", "", f"`{cov['compliance']}`", ""]
    with open(os.path.join(out_root, "REPORT_TABLES.md"), "w") as f:
        f.write("\n".join(L) + "\n")
    write_readme(out_root, L, cov)
    log(f"   report written: {os.path.join(out_root, 'REPORT_TABLES.md')} and README_PHASE3.md "
        f"({len(rows)} finished jobs)")
    return {"rows": len(rows), "best": best["job"] if best else None}


def write_readme(out_root: str, tables: List[str], cov: Dict) -> None:
    R = ["# CS-781 Capstone -- Phase 3: Rhetorical Role Prediction", "",
         "Sentence-level rhetorical-role classification of Indian legal judgments (Corporate Law "
         "and IT domains, 13 labels) with the four Phase-1 foundation models (M1 GPT-2 style, M2 "
         "Llama style, M3 Mistral style with sliding-window attention, M4 BLOOM style with ALiBi) "
         "and parameter-efficient fine-tuning implemented from scratch in plain PyTorch.", "",
         "## Architecture", "",
         "* `RRModel` = Phase-1 trunk (embeddings -> blocks -> final norm, LM head dropped) + "
         "sentence head. Documents are packed into <= 1024-token windows; every sentence ends "
         "with the EOS token, whose hidden state is the sentence vector; a domain prefix "
         "(`[CL]`/`[IT]`) starts each window. Long documents use overlapping windows; each "
         "sentence is predicted from the window where it has the most context.",
         "* Heads: LayerNorm -> Dropout -> Linear(768, 13) (default); a BiGRU over the sentence "
         "sequence (`seqctx`) in the ablation.",
         "* Loss: inverse-sqrt-frequency weighted cross-entropy (Macro-F1 punishes ignoring rare "
         "labels); focal / logit-adjusted / plain CE in the ablation.",
         "* Post-hoc per-class logit biases, applied only when a 2-fold held-out check on dev "
         "shows a gain.", "",
         "## Techniques", ""]
    for t, v in TECHNIQUES.items():
        R.append(f"* **{t}** -- {v['title']}" + ("" if v["peft"] or t == "kd"
                                                 else " (reference, not counted)"))
    R += ["", "rsLoRA shares the LoRA module (only the scale alpha/sqrt(r) and the rank differ) "
          "but is listed by the brief as its own technique, so it runs as its own experiment.",
          "QLoRA is simulated without bitsandbytes: block-wise NF4 storage (block 64, double-"
          "quantised scales) with on-the-fly de-quantisation.", "",
          "## How to run", "",
          "1. `00_data_tests_pilot`: data gates (G0-G2), CPU selftest, causal-vs-bidir pilot -> "
          "`PILOT.json`.",
          "2. E1, E2, E6 (priority 1) then E3, E4, E5 (priority 2), then E7, E8, E9, E0, E10 "
          "(priority 3) -- each is an independent Kaggle notebook (T4 x2): "
          "`python run_experiment.py --exp E<k>`.",
          "3. E11: `python run_experiment.py --exp E11` (ensemble + this report) with the other "
          "outputs attached.",
          "4. Upload `phase3_out/<exp>/BEST/submission.zip` for each experiment.", "",
          "## Results", ""]
    R += tables[2:]
    R += ["", "## Rule compliance", "", f"* experiments counted: "
          f"{cov['compliance']['experiments_counted']} (min {SPEC['rules']['min_experiments']})",
          f"* PEFT techniques: {cov['compliance']['peft_techniques']} (min "
          f"{SPEC['rules']['min_peft_techniques']}; LoRMA done: {cov['compliance']['lorma_done']})",
          f"* KD experiment done: {cov['compliance']['kd_done']}",
          "* every experiment starts from a Phase-1 model; no external weights, corpora or APIs;"
          " no transformers / peft / bitsandbytes / sklearn.", "", "## References", ""]
    R += [f"* {c}" for c in CITATIONS]
    with open(os.path.join(out_root, "README_PHASE3.md"), "w") as f:
        f.write("\n".join(R) + "\n")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=OUT_ROOT)
    ap.add_argument("--codabench", default=None)
    a = ap.parse_args()
    build_report(a.out, default_input_roots(), a.codabench)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase3/rr_ensemble.py
"""
rr_ensemble.py -- E11 ensemble over the saved logits of every BEST model, and the per-domain
combination used by the E10 ablation (separate CL and IT models -> one submission).

    python rr_ensemble.py [--out /kaggle/working/phase3_out]

Members: every <exp>/BEST (E0..E9) found under --out or under an attached `phase3_out` folder,
with raw dev/test logits saved by rr_predict (no model is re-run). Logits are averaged after
log-softmax normalisation (= geometric mean of the probabilities; raw logits of different
techniques live on different scales). Candidates: uniform over all members, dev-F1-weighted
over all members, uniform over the top-3 by dev F1; the best on dev is kept. CAUTION: dev is
used both to pick the members/weights and to report, so the ensemble's dev score is optimistic.
The usual 2-fold held-out check decides whether post-hoc biases are applied.

Inputs : BEST/logits/{dev,test}_logits.npy + *_index.json, BEST/DONE.json, data cache
Outputs: E11_ensemble/ensemble/{DONE.json, dev/, submission/, logits/}, E11_ensemble/BEST/
"""
import argparse
import os
from typing import Dict, List, Optional

import numpy as np

from rr_common import (DATA_DIR, EXIT, EXPERIMENTS, OUT_MARKER, OUT_ROOT, default_input_roots,
                       find_files, read_json, write_json)
from rr_data import load_cache
from rr_eval import full_report, headline, labels_from_logits
from rr_predict import done_record, finalize_from_logits, load_logits


def _log_softmax(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float64)
    m = x.max(1, keepdims=True)
    return x - m - np.log(np.exp(x - m).sum(1, keepdims=True))


def find_best_dirs(out_root: str, roots: List[str]) -> Dict[str, str]:
    """{exp_id: BEST dir} for E0..E9 (local output first, then attached inputs)."""
    found = {}
    for exp, spec in EXPERIMENTS.items():
        if spec["technique"] in ("ablation", "ensemble"):
            continue
        cands = [os.path.join(out_root, spec["folder"], "BEST")]
        for s in find_files("SELECTION.json", roots, max_hits=5000):
            parts = s.split(os.sep)
            if OUT_MARKER in parts and spec["folder"] in parts and parts[-2] == "BEST":
                cands.append(os.path.dirname(s))
        for c in cands:
            if os.path.exists(os.path.join(c, "logits", "dev_logits.npy")) and \
                    os.path.exists(os.path.join(c, "DONE.json")):
                found[exp] = c
                break
    return found


def average(members: List[Dict], weights: Optional[List[float]] = None) -> Dict[str, np.ndarray]:
    w = np.asarray(weights if weights is not None else [1.0] * len(members), dtype=np.float64)
    w = w / w.sum()
    keys = members[0].keys()
    return {k: sum(wi * _log_softmax(m[k]) for wi, m in zip(w, members)).astype(np.float32)
            for k in keys}


def run_ensemble(out_root: str, roots: List[str], exp_dir: str, data_dir: str, log=print) -> int:
    meta, docs = load_cache(data_dir)
    best = find_best_dirs(out_root, roots)
    log(f"   ensemble candidates: {sorted(best)}")
    mem = []
    for exp, d in sorted(best.items()):
        dev, test = load_logits(os.path.join(d, "logits"), "dev"), \
            load_logits(os.path.join(d, "logits"), "test")
        done = read_json(os.path.join(d, "DONE.json"))
        if dev is None or test is None or set(test) != set(meta["split"]["test"]):
            log(f"   {exp}: incomplete logits, skipped")
            continue
        mem.append({"exp": exp, "dir": d, "dev": dev, "test": test, "f1": done["dev_macro_f1"],
                    "model": done.get("model"), "technique": done.get("technique")})
    if len(mem) < 2:
        log("   fewer than 2 BEST models with saved logits -- nothing to ensemble")
        return EXIT["missing_dependency"]
    dev_keys = sorted(mem[0]["dev"])
    gold = {k: docs[k]["labels"] for k in dev_keys}
    dom = {k: docs[k]["domain"] for k in dev_keys}
    order = sorted(range(len(mem)), key=lambda i: -mem[i]["f1"])
    cands = {"uniform_all": (list(range(len(mem))), None),
             "f1_weighted_all": (list(range(len(mem))), [m["f1"] for m in mem]),
             "uniform_top3": (order[:3], None)}
    scores = {}
    for name, (idx, w) in cands.items():
        dev = average([mem[i]["dev"] for i in idx], w)
        scores[name] = headline(full_report(gold, labels_from_logits(dev), dom, meta["labels"]))
        log(f"   {name:<16} members {[mem[i]['exp'] for i in idx]} -> dev pooled Macro-F1 "
            f"{scores[name]:.4f}")
    pick = max(scores, key=lambda n: (scores[n], n == "uniform_all"))
    idx, w = cands[pick]
    dev = average([mem[i]["dev"] for i in idx], w)
    test = average([mem[i]["test"] for i in idx], w)
    run_dir = os.path.join(exp_dir, "ensemble")
    fin = finalize_from_logits(dev, test, meta, docs, run_dir, log)
    cfg = {"exp": "E11", "job": "ensemble", "model": "+".join(mem[i]["model"] for i in idx),
           "technique": "ensemble", "variant": pick}
    done = done_record(cfg, fin, {"members": [{k: mem[i][k] for k in ("exp", "dir", "f1", "model",
                                                                     "technique")} for i in idx],
                                  "weights": w, "candidates": scores,
                                  "efficiency": {"trainable_params": 0},
                                  "note": "dev score is optimistic (dev used for selection)"})
    write_json(os.path.join(run_dir, "DONE.json"), done)
    from rr_select import select
    select(exp_dir, log=log)
    return EXIT["ok"] if fin["validation"] and fin["validation"]["ok"] else \
        EXIT["validation_failure"]


def combine_domains(src_dirs: List[str], run_dir: str, data_dir: str, job: Dict,
                    log=print) -> Dict:
    """E10 per-domain variant: CL-only and IT-only runs -> one pooled report + submission."""
    meta, docs = load_cache(data_dir)
    dev, test, eff = {}, {}, {"trainable_params": 0, "total_params": 0, "train_minutes": 0.0}
    for d in src_dirs:
        dv, ts = load_logits(os.path.join(d, "logits"), "dev"), \
            load_logits(os.path.join(d, "logits"), "test")
        if dv is None or ts is None:
            raise FileNotFoundError(f"no saved logits in {d}")
        dev.update(dv)
        test.update(ts)
        e = (read_json(os.path.join(d, "DONE.json")) or {}).get("efficiency") or {}
        for k in eff:
            eff[k] += e.get(k) or 0
    fin = finalize_from_logits(dev, test, meta, docs, run_dir, log)
    first = read_json(os.path.join(src_dirs[0], "logs", "run_config.json")) or {}
    cfg = {**{k: first.get(k) for k in ("exp", "model", "arch", "technique", "input_mode",
                                        "head", "attn_mode", "loss")},
           "job": os.path.basename(run_dir), "variant": job.get("variant", "per_domain"),
           "domains": ["CL", "IT"]}
    done = done_record(cfg, fin, {"sources": src_dirs, "efficiency": eff})
    write_json(os.path.join(run_dir, "DONE.json"), done)
    return done


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=OUT_ROOT)
    ap.add_argument("--data-dir", default=DATA_DIR)
    a = ap.parse_args()
    exp_dir = os.path.join(a.out, EXPERIMENTS["E11"]["folder"])
    raise SystemExit(run_ensemble(a.out, default_input_roots(), exp_dir, a.data_dir))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase3/run_experiment.py
"""
run_experiment.py -- one notebook = one technique x all four Phase-1 models, as a per-GPU job
queue (two models train in parallel on a T4 x2, one subprocess per job and GPU).

    python run_experiment.py --exp E1                     # LoRMA+ on M4, M2, M3, M1
    python run_experiment.py --exp E6                     # teacher_M* (full FT) -> students
    python run_experiment.py --exp E6 --teacher-dir /kaggle/input/.../phase3_out/E0_full_ft
    python run_experiment.py --exp E2 --models M4,M2      # split a notebook across accounts
    python run_experiment.py --exp P0                     # causal-vs-bidir pilot -> PILOT.json
    python run_experiment.py --exp E10 --models M4 --ablation-techniques lorma,lora
    python run_experiment.py --exp E11                    # ensemble + final report (CPU)

Flow: data cache + gates G0-G2 (exit 3 on failure) -> CPU selftest (once per notebook) ->
attention mode (PILOT.json if attached, else bidir) -> GPU sanity check on every GPU (a GPU
that fails is dropped; fallback to one GPU) -> job queue. Per job: [restore earlier outputs]
-> smoke (real-model self-checks + throughput -> per-job time budget) -> overfit-32 (first job
of the notebook) -> LR probe x{1/3,1,3} -> train (30-min checkpoints, resume) -> predict (dev
report, bias decision, validated submission.zip, raw logits) -> DONE.json. After every job and
at the end: best-of-4 selection into BEST/ (valid even if the 11 h budget is hit), RESULTS.json,
EFFICIENCY.json, INDEX.md.

Exit codes: 0 ok | 2 missing dependency (Phase-1 weights, data, KD teacher) | 3 gate or
selftest failure | 4 submission validation failure | 1 other error / nothing finished.
This process never touches the GPU itself: every GPU job is a subprocess, so a crash or a CUDA
error in one job cannot take down the notebook.
"""
import argparse
import json
import os
import shutil
import signal
import subprocess
import sys
import threading
import time
from typing import Dict, List, Optional, Tuple

from rr_common import (DATA_DIR, DEFAULT_EPOCHS, DEFAULT_LR, DEFAULT_MODEL_ORDER, EXIT,
                       EXPERIMENTS, MODELS, OUT_MARKER, OUT_ROOT, PILOT_FOLDER, SPEC,
                       DiskFull, GateFailure, default_input_roots, dir_size_gb, disk_guard,
                       find_files, find_phase1_models, free_gb, module_md5s, read_json,
                       student_layers, write_json)

HERE = os.path.dirname(os.path.abspath(__file__))
TRAIN = os.path.join(HERE, "rr_train.py")
PREDICT = os.path.join(HERE, "rr_predict.py")
SELFTEST = os.path.join(HERE, "selftest.py")

# E10: variants applied to each ablation technique (doc + linear + weighted_ce = baseline)
ABLATIONS: Dict[str, Dict] = {
    "base": {},
    "sent": {"input_mode": "sent"},
    "ctx1": {"input_mode": "ctx_1"},
    "seqctx": {"head": "seqctx"},
    "ce": {"loss": "ce"},
    "focal": {"loss": "focal"},
    "logitadj": {"loss": "logit_adjust"},
    "onlyCL": {"domains": ["CL"]},
    "onlyIT": {"domains": ["IT"]},
}
EFF_BATCH = {"doc": 8, "sent": 64, "ctx": 32}          # windows per optimizer step
MICRO_BATCH = {"doc": 4, "sent": 32, "ctx": 16}         # start value of the memory probe


# =============================================================================================
# subprocess helpers
# =============================================================================================
_print_lock = threading.Lock()


def run_cmd(cmd: List[str], log_path: str, env: Optional[Dict] = None,
            timeout: Optional[float] = None, prefix: str = "") -> Tuple[int, bool]:
    """Stream a subprocess's output (prefixed) to the notebook and a log file, with a hard
    wall-clock timeout. Returns (exit_code, timed_out); exit_code is -9 on timeout."""
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    full_env = dict(os.environ)
    full_env["PYTHONPATH"] = HERE + os.pathsep + full_env.get("PYTHONPATH", "")
    full_env.setdefault("PYTHONUNBUFFERED", "1")
    full_env.setdefault("NCCL_IB_DISABLE", "1")
    full_env.setdefault("TORCH_NCCL_ASYNC_ERROR_HANDLING", "1")
    full_env.setdefault("OMP_NUM_THREADS", "2")
    if env:
        full_env.update(env)
    lf = open(log_path, "a", encoding="utf-8")
    lf.write(f"\n$ {' '.join(cmd)}\n")
    lf.flush()
    p = subprocess.Popen(cmd, cwd=HERE, env=full_env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)

    def pump():
        for line in p.stdout:
            with _print_lock:
                print(prefix + line, end="", flush=True)
            lf.write(line)
            lf.flush()

    th = threading.Thread(target=pump, daemon=True)
    th.start()
    timed_out = False
    try:
        code = p.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        timed_out = True
        try:
            os.killpg(p.pid, signal.SIGKILL)
        except OSError:
            pass
        p.wait()
        code = -9
    th.join(timeout=10)
    lf.close()
    return code, timed_out


def env_info() -> Dict:
    code = ("import json, torch, sys; print(json.dumps({'python': sys.version.split()[0], "
            "'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpus': "
            "[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}))")
    try:
        out = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True,
                             timeout=180)
        return json.loads(out.stdout.strip().splitlines()[-1])
    except Exception as e:
        return {"error": str(e)}


def gpu_sanity(gpu: str, timeout: int = 180) -> bool:
    """A small CUDA matmul + backward on one GPU (the 'smoke test on both GPUs')."""
    code = ("import torch; x = torch.randn(2048, 2048, device='cuda', requires_grad=True); "
            "y = (x.half() @ x.half()).float().sum(); y.backward(); "
            "torch.cuda.synchronize(); print('ok', torch.cuda.get_device_name(0))")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu)
    try:
        out = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True,
                             timeout=timeout, env=env)
        return out.returncode == 0 and "ok" in out.stdout
    except Exception:
        return False


def train_cmd(run_dir: str, mode: str, nproc: int = 1, extra=()) -> List[str]:
    base = ["--run-dir", run_dir, "--mode", mode, *extra]
    if nproc > 1:
        return [sys.executable, "-m", "torch.distributed.run", "--standalone",
                "--nproc_per_node", str(nproc), TRAIN, *base]
    return [sys.executable, TRAIN, *base]


# =============================================================================================
# configs and jobs
# =============================================================================================
def make_run_config(a, exp: str, folder: str, job: Dict, meta: Dict, data_dir: str,
                    phase1_dir: str, attn: str, env: Dict) -> Dict:
    technique, mid = job["technique"], job["model"]
    full = technique in ("full_ft", "kd")
    lr = (a.lr if (a.lr and not job.get("is_teacher")) else None) or DEFAULT_LR[technique]
    cfg = {
        "exp": exp, "exp_folder": folder, "job": job["folder"], "model": mid,
        "arch": MODELS[mid]["arch"], "kind": MODELS[mid]["kind"], "technique": technique,
        "phase1_dir": phase1_dir, "data_dir": data_dir, "labels": meta["labels"],
        "n_labels": len(meta["labels"]), "attn_mode": attn, "head": a.head, "pool": a.pool,
        "input_mode": a.input_mode, "max_len": SPEC["context"],
        "max_sent_tokens": a.max_sent_tokens, "train_stride_frac": 0.75,
        "infer_stride_frac": 0.5, "window_combine": a.window_combine, "domain_prefix": True,
        "domains": list(SPEC["domains"]), "loss": a.loss, "focal_gamma": 2.0,
        "logit_adjust_tau": 1.0, "lr": lr, "head_lr": 10 * lr if full else max(1e-3, lr),
        "weight_decay": 0.01, "warmup_frac": 0.06, "schedule": "cosine", "grad_clip": 1.0,
        "dropout": 0.1,
        "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.05, "lorma_r": 16, "lorma_alpha": 32,
        "rslora_r": 32, "rslora_alpha": 16,
        "adalora_init_r": 12, "adalora_target_r": 8, "adalora_alpha": 32,
        "adalora_tinit_frac": 0.1, "adalora_tfinal_frac": 0.2, "adalora_delta_t": 10,
        "adalora_beta1": 0.85, "adalora_beta2": 0.85, "adalora_orth": 0.1,
        "qlora_r": 16, "qlora_alpha": 32, "qlora_block": 64, "qlora_double_quant": True,
        "prefix_len": 20, "prefix_reparam": True, "prefix_hidden": 256, "prompt_len": 20,
        "kd_alpha": 0.5, "kd_T": 2.0, "kd_hidden_weight": a.kd_hidden_weight,
        "teacher_dir": job.get("teacher_dir"),
        "layer_ids": student_layers(mid, phase1_dir) if technique == "kd" else None,
        "epochs": (a.epochs if (a.epochs and not job.get("is_teacher")) else None)
        or DEFAULT_EPOCHS[technique],
        "patience": a.patience, "micro_batch": a.micro_batch, "eval_batch": 16, "seed": 42,
        "ckpt_every_min": a.ckpt_every_min, "deadline_unix": 0, "deadline_margin_s": 300,
        "watchdog_min_f1": a.watchdog_min_f1, "log_every": 25, "probe_steps": a.probe_steps,
        "bias_tuning": True, "variant": job.get("variant"), "refit": False,
        "skip_job_checks": a.skip_job_checks,
        "module_md5": module_md5s(HERE), "environment": env,
    }
    over = job.get("overrides") or {}
    cfg.update(over)
    mode = cfg["input_mode"].split("_")[0]
    if "effective_batch" not in over:
        cfg["effective_batch"] = a.effective_batch or EFF_BATCH[mode]
    if "micro_batch" not in over:
        cfg["micro_batch"] = a.micro_batch or MICRO_BATCH[mode]
    cfg["n_virtual"] = int(cfg["prompt_len"]) if technique == "prompt" else 0
    return cfg


def make_jobs(a, exp: str, technique: str, models: List[str], exp_dir: str) -> List[Dict]:
    def job(mid, folder, tech, **kw):
        return {"model": mid, "folder": folder, "technique": tech, "kind": "train",
                "after": [], "overrides": {}, **kw}
    jobs: List[Dict] = []
    if exp == "P0":
        for mode in ("causal", "bidir"):
            jobs.append(job("M2", f"lora_{mode}_M2", "lora", overrides={"attn_mode": mode,
                                                                        "epochs": 1},
                            skip_probe=True, skip_overfit=True, variant=f"pilot_{mode}"))
    elif technique == "kd":
        teachers, students = [], []
        for mid in models:
            sfolder = MODELS[mid]["folder"]
            if a.teacher_dir:
                students.append(job(mid, sfolder, "kd", teacher_dir=a.teacher_dir))
            else:
                tfolder = "teacher_" + sfolder
                teachers.append(job(mid, tfolder, "full_ft", is_teacher=True,
                                    variant="kd_teacher"))
                students.append(job(mid, sfolder, "kd", after=[tfolder],
                                    teacher_dir=os.path.join(exp_dir, tfolder)))
        if teachers:                             # interleave: 2 teachers, their students, ...
            for i in range(0, len(models), 2):
                jobs += teachers[i:i + 2] + students[i:i + 2]
        else:
            jobs = students
    elif technique == "ablation":
        for tech in [t.strip() for t in a.ablation_techniques.split(",") if t.strip()]:
            for mid in models:
                for vname, ov in ABLATIONS.items():
                    jobs.append(job(mid, f"{tech}_{vname}_{mid}", tech, overrides=dict(ov),
                                    variant=vname, skip_probe=True, skip_overfit=True))
                jobs.append({"model": mid, "folder": f"{tech}_perdomain_{mid}", "technique": tech,
                             "kind": "combine", "variant": "per_domain",
                             "after": [f"{tech}_onlyCL_{mid}", f"{tech}_onlyIT_{mid}"]})
    else:
        for mid in models:
            jobs.append(job(mid, MODELS[mid]["folder"], technique))
    return jobs


# =============================================================================================
# restore / pilot / index
# =============================================================================================
def restore_from_inputs(exp_folder: str, out_root: str, roots: List[str], log) -> None:
    """Copy finished (DONE.json) or resumable (last.pt) job folders of this experiment from
    attached datasets into the working output dir, unless already present."""
    for hit in find_files("DONE.json", roots, max_hits=5000) + \
            find_files("last.pt", roots, max_hits=5000):
        parts = hit.split(os.sep)
        if OUT_MARKER not in parts or exp_folder not in parts:
            continue
        i = parts.index(exp_folder)
        if i + 1 >= len(parts) - 1:
            continue
        job_folder = parts[i + 1]
        src = os.sep.join(parts[:i + 2])
        dst = os.path.join(out_root, exp_folder, job_folder)
        if job_folder == "BEST" or os.path.exists(os.path.join(dst, "DONE.json")):
            continue
        if os.path.exists(dst) and not os.path.exists(os.path.join(src, "DONE.json")):
            continue                                    # never overwrite local progress
        log(f"   restoring {src} -> {dst}")
        shutil.copytree(src, dst, dirs_exist_ok=True)


def resolve_attn_mode(a, out_root: str, roots: List[str], log) -> Tuple[str, str]:
    if a.attn_mode in ("causal", "bidir"):
        return a.attn_mode, "command line"
    cands = [os.path.join(out_root, "PILOT.json")] + find_files("PILOT.json", roots)
    for c in cands:
        p = read_json(c)
        if p and p.get("winner") in ("causal", "bidir"):
            return p["winner"], f"pilot {c}"
    log("   !!! no PILOT.json attached: using attention mode 'bidir' (Phase-2 default). Run "
        "the 00 notebook (or --exp P0) and attach its output to use the pilot's winner.")
    return "bidir", "default (no pilot)"


def write_pilot(out_root: str, exp_dir: str, log) -> Optional[Dict]:
    res = {}
    for mode in ("causal", "bidir"):
        d = read_json(os.path.join(exp_dir, f"lora_{mode}_M2", "DONE.json"))
        if d:
            res[mode] = {"dev_macro_f1": d["dev_macro_f1"],
                         "dev_per_domain": d.get("dev_per_domain_macro_f1"),
                         "train_minutes": (d.get("efficiency") or {}).get("train_minutes")}
    if len(res) < 2:
        log("   pilot incomplete -- PILOT.json not written")
        return None
    winner = max(res, key=lambda m: (res[m]["dev_macro_f1"], m == "bidir"))
    pilot = {"winner": winner, "results": res, "setup": "M2 + LoRA, 1 epoch each",
             "written": time.strftime("%Y-%m-%d %H:%M:%S")}
    write_json(os.path.join(out_root, "PILOT.json"), pilot)
    write_json(os.path.join(exp_dir, "PILOT.json"), pilot)
    log(f"   PILOT: causal {res['causal']['dev_macro_f1']:.4f} vs bidir "
        f"{res['bidir']['dev_macro_f1']:.4f} -> attention mode '{winner}' (PILOT.json)")
    return pilot


def write_index(out_root: str, exp_folder: str, records: List[Dict], note: str) -> None:
    st_path = os.path.join(out_root, "index_state.json")
    state = read_json(st_path, {})
    state[exp_folder] = {"records": records, "note": note,
                         "updated": time.strftime("%Y-%m-%d %H:%M:%S")}
    write_json(st_path, state)
    lines = ["# Phase-3 outputs (generated by run_experiment.py)", "",
             f"Output size: {dir_size_gb(out_root):.2f} GB | free disk: {free_gb(out_root):.1f} GB",
             ""]
    for ef in sorted(state):
        s = state[ef]
        lines += [f"## {ef}  (updated {s['updated']})", "",
                  "| job | gpu | status | dev Macro-F1 | CL | IT | minutes | note |",
                  "|---|---|---|---|---|---|---|---|"]
        for r in s["records"]:
            f = lambda k: f"{r[k]:.4f}" if isinstance(r.get(k), float) else ""  # noqa: E731
            mins = f"{r['minutes']:.1f}" if isinstance(r.get("minutes"), (int, float)) else ""
            lines.append(f"| {r['job']} | {r.get('gpu', '')} | {r['status']} | "
                         f"{f('dev_macro_f1')} | {f('cl')} | {f('it')} | {mins} | "
                         f"{r.get('note', '')} |")
        sel = read_json(os.path.join(out_root, ef, "BEST", "SELECTION.json"))
        if sel:
            lines += ["", f"**Upload:** `{ef}/BEST/submission.zip` (winner {sel['winner']}, "
                          f"dev Macro-F1 {sel['dev_macro_f1']:.4f})"]
        if s.get("note"):
            lines += ["", f"**Note:** {s['note']}"]
        lines.append("")
    lines += ["Save this session's output (Save Version) before it ends; attach it to the next "
              "session so finished jobs are skipped and unfinished ones resume.", ""]
    with open(os.path.join(out_root, "INDEX.md"), "w") as f:
        f.write("\n".join(lines))


def results_tables(exp_dir: str) -> None:
    """RESULTS.json (one row per finished job) and EFFICIENCY.json."""
    rows, eff = [], []
    for job in sorted(os.listdir(exp_dir)):
        d = read_json(os.path.join(exp_dir, job, "DONE.json"))
        if job == "BEST" or not d:
            continue
        e = d.get("efficiency") or {}
        rows.append({"job": job, "model": d.get("model"), "technique": d.get("technique"),
                     "variant": d.get("variant"), "dev_macro_f1": d["dev_macro_f1"],
                     "dev_macro_precision": d.get("dev_macro_precision"),
                     "dev_macro_recall": d.get("dev_macro_recall"),
                     "dev_per_domain_macro_f1": d.get("dev_per_domain_macro_f1"),
                     "dev_domain_avg_macro_f1": d.get("dev_domain_avg_macro_f1"),
                     "bias_applied": d.get("bias_applied"),
                     "submission_valid": d.get("submission_valid"),
                     "kd_report": d.get("kd_report"), "peft_report": d.get("peft_report"),
                     **{k: e.get(k) for k in ("total_params", "trainable_params",
                                              "trainable_pct", "train_minutes")}})
        eff.append({"job": job, "model": d.get("model"), "technique": d.get("technique"), **e})
    write_json(os.path.join(exp_dir, "RESULTS.json"), rows)
    write_json(os.path.join(exp_dir, "EFFICIENCY.json"), eff)


# =============================================================================================
# the job runner
# =============================================================================================
class Runner:
    def __init__(self, a, exp, folder, exp_dir, out_root, jobs, meta, data_dir, p1, attn, env,
                 deadline, say):
        self.a, self.exp, self.folder, self.exp_dir, self.out_root = a, exp, folder, exp_dir, out_root
        self.jobs, self.meta, self.data_dir, self.p1, self.attn = jobs, meta, data_dir, p1, attn
        self.env, self.deadline, self.say = env, deadline, say
        self.lock = threading.Lock()
        self.status: Dict[str, str] = {}          # folder -> pending|running|done|failed|skipped
        self.records: Dict[str, Dict] = {}
        self.overfit_claimed = a.skip_overfit
        self.validation_failed = False
        self.missing_dependency = False
        for j in jobs:
            self.status[j["folder"]] = "pending"
            self.records[j["folder"]] = {"job": j["folder"], "status": "pending"}

    # ---------------------------------------------------------------- queue
    def _next(self) -> Optional[Dict]:
        """Next runnable job (deps done), None if nothing left, 'wait' if blocked."""
        with self.lock:
            waiting = False
            for j in self.jobs:
                if self.status[j["folder"]] != "pending":
                    continue
                deps = [self.status.get(d, "done" if self._done(d) else "missing")
                        for d in j["after"]]
                if any(s in ("failed", "skipped", "missing") for s in deps):
                    self.status[j["folder"]] = "failed"
                    self.records[j["folder"]].update(status="FAILED", note="dependency failed")
                    continue
                if all(s == "done" for s in deps):
                    self.status[j["folder"]] = "running"
                    return j
                waiting = True
            return "wait" if waiting else None

    def _done(self, folder: str) -> bool:
        return os.path.exists(os.path.join(self.exp_dir, folder, "DONE.json"))

    def worker(self, gpu: str) -> None:
        while True:
            j = self._next()
            if j is None:
                return
            if j == "wait":
                time.sleep(5)
                continue
            ok = False
            try:
                ok = self.run_job(j, gpu)
            except DiskFull as e:
                self.records[j["folder"]].update(status="STOPPED", note=str(e))
            except Exception as e:                     # never kill the other worker
                self.records[j["folder"]].update(status="FAILED", note=f"orchestrator: {e}")
            with self.lock:
                self.status[j["folder"]] = "done" if ok else (
                    "skipped" if self.records[j["folder"]]["status"] in ("not started",
                                                                         "PAUSED") else "failed")
                self.select_and_index()

    def select_and_index(self) -> None:
        from rr_select import select
        try:
            if self.exp != "P0":                       # the pilot is not an upload
                select(self.exp_dir, log=lambda m: self.say(m))
            results_tables(self.exp_dir)
        except Exception as e:
            self.say(f"   selection failed: {e}")
        write_index(self.out_root, self.folder, list(self.records.values()), "")

    # ---------------------------------------------------------------- one job
    def run_job(self, j: Dict, gpu: str) -> bool:
        a, folder = self.a, j["folder"]
        rec = self.records[folder]
        rec["gpu"] = gpu
        run_dir = os.path.join(self.exp_dir, folder)
        logs = os.path.join(run_dir, "logs")
        os.makedirs(logs, exist_ok=True)
        cons = os.path.join(logs, "console.log")
        pre = f"[{folder}@{gpu if gpu.startswith('cpu') else 'gpu' + gpu}] "
        t0 = time.time()
        done = read_json(os.path.join(run_dir, "DONE.json"))
        if done:
            rec.update(status="done (already finished)", dev_macro_f1=done["dev_macro_f1"],
                       cl=(done.get("dev_per_domain_macro_f1") or {}).get("CL"),
                       it=(done.get("dev_per_domain_macro_f1") or {}).get("IT"))
            self.say(f"{pre}already finished (dev Macro-F1 {done['dev_macro_f1']:.4f}), skipping")
            return True
        if j["kind"] == "combine":
            from rr_ensemble import combine_domains
            srcs = [os.path.join(self.exp_dir, d) for d in j["after"]]
            done = combine_domains(srcs, run_dir, self.data_dir, j, log=lambda m: self.say(pre + m))
            rec.update(status="done", dev_macro_f1=done["dev_macro_f1"],
                       minutes=(time.time() - t0) / 60)
            return True
        if j["model"] not in self.p1:
            rec.update(status="FAILED", note="Phase-1 weights not attached")
            self.missing_dependency = True
            return False
        if j["technique"] == "kd":
            from rr_kd import resolve_teacher
            if not resolve_teacher(j["teacher_dir"], j["model"]):
                rec.update(status="FAILED", note=f"KD teacher missing in {j['teacher_dir']}")
                self.say(f"{pre}KD teacher for {j['model']} not found in {j['teacher_dir']}")
                self.missing_dependency = True
                return False
        disk_guard(self.out_root, a.min_free_gb, self.say, keep=os.path.join(run_dir, "checkpoints"))
        if self.deadline - time.time() < 15 * 60:
            rec.update(status="not started", note="notebook time budget exhausted")
            return False
        self.say(f"\n{pre}-- {self.folder} / {folder} ({j['technique']}, "
                 f"{MODELS[j['model']]['arch']}) --")
        env = {"CUDA_VISIBLE_DEVICES": "" if gpu.startswith("cpu") else gpu}
        cfg_path = os.path.join(logs, "run_config.json")
        old = read_json(cfg_path)
        resuming = bool(old) and os.path.exists(os.path.join(run_dir, "checkpoints", "last.pt"))
        cfg = make_run_config(a, self.exp, self.folder, j, self.meta, self.data_dir,
                              self.p1[j["model"]], self.attn, self.env)
        if old:                                           # keep the probed lr
            for k in ("lr", "head_lr", "lr_probe"):
                if k in old:
                    cfg[k] = old[k]
        write_json(cfg_path, cfg)

        # ---- smoke: real-model self-checks + throughput ----------------------------------------
        smoke = read_json(os.path.join(logs, "smoke.json"))
        if smoke is None:
            code, to = run_cmd(train_cmd(run_dir, "smoke"), cons, env, a.smoke_timeout_s, pre)
            smoke = read_json(os.path.join(logs, "smoke.json"))
            if code != 0 or smoke is None:
                why = "self-checks failed" if code == 6 else ("timeout" if to else f"exit {code}")
                rec.update(status="FAILED", note=f"smoke: {why} (see logs/console.log)")
                if code == 6:
                    self.validation_failed = True
                return False
        proj = float(smoke.get("projected_train_minutes", 0) or 0)
        skip_probe = a.skip_probe or j.get("skip_probe") or resuming
        probe_min = 0.0 if (skip_probe or os.path.exists(os.path.join(logs, "lr_probe.json"))) \
            else proj * 0.45
        need = (proj + probe_min) * 1.3 + 5
        left = (self.deadline - time.time()) / 60
        self.say(f"{pre}projected {proj:.1f} min train + {probe_min:.1f} min probe | {left:.0f} "
                 f"min left in the notebook budget")
        if need > left and not resuming:
            rec.update(status="not started", note=f"needs ~{need:.0f} min, {left:.0f} left")
            return False

        # ---- overfit-32 (first job of the notebook) ----------------------------------------------
        with self.lock:
            do_overfit = not self.overfit_claimed and not j.get("skip_overfit")
            if do_overfit:
                self.overfit_claimed = True
        if do_overfit and read_json(os.path.join(logs, "overfit.json")) is None:
            run_cmd(train_cmd(run_dir, "overfit"), cons, env, 1800, pre)
            ov = read_json(os.path.join(logs, "overfit.json"))
            if ov is None or not ov.get("passed"):
                fatal = ov is None or ov.get("fatal")
                self.say(f"{pre}overfit-32 {'FAILED' if ov else 'crashed'}"
                         f"{' -> skipping this job' if fatal else ' (warning only)'}")
                if fatal:
                    rec.update(status="FAILED", note="overfit-32 failed: check GATES.json")
                    return False

        # ---- LR probe ----------------------------------------------------------------------------
        if not skip_probe:
            pr = read_json(os.path.join(logs, "lr_probe.json"))
            if pr is None:
                run_cmd(train_cmd(run_dir, "probe"), cons, env, max(600, proj * 60 * 1.5), pre)
                pr = read_json(os.path.join(logs, "lr_probe.json"))
            if pr:
                cfg["lr"] = pr["best_lr"]
                if j["technique"] in ("full_ft", "kd"):
                    cfg["head_lr"] = 10 * cfg["lr"]
                cfg["lr_probe"] = pr
                self.say(f"{pre}LR probe -> lr {cfg['lr']:.2e}")
            else:
                self.say(f"{pre}LR probe failed; keeping the default learning rate")

        # ---- train (per-job time budget from the measured throughput) -------------------------
        job_deadline = min(self.deadline, time.time() + 60 * (2.0 * proj + 15))
        if resuming:
            job_deadline = self.deadline
        cfg["deadline_unix"] = job_deadline
        write_json(cfg_path, cfg)
        code, to = run_cmd(train_cmd(run_dir, "train"), cons, env,
                           max(60.0, job_deadline - time.time() + 600), pre)
        if code not in (0, 2, 4, 5) and not to:
            self.say(f"{pre}training exited with {code} -> one retry (resumes from last.pt)")
            code, to = run_cmd(train_cmd(run_dir, "train"), cons, env,
                               max(60.0, self.deadline - time.time()), pre)
        if code == 2 or to:
            best_exists = os.path.exists(os.path.join(run_dir, "checkpoints", "best.pt"))
            if not best_exists:
                rec.update(status="PAUSED", note="time budget; last.pt saved, attach output "
                                                 "and rerun to resume")
                return False
            self.say(f"{pre}time budget reached: predicting with the best checkpoint so far")
        elif code != 0:
            why = {4: "non-finite gradients (lower --lr)", 5: "early-fail watchdog (dev F1 ~ 0)",
                   3: "out of memory"}.get(code, f"exit {code}")
            rec.update(status="FAILED", note=f"training: {why}")
            return False

        # ---- predict + submission -------------------------------------------------------------------
        code, _ = run_cmd([sys.executable, PREDICT, "--run-dir", run_dir], cons, env, 3600, pre)
        done = read_json(os.path.join(run_dir, "DONE.json"))
        if code != 0 or not done:
            if code == EXIT["validation_failure"]:
                self.validation_failed = True
            rec.update(status="FAILED", note=f"prediction/submission step failed (exit {code})")
            return False
        for f in ("last.pt", "last.pt.tmp"):
            p = os.path.join(run_dir, "checkpoints", f)
            if os.path.exists(p):
                os.remove(p)
        tc = os.path.join(run_dir, "teacher_cache.pt")
        if os.path.exists(tc):
            os.remove(tc)
        mins = (time.time() - t0) / 60
        rec.update(status="done", dev_macro_f1=done["dev_macro_f1"], minutes=mins,
                   cl=(done.get("dev_per_domain_macro_f1") or {}).get("CL"),
                   it=(done.get("dev_per_domain_macro_f1") or {}).get("IT"))
        self.say(f"{pre}finished in {mins:.1f} min | dev pooled Macro-F1 "
                 f"{done['dev_macro_f1']:.4f} | output so far {dir_size_gb(self.out_root):.2f} GB")
        return True


# =============================================================================================
# main
# =============================================================================================
def build_parser() -> argparse.ArgumentParser:
    ap = argparse.ArgumentParser()
    ap.add_argument("--exp", required=True, help="E0..E11 or P0 (pilot)")
    ap.add_argument("--models", default=None, help="comma list, default M4,M2,M3,M1")
    ap.add_argument("--gpus", default="auto", help="auto | comma list of GPU ids | cpu")
    ap.add_argument("--resume", default="auto", choices=["auto", "none"])
    ap.add_argument("--time-budget-h", type=float, default=11.0)
    ap.add_argument("--out", default=OUT_ROOT)
    ap.add_argument("--data-dir", default=DATA_DIR)
    ap.add_argument("--attn-mode", default="auto", choices=["auto", "bidir", "causal"])
    ap.add_argument("--input-mode", default="doc")
    ap.add_argument("--head", default="linear", choices=["linear", "seqctx"])
    ap.add_argument("--pool", default="marker", choices=["marker", "mean"])
    ap.add_argument("--loss", default="weighted_ce",
                    choices=["ce", "weighted_ce", "focal", "logit_adjust"])
    ap.add_argument("--window-combine", default="center", choices=["center", "mean"])
    ap.add_argument("--max-sent-tokens", type=int, default=256)
    ap.add_argument("--lr", type=float, default=None)
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--patience", type=int, default=4)
    ap.add_argument("--effective-batch", type=int, default=None)
    ap.add_argument("--micro-batch", type=int, default=None)
    ap.add_argument("--ckpt-every-min", type=float, default=30)
    ap.add_argument("--watchdog-min-f1", type=float, default=0.01)
    ap.add_argument("--probe-steps", type=int, default=None)
    ap.add_argument("--skip-probe", action="store_true")
    ap.add_argument("--skip-overfit", action="store_true")
    ap.add_argument("--skip-selftest", action="store_true")
    ap.add_argument("--skip-job-checks", action="store_true")
    ap.add_argument("--teacher-dir", default=None)
    ap.add_argument("--kd-hidden-weight", type=float, default=0.0)
    ap.add_argument("--ablation-techniques", default="lorma,lora")
    ap.add_argument("--smoke-timeout-s", type=int, default=1200)
    ap.add_argument("--min-free-gb", type=float, default=3.0)
    return ap


def main() -> None:
    a = build_parser().parse_args()

    t_start = time.time()
    deadline = t_start + a.time_budget_h * 3600
    exp = a.exp
    if exp == "P0":
        spec = {"folder": PILOT_FOLDER, "technique": "lora", "title": "attention-mode pilot"}
    elif exp in EXPERIMENTS:
        spec = EXPERIMENTS[exp]
    else:
        sys.exit(f"unknown experiment {exp!r}")
    technique, folder = spec["technique"], spec["folder"]
    out_root = os.path.abspath(a.out)
    exp_dir = os.path.join(out_root, folder)
    os.makedirs(exp_dir, exist_ok=True)
    console = os.path.join(exp_dir, "console.log")
    roots = default_input_roots()

    def say(msg: str) -> None:
        with _print_lock:
            print(msg, flush=True)
        with open(console, "a", encoding="utf-8") as f:
            f.write(time.strftime("%H:%M:%S ") + msg + "\n")

    say(f"== {folder}: {spec['title']} | budget {a.time_budget_h} h | deadline of the brief "
        f"{SPEC['deadline']}")

    # ---- data + gates ---------------------------------------------------------------------------
    from rr_data import find_or_build_cache, load_cache
    try:
        data_dir = find_or_build_cache(os.path.abspath(a.data_dir), roots, log=say)
    except GateFailure as e:
        say(f"STOP AND ASK (data gate): {e}")
        sys.exit(EXIT["gate_failure"])
    except FileNotFoundError as e:
        say(f"MISSING INPUT: {e}")
        sys.exit(EXIT["missing_dependency"])
    meta, _docs = load_cache(data_dir)
    del _docs
    shutil.copy2(os.path.join(data_dir, "GATES.json"), os.path.join(out_root, "GATES.json"))
    if technique == "ensemble":                      # E11: CPU only, works on saved logits
        from rr_aggregate import build_report
        from rr_ensemble import run_ensemble
        code = run_ensemble(out_root, roots, exp_dir, data_dir, log=say)
        build_report(out_root, roots, log=say)
        sys.exit(code)
    p1 = find_phase1_models(roots)
    say(f"   Phase-1 weights found for: {sorted(p1)}")
    if not p1:
        say("MISSING: no Phase-1 model (final/config.json + model.pt) under the inputs")
        sys.exit(EXIT["missing_dependency"])

    # ---- CPU selftest (tiny models), once per output folder -------------------------------------
    st_path = os.path.join(out_root, "SELFTEST.json")
    if not a.skip_selftest and not (read_json(st_path) or {}).get("ok"):
        code, _ = run_cmd([sys.executable, SELFTEST, "--out", st_path, "--data-dir", data_dir],
                          os.path.join(out_root, "selftest.log"), {"CUDA_VISIBLE_DEVICES": ""},
                          1800, "[selftest] ")
        if code != 0 or not (read_json(st_path) or {}).get("ok"):
            say("STOP: selftest failed (see selftest.log / SELFTEST.json)")
            sys.exit(EXIT["gate_failure"])

    attn, why = ("bidir", "pilot jobs set their own mode") if exp == "P0" else \
        resolve_attn_mode(a, out_root, roots, say)
    say(f"   attention mode: {attn} ({why})")
    models = [m.strip() for m in (a.models or ",".join(
        ["M4"] if technique == "ablation" else DEFAULT_MODEL_ORDER)).split(",") if m.strip()]
    jobs = make_jobs(a, exp, technique, models, exp_dir)
    if a.resume == "auto":
        restore_from_inputs(folder, out_root, roots, say)

    # ---- workers: GPU sanity on every GPU, fallback to the ones that pass ---------------------
    env = env_info()
    say(f"   environment: {env}")
    if a.gpus.startswith("cpu"):
        gpus = [g.strip() for g in a.gpus.split(",")]       # e.g. "cpu" or "cpu0,cpu1" (tests)
    else:
        cand = [g.strip() for g in a.gpus.split(",")] if a.gpus != "auto" else \
            [str(i) for i in range(len(env.get("gpus") or []))]
        res = {}
        ths = [threading.Thread(target=lambda g=g: res.__setitem__(g, gpu_sanity(g)))
               for g in cand]
        for t in ths:
            t.start()
        for t in ths:
            t.join()
        gpus = [g for g in cand if res.get(g)]
        bad = [g for g in cand if not res.get(g)]
        say(f"   GPU sanity: ok {gpus} | failed {bad}" + (" -> falling back" if bad else ""))
        if not gpus:
            say("   no usable GPU -> running on CPU (slow)")
            gpus = ["cpu"]
    runner = Runner(a, exp, folder, exp_dir, out_root, jobs, meta, data_dir, p1, attn, env,
                    deadline, say)
    say(f"   {len(jobs)} jobs on {len(gpus)} worker(s) {gpus}: {[j['folder'] for j in jobs]}")
    threads = [threading.Thread(target=runner.worker, args=(g,), daemon=True) for g in gpus]
    for i, t in enumerate(threads):
        t.start()
        if i == 0 and len(threads) > 1:
            time.sleep(20)                         # stagger CPU-heavy start-up
    for t in threads:
        t.join()

    # ---- best-of-N + tables ------------------------------------------------------------------------
    runner.select_and_index()
    pilot = write_pilot(out_root, exp_dir, say) if exp == "P0" else None
    sel = read_json(os.path.join(exp_dir, "BEST", "SELECTION.json"))
    code = EXIT["ok"]
    if sel:
        from rr_data import load_cache as _lc
        from rr_predict import official_test_counts, validate_submission
        m2, d2 = _lc(data_dir)
        vr = validate_submission(os.path.join(exp_dir, "BEST", "submission.zip"),
                                 official_test_counts(m2, d2), m2["labels"])
        write_json(os.path.join(exp_dir, "BEST", "validation_report.json"), vr)
        if not vr["ok"]:
            say(f"   !!! BEST submission FAILED validation: {vr['errors'][:3]}")
            code = EXIT["validation_failure"]
        else:
            say(f"\n   UPLOAD: {os.path.join(exp_dir, 'BEST', 'submission.zip')} (winner "
                f"{sel['winner']}, dev pooled Macro-F1 {sel['dev_macro_f1']:.4f}) -- validated")
    if runner.validation_failed:
        code = EXIT["validation_failure"]
    elif runner.missing_dependency:
        code = EXIT["missing_dependency"]
    elif not sel and not pilot:
        code = EXIT["error"]
    say(f"\n== {folder} finished this session in {(time.time() - t_start) / 60:.1f} min "
        f"(exit {code})")
    for r in runner.records.values():
        say(f"   {r['job']:<32} {r['status']:<28} "
            f"{r.get('dev_macro_f1', float('nan')):.4f}  {r.get('note', '')}")
    say("   Save this session's output (Save Version) -- it is the input of later sessions.")
    sys.exit(code)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase3/selftest.py
"""
selftest.py -- the tests the Phase-3 code must pass (section 11 of the code-generation prompt).

    python selftest.py [--out SELFTEST.json] [--data-dir DIR] [--full-size]

CPU only, tiny random models of the four Phase-1 architectures (no weights needed), synthetic
documents (plus the real dev/test documents when a data cache is given):
  1  Macro-P/R/F1 against hand-computed cases; the empty-class rule
  2  window packing: every sentence covered, markers at the right positions, unpack(pack(doc))
     recovers order and count -- incl. a document > 1024 tokens and a one-sentence document,
     for doc / sent / ctx_k, bidir / causal, with and without virtual tokens
  3  output length == number of sentences for every dev doc; the validator rejects corrupted
     submissions
  4  every PEFT technique: trainable count > 0 (and <= MAX_TRAINABLE_PCT with --full-size);
     zero-init variants reproduce the base logits (LoRA/LoRMA/rsLoRA/AdaLoRA B or lambda = 0,
     IA3 ones, prefix/prompt P = 0, QLoRA = its NF4 base); merge == unmerged within 1e-4;
     rsLoRA scale; AdaLoRA cubic budget + pruning to the target; NF4 code book, round-trip
     error bound and memory
  5  prefix (and soft-prompt) positions are visible to every query under all four attention
     variants, causal and bidir (incl. M3's sliding window)
  6  overfit: 32 sentences reach > 95% train accuracy within 300 steps (rr_train, subprocess)
  7  checkpoint -> kill -> resume reproduces the step counter and a finite loss (subprocess)

`job_checks(cfg, meta, docs, device)` runs the technique-specific checks on the REAL Phase-1
model; rr_train calls it in smoke mode before the throughput measurement.
"""
import argparse
import copy
import json
import math
import os
import pickle
import subprocess
import sys
import tempfile
import time
import traceback
from typing import Callable, Dict, List

import numpy as np
import torch

from rr_common import MAX_TRAINABLE_PCT, MODELS, SPEC, write_json
from rr_data import (build_doc_windows, build_windows, check_packing, combine_slot_scores,
                     load_cache)
from rr_eval import macro_f1, macro_scores, tune_bias
from rr_models import build_rr_model, collate, model_size
from rr_peft import (NF4_CODE, NF4Linear, RSLoRALinear, dequantize_nf4, get_technique,
                     nf4_codebook_from_normal, quantize_nf4)

HERE = os.path.dirname(os.path.abspath(__file__))
V_TINY = 400
TINY = {
    "M1": ("GPT2StyleLM", {"vocab_size": V_TINY, "n_positions": 1024, "n_embd": 32, "n_layer": 4,
                           "n_head": 4, "mlp_ratio": 4}),
    "M2": ("LlamaStyleLM", {"vocab_size": V_TINY, "max_position_embeddings": 1024,
                            "hidden_size": 32, "intermediate_size": 64, "num_hidden_layers": 4,
                            "num_attention_heads": 4, "num_key_value_heads": 2,
                            "sliding_window": None}),
    "M3": ("LlamaStyleLM", {"vocab_size": V_TINY, "max_position_embeddings": 1024,
                            "hidden_size": 32, "intermediate_size": 64, "num_hidden_layers": 4,
                            "num_attention_heads": 4, "num_key_value_heads": 2,
                            "sliding_window": 8}),
    "M4": ("BloomStyleLM", {"vocab_size": V_TINY, "hidden_size": 32, "n_layer": 4, "n_head": 4,
                            "mlp_ratio": 4}),
}
FULL = {
    "M1": ("GPT2StyleLM", {"vocab_size": 32000, "n_positions": 2048, "n_embd": 768, "n_layer": 14,
                           "n_head": 12, "mlp_ratio": 4}),
    "M2": ("LlamaStyleLM", {"vocab_size": 32000, "max_position_embeddings": 2048,
                            "hidden_size": 768, "intermediate_size": 3072, "num_hidden_layers": 12,
                            "num_attention_heads": 12, "num_key_value_heads": 4,
                            "sliding_window": None}),
    "M3": ("LlamaStyleLM", {"vocab_size": 32000, "max_position_embeddings": 2048,
                            "hidden_size": 768, "intermediate_size": 3072, "num_hidden_layers": 12,
                            "num_attention_heads": 12, "num_key_value_heads": 4,
                            "sliding_window": 512}),
    "M4": ("BloomStyleLM", {"vocab_size": 32000, "hidden_size": 768, "n_layer": 14, "n_head": 12,
                            "mlp_ratio": 4}),
}
PEFT = ["lorma", "lora", "rslora", "adalora", "qlora", "ia3", "prefix", "prompt"]


# =============================================================================================
# fixtures
# =============================================================================================
def write_phase1(root: str, mid: str, spec=TINY, seed: int = 0) -> str:
    """A Phase-1-style final/ folder (config.json + model.pt) with random weights."""
    from phase1_models import MODEL_CLASSES
    cls_name, conf = spec[mid]
    d = os.path.join(root, f"p1_{mid}", "final")
    os.makedirs(d, exist_ok=True)
    torch.manual_seed(seed)
    cls, ccls = MODEL_CLASSES[cls_name]
    lm = cls(ccls(**conf))
    torch.save(lm.state_dict(), os.path.join(d, "model.pt"))
    with open(os.path.join(d, "config.json"), "w") as f:
        json.dump({"model_class": cls_name, "arch": MODELS[mid]["arch"], "config": conf}, f)
    return d


def synthetic_cache(root: str, n_train: int = 6, n_dev: int = 2, n_test: int = 2,
                    seed: int = 0) -> str:
    """meta.json + docs.pkl with random token ids; labels follow a learnable rule."""
    rng = np.random.default_rng(seed)
    labels = SPEC["labels"]
    docs, split = {}, {"train": [], "dev": [], "test": []}
    test_ids = {}
    for dom in SPEC["domains"]:
        for sp, n in (("train", n_train), ("dev", n_dev), ("test", n_test)):
            for i in range(n):
                did = f"{dom}_{sp}_{i}"
                key = f"{dom}/{did}"
                ns = int(rng.integers(8, 30))
                lab = rng.integers(0, len(labels), ns)
                ids = []
                for j in range(ns):
                    L = int(rng.integers(3, 25))
                    s = rng.integers(3 + 30 * (lab[j] % 10), 33 + 30 * (lab[j] % 10), L)
                    ids.append(s.astype(np.int32))
                docs[key] = {"key": key, "id": did, "domain": dom, "split": sp,
                             "text": [f"s{j}" for j in range(ns)], "ids": ids,
                             "labels": None if sp == "test" else lab.astype(np.int64)}
                split[sp].append(key)
                if sp == "test":
                    test_ids.setdefault(dom, []).append(did)
    meta = {"version": 3, "labels": labels, "split": split, "test_ids": test_ids, "eos_id": 0,
            "pad_id": 1, "unk_id": 2, "vocab_size": V_TINY, "tokenizer_md5": "synthetic",
            "domain_prefix_ids": {"CL": [350, 351], "IT": [352, 353]}}
    os.makedirs(root, exist_ok=True)
    with open(os.path.join(root, "docs.pkl"), "wb") as f:
        pickle.dump(docs, f)
    write_json(os.path.join(root, "meta.json"), meta)
    write_json(os.path.join(root, "GATES.json"), {"status": "PASSED", "synthetic": True})
    return root


def base_cfg(technique: str, mid: str, phase1_dir: str, data_dir: str, **over) -> Dict:
    """A complete run_config (the orchestrator's defaults) for tests."""
    import run_experiment as rx
    a = rx.build_parser().parse_args(["--exp", "E1"])
    meta = json.load(open(os.path.join(data_dir, "meta.json")))
    job = {"model": mid, "folder": f"test_{technique}_{mid}", "technique": technique,
           "overrides": over}
    cfg = rx.make_run_config(a, "TEST", "TEST", job, meta, data_dir, phase1_dir, "bidir", {})
    cfg["n_virtual"] = int(cfg["prompt_len"]) if technique == "prompt" else 0
    return cfg


def fake_meta(prefix=(350, 351)) -> Dict:
    return {"eos_id": 0, "pad_id": 1, "domain_prefix_ids": {"CL": list(prefix),
                                                            "IT": list(prefix)}}


# =============================================================================================
# 1. metrics
# =============================================================================================
def t1_metrics() -> Dict:
    labs = ["A", "B", "C"]
    r = macro_scores([0, 0, 1, 1], [0, 1, 1, 1], labs)
    assert abs(r["macro_f1"] - (2 / 3 + 0.8) / 2) < 1e-9, r["macro_f1"]
    assert abs(r["macro_precision"] - (1 + 2 / 3) / 2) < 1e-9
    assert abs(r["macro_recall"] - (0.5 + 1.0) / 2) < 1e-9
    assert r["classes_skipped"] == ["C"] and r["classes_used"] == ["A", "B"]
    r2 = macro_scores([0, 0], [0, 1], labs)               # predicted-but-absent class counts as 0
    assert abs(r2["macro_f1"] - (2 / 3 + 0.0) / 2) < 1e-9, r2["macro_f1"]
    r3 = macro_scores([0, 1, 2], [0, 1, 2], labs)
    assert r3["macro_f1"] == 1.0
    r4 = macro_scores([0, 0, 0], [1, 1, 1], labs)
    assert r4["macro_f1"] == 0.0 and r4["classes_used"] == ["A", "B"]
    # sklearn-documented example: y=[0,1,2,0,1,2], p=[0,2,1,0,0,1] -> macro F1 0.2667
    r5 = macro_scores([0, 1, 2, 0, 1, 2], [0, 2, 1, 0, 0, 1], labs)
    assert abs(r5["macro_f1"] - 0.26666666666666666) < 1e-9, r5["macro_f1"]
    rng = np.random.default_rng(0)
    for _ in range(20):
        g, p = rng.integers(0, 5, 50), rng.integers(0, 5, 50)
        assert abs(macro_f1(g, p, 13) - macro_scores(g, p, [str(i) for i in range(13)])
                   ["macro_f1"]) < 1e-12
    lg = rng.normal(size=(200, 4))
    gold = rng.integers(0, 4, 200)
    b, f = tune_bias(lg, gold)
    assert f >= macro_f1(gold, lg.argmax(1), 4) - 1e-12 and np.all(np.abs(b) <= 3)
    return {"cases": 6, "rule": "skip classes with zero support and zero predictions"}


# =============================================================================================
# 2. packing
# =============================================================================================
def _doc(key, lens, rng, labels=True):
    ids = [rng.integers(3, 300, L).astype(np.int32) for L in lens]
    return {"key": key, "domain": "CL", "ids": ids,
            "labels": rng.integers(0, 13, len(lens)).astype(np.int64) if labels else None}


def t2_packing() -> Dict:
    rng = np.random.default_rng(1)
    meta = fake_meta()
    docs = [_doc("CL/one", [12], rng), _doc("CL/long", [40] * 60, rng),
            _doc("CL/trunc", [10, 300, 7, 1, 256, 255], rng),
            _doc("CL/var", list(rng.integers(1, 200, 80)), rng)]
    assert sum(len(x) + 1 for x in docs[1]["ids"]) > 1024
    n = 0
    for mode in ("doc", "sent", "ctx_1", "ctx_2"):
        for attn in ("bidir", "causal"):
            for nv in (0, 20):
                cfg = {"input_mode": mode, "attn_mode": attn, "max_len": 1024,
                       "max_sent_tokens": 256, "n_virtual": nv}
                for d in docs:
                    for purpose in ("train", "infer"):
                        r = check_packing(d, cfg, meta, purpose)
                        n += 1
                        if mode == "doc" and d["key"] == "CL/long":
                            assert r["windows"] >= 3 and r["max_occurrences"] >= 2
    cfg = {"input_mode": "doc", "attn_mode": "bidir", "max_len": 1024, "max_sent_tokens": 256}
    wins = build_doc_windows(docs[1], cfg, meta, "infer")
    slot = [w["sent_idx"].astype(np.float32)[:, None].repeat(13, 1) for w in wins]
    for how in ("center", "mean"):
        comb = combine_slot_scores(wins, slot, {"CL/long": 60}, how)["CL/long"]
        assert np.allclose(comb[:, 0], np.arange(60)), how
    return {"configurations_checked": n}


# =============================================================================================
# 3. output length + validator
# =============================================================================================
def t3_outputs(data_dir: str) -> Dict:
    from rr_predict import g2_validator_roundtrip, official_test_counts
    meta, docs = load_cache(data_dir)
    cfg = {"input_mode": "doc", "attn_mode": "bidir", "max_len": 1024, "max_sent_tokens": 256,
           "n_virtual": 0}
    rng = np.random.default_rng(2)
    checked = 0
    for split in ("dev", "test"):
        keys = meta["split"][split]
        wins = build_windows(docs, keys, cfg, meta, "infer")
        slot = [rng.normal(size=(len(w["markers"]), 13)).astype(np.float32) for w in wins]
        comb = combine_slot_scores(wins, slot, {k: len(docs[k]["ids"]) for k in keys})
        for k in keys:
            assert comb[k].shape == (len(docs[k]["text"]), 13), k
            checked += 1
    with tempfile.TemporaryDirectory() as td:
        g2 = g2_validator_roundtrip(official_test_counts(meta, docs), meta["labels"], td)
    return {"docs_checked": checked, "rejected_variants": sorted(g2["rejected"])}


# =============================================================================================
# 4. PEFT
# =============================================================================================
def _batch(meta, docs, n=2, device="cpu", attn="bidir", n_virtual=0, max_tokens=None):
    cfg = {"input_mode": "doc", "attn_mode": attn, "max_len": max_tokens or 1024,
           "max_sent_tokens": 256, "n_virtual": n_virtual}
    keys = meta["split"]["dev"][:2]
    wins = build_windows(docs, keys, cfg, meta, "infer")
    wins = sorted(wins, key=lambda w: len(w["tokens"]))[:n]
    return collate(wins, list(range(len(wins))), meta["pad_id"], device)


def _fwd(m, b):
    with torch.no_grad():
        return m(b["input_ids"], b["attention_mask"], b["markers"], b["starts"],
                 b["marker_mask"])[0]


def _randomize(m, tech: str, scale: float = 0.05) -> None:
    g = torch.Generator().manual_seed(7)
    with torch.no_grad():
        for n, p in m.named_parameters():
            if not p.requires_grad or n.startswith("head."):
                continue
            noise = torch.randn(p.shape, generator=g).to(p.device, p.dtype) * scale
            if tech == "ia3":
                p.copy_(1 + noise)
            else:
                p.add_(noise)


def _qlora_reference(m0, cfg):
    """The model whose target linears are NF4 (no LoRA): QLoRA with B = 0 must equal it."""
    from ner_peft import TARGETS, _get, _set
    ref = copy.deepcopy(m0)
    for blk in ref.trunk.blocks:
        for path in TARGETS[ref.trunk.kind]:
            _set(blk, path, NF4Linear(_get(blk, path), cfg["qlora_block"],
                                      cfg["qlora_double_quant"]))
    return ref


def peft_checks(m0, cfg: Dict, b, tech: str, check_pct: bool, tol_zero: float = 1e-5,
                tol_merge: float = 1e-4) -> Dict:
    out0 = _fwd(m0, b)
    res = {}
    # zero-init reproduces the base (P = 0 for prefix / prompt; NF4 base for QLoRA)
    zc = dict(cfg, technique=tech, prefix_len=0, prompt_len=0)
    mz = copy.deepcopy(m0)
    get_technique(tech).apply(mz, zc, log=lambda *x: None)
    mz.eval()
    ref = _fwd(_qlora_reference(m0, zc), b) if tech == "qlora" else out0
    dz = float((_fwd(mz, b) - ref).abs().max())
    res["zero_init_max_abs_diff"] = dz
    assert dz <= tol_zero * max(1.0, float(ref.abs().max())), f"{tech}: zero-init diff {dz}"
    # trainable count / pct, and a training step produces finite grads
    m1 = copy.deepcopy(m0)
    t = get_technique(tech)
    t.apply(m1, dict(cfg, technique=tech), log=lambda *x: None)
    size = model_size(m1)
    res.update(trainable=size["trainable_params"], trainable_pct=size["trainable_pct"])
    assert size["trainable_params"] > 0
    if check_pct and tech in MAX_TRAINABLE_PCT:
        assert size["trainable_pct"] <= MAX_TRAINABLE_PCT[tech], \
            f"{tech}: {size['trainable_pct']:.2f}% > {MAX_TRAINABLE_PCT[tech]}%"
    m1.train()
    logits, _ = m1(b["input_ids"], b["attention_mask"], b["markers"], b["starts"],
                   b["marker_mask"])
    loss = logits[b["marker_mask"]].float().logsumexp(-1).mean()
    extra = t.extra_loss(m1)
    if extra is not None:
        loss = loss + extra
    loss.backward()
    gr = [p.grad for n, p in m1.named_parameters() if p.requires_grad and not n.startswith("head.")]
    assert any(g is not None and float(g.abs().sum()) > 0 for g in gr), f"{tech}: no gradient"
    assert all(g is None or bool(torch.isfinite(g).all()) for g in gr)
    m1.zero_grad(set_to_none=True)
    # merge == unmerged
    m1.eval()
    _randomize(m1, tech)
    out_u = _fwd(m1, b)
    if t.merge(m1):
        out_m = _fwd(m1, b)
        dm = float((out_m - out_u).abs().max())
        res["merge_max_abs_diff"] = dm
        assert dm <= tol_merge * max(1.0, float(out_u.abs().max())), f"{tech}: merge diff {dm}"
    else:
        res["merge"] = "not applicable"
    return res


def t4_peft(data_dir: str, root: str) -> Dict:
    meta, docs = load_cache(data_dir)
    out = {}
    for mid in MODELS:
        p1 = write_phase1(root, mid)
        m0, _ = build_rr_model(p1, 13, "bidir", "linear")
        m0.eval()
        for tech in PEFT:
            cfg = base_cfg(tech, mid, p1, data_dir)
            b = _batch(meta, docs, n_virtual=cfg["n_virtual"])
            out[f"{mid}/{tech}"] = peft_checks(m0, cfg, b, tech, False)
    # rsLoRA scale formula
    lin = torch.nn.Linear(16, 16)
    for r, al in ((4, 8), (32, 16), (64, 16)):
        assert abs(RSLoRALinear(lin, r, al).scale - al / math.sqrt(r)) < 1e-12
    out["rslora_scale"] = "alpha/sqrt(r) verified"
    out["adalora"] = t4_adalora(root, data_dir)
    out["nf4"] = t4_nf4()
    return out


def t4_adalora(root: str, data_dir: str) -> Dict:
    p1 = write_phase1(root, "M2")
    m, _ = build_rr_model(p1, 13)
    cfg = base_cfg("adalora", "M2", p1, data_dir)
    t = get_technique("adalora")
    t.apply(m, cfg, log=lambda *x: None)
    T = 200
    bs = [t.budget(s, T) for s in range(T)]
    assert bs[0] == t.b0 and bs[-1] == t.bT
    assert all(x >= y for x, y in zip(bs, bs[1:])), "budget must not increase"
    g = torch.Generator().manual_seed(0)
    for step in range(T):
        for _, mod in t.modules_:
            for p in (mod.ada_P, mod.ada_E, mod.ada_Q):
                p.grad = torch.randn(p.shape, generator=g)
            mod.ada_E.data += 0.01 * torch.randn(mod.ada_E.shape, generator=g)
        t.after_backward(m, step, T)
    rep = t.final_report(m)
    assert rep["adalora_budget_final"] == t.bT, rep["adalora_budget_final"]
    x = torch.randn(2, 5, m.trunk.hidden_size)
    m.eval()
    for _, mod in t.modules_[:3]:               # pruned singular values contribute nothing
        y = mod(x[..., :mod.base.in_features]) if mod.base.in_features == x.shape[-1] else None
        if y is not None:
            with torch.no_grad():
                e = mod.ada_E.clone()
                mod.ada_E.mul_(mod.ada_mask)
                y2 = mod(x)
                mod.ada_E.copy_(e)
            assert torch.allclose(y, y2, atol=1e-6)
    return {"b0": t.b0, "bT": t.bT, "final": rep["adalora_budget_final"],
            "per_layer": rep["adalora_rank_per_layer"]}


def t4_nf4() -> Dict:
    code = nf4_codebook_from_normal()
    assert torch.allclose(code, torch.tensor(NF4_CODE), atol=1e-6), (code, NF4_CODE)
    torch.manual_seed(0)
    lin = torch.nn.Linear(512, 256)
    w = lin.weight.detach()
    packed, absmax, n = quantize_nf4(w, 64)
    deq = dequantize_nf4(packed, absmax, n, 64, torch.tensor(NF4_CODE)).view_as(w)
    half_gap = float(torch.diff(torch.tensor(NF4_CODE)).max()) / 2
    bound = absmax.repeat_interleave(64)[:n].view_as(w) * (half_gap + 1e-6)
    assert torch.all((w - deq).abs() <= bound), "NF4 round-trip error above the bound"
    rel = float((w - deq).norm() / w.norm())
    assert rel < 0.12, rel
    q = NF4Linear(lin, 64, True)
    rel_dq = float((q.dequantize() - w).norm() / w.norm())
    assert rel_dq < 0.13, rel_dq
    fp16 = 2 * w.numel()
    assert q.storage_bytes() * 3 < fp16, (q.storage_bytes(), fp16)
    x = torch.randn(3, 512)
    assert torch.allclose(q(x), torch.nn.functional.linear(x, q.dequantize(), lin.bias),
                          atol=1e-5)
    return {"relative_error": rel, "relative_error_double_quant": rel_dq,
            "bytes_fp16": fp16, "bytes_nf4_dq": q.storage_bytes(),
            "ratio": fp16 / q.storage_bytes()}


# =============================================================================================
# 5. prefix / prompt visibility
# =============================================================================================
def visibility(m, b, param: torch.Tensor, positions: List[int]) -> List[int]:
    """Positions whose output does NOT depend on `param` (must be empty)."""
    h = m.trunk(b["input_ids"], b["attention_mask"])
    # a fixed random projection (the plain sum of a LayerNorm output is constant)
    r = torch.randn(h.shape[-1], generator=torch.Generator().manual_seed(3)).to(h.device)
    blind = []
    for t in positions:
        g, = torch.autograd.grad((h[0, t].float() * r).sum(), param, retain_graph=True,
                                 allow_unused=True)
        if g is None or float(g.abs().sum()) == 0:
            blind.append(t)
    return blind


def t5_visibility(data_dir: str, root: str) -> Dict:
    out = {}
    x = torch.randint(3, V_TINY, (1, 24))
    b = {"input_ids": x, "attention_mask": torch.ones_like(x)}
    for mid in MODELS:
        p1 = write_phase1(root, mid)
        for attn in ("causal", "bidir"):
            for tech in ("prefix", "prompt"):
                m, _ = build_rr_model(p1, 13, attn)
                cfg = base_cfg(tech, mid, p1, data_dir, prefix_len=4, prompt_len=4,
                               prefix_reparam=False)
                get_technique(tech).apply(m, cfg, log=lambda *a: None)
                m.eval()
                param = m.trunk.prefix.prefix_kv if tech == "prefix" else m.trunk.prompt
                blind = visibility(m, b, param, list(range(24)))
                assert not blind, f"{mid}/{attn}/{tech}: positions {blind} cannot see it"
                out[f"{mid}/{attn}/{tech}"] = "visible to all 24 queries"
    return out


# =============================================================================================
# 6 + 7. overfit and kill/resume (subprocesses running rr_train.py)
# =============================================================================================
def _run(cmd, cwd=HERE, timeout=1200):
    env = dict(os.environ, PYTHONPATH=HERE, CUDA_VISIBLE_DEVICES="",
               PHASE3_INPUT_ROOTS=os.environ.get("PHASE3_INPUT_ROOTS", "/nonexistent"))
    p = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True, timeout=timeout)
    return p.returncode, p.stdout + p.stderr


def t6_overfit(data_dir: str, root: str) -> Dict:
    p1 = write_phase1(root, "M2")
    run = os.path.join(root, "run_overfit")
    cfg = base_cfg("lorma", "M2", p1, data_dir)
    write_json(os.path.join(run, "logs", "run_config.json"), cfg)
    code, out = _run([sys.executable, "rr_train.py", "--run-dir", run, "--mode", "overfit",
                      "--max-steps", "300"])
    ov = json.load(open(os.path.join(run, "logs", "overfit.json"))) if code == 0 else None
    assert ov is not None, out[-2000:]
    assert ov["passed"] and ov["n_sentences"] == 32, ov
    return {"train_accuracy": ov["train_accuracy"], "steps": ov["global_step"]}


def t7_resume(data_dir: str, root: str) -> Dict:
    p1 = write_phase1(root, "M4")
    run = os.path.join(root, "run_resume")
    cfg = base_cfg("lora", "M4", p1, data_dir, epochs=2, effective_batch=2, micro_batch=2,
                   watchdog_min_f1=-1.0)
    write_json(os.path.join(run, "logs", "run_config.json"), cfg)
    code, out = _run([sys.executable, "rr_train.py", "--run-dir", run, "--mode", "train",
                      "--kill-at-step", "3"])
    assert code == 137, f"expected the simulated kill (137), got {code}: {out[-1500:]}"
    assert os.path.exists(os.path.join(run, "checkpoints", "last.pt"))
    code, out = _run([sys.executable, "rr_train.py", "--run-dir", run, "--mode", "train"])
    assert code == 0, out[-2000:]
    rj = json.load(open(os.path.join(run, "logs", "resume.json")))
    s = json.load(open(os.path.join(run, "logs", "train_summary.json")))
    assert rj["resumed_at_step"] == 3, rj
    assert s["resumed_from_step"] == 3 and s["global_step"] > 3
    assert s["last_loss"] is not None and math.isfinite(s["last_loss"])
    return {"resumed_at_step": 3, "final_step": s["global_step"], "last_loss": s["last_loss"]}


def t_full_size(root: str, data_dir: str) -> Dict:
    """Trainable % of every technique on the REAL-size architectures (random weights)."""
    out = {}
    for mid in MODELS:
        p1 = write_phase1(root, mid, FULL)
        m0, _ = build_rr_model(p1, 13)
        for tech in PEFT:
            m = copy.deepcopy(m0)
            cfg = base_cfg(tech, mid, p1, data_dir)
            get_technique(tech).apply(m, cfg, log=lambda *a: None)
            pct = model_size(m)["trainable_pct"]
            assert pct <= MAX_TRAINABLE_PCT[tech], f"{mid}/{tech}: {pct:.2f}%"
            out[f"{mid}/{tech}"] = round(pct, 4)
            del m
        del m0
    return out


# =============================================================================================
# job checks on the REAL model (called by rr_train --mode smoke)
# =============================================================================================
def job_checks(cfg: Dict, meta: Dict, docs: Dict, device) -> Dict:
    res: Dict = {"checks": {}, "failed": []}

    def run(name: str, fn: Callable) -> None:
        try:
            res["checks"][name] = fn()
        except Exception as e:
            res["failed"].append(name)
            res["checks"][name] = f"FAILED: {e}\n{traceback.format_exc()[-1500:]}"

    def outputs():
        keys = [k for k in meta["split"]["dev"] + meta["split"]["test"]
                if docs[k]["domain"] in set(cfg["domains"])]
        wins = build_windows(docs, keys, cfg, meta, "infer")
        slot = [np.zeros((len(w["markers"]), cfg["n_labels"]), np.float32) for w in wins]
        comb = combine_slot_scores(wins, slot, {k: len(docs[k]["ids"]) for k in keys})
        assert all(comb[k].shape[0] == len(docs[k]["text"]) for k in keys)
        return {"docs": len(keys), "windows": len(wins)}
    run("output_lengths", outputs)
    tech = cfg["technique"]
    if tech in PEFT:
        def peft():
            m0, _ = build_rr_model(cfg["phase1_dir"], cfg["n_labels"], cfg["attn_mode"],
                                   cfg["head"], cfg.get("layer_ids"), cfg.get("dropout", 0.1),
                                   cfg.get("pool", "marker"))
            m0.to(device).eval()
            b = _batch(meta, docs, 2, device, cfg["attn_mode"], cfg["n_virtual"], 512)
            real_size = model_size(m0)["total_params"] >= SPEC["rules"]["foundation_params"][0]
            r = peft_checks(m0, cfg, b, tech, real_size, 1e-5, 1e-3)
            r["trainable_pct_checked"] = real_size
            del m0
            return r
        run("peft_zero_merge_pct", peft)
        if tech == "prefix" or tech == "prompt":
            def vis():
                m, _ = build_rr_model(cfg["phase1_dir"], cfg["n_labels"], cfg["attn_mode"])
                get_technique(tech).apply(m, dict(cfg, prefix_reparam=False), log=lambda *a: None)
                m.to(device).eval()
                b = _batch(meta, docs, 1, device, cfg["attn_mode"], cfg["n_virtual"])
                T = int(b["attention_mask"][0].sum())
                pos = sorted({0, 1, T // 2, T - 1} | ({min(T - 1, 600)} if T > 600 else set()))
                param = m.trunk.prefix.prefix_kv if tech == "prefix" else m.trunk.prompt
                blind = visibility(m, b, param, pos)
                assert not blind, f"positions {blind} cannot see the {tech}"
                return {"positions": pos}
            run("virtual_token_visibility", vis)
    if device.type == "cuda":
        torch.cuda.empty_cache()
    res["ok"] = not res["failed"]
    return res


# =============================================================================================
def run_all(data_dir: str = None, full_size: bool = False, log=print) -> Dict:
    res = {"tests": {}, "failed": [], "started": time.strftime("%Y-%m-%d %H:%M:%S")}
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
    with tempfile.TemporaryDirectory() as root:
        syn = synthetic_cache(os.path.join(root, "data"))
        real = data_dir if data_dir and os.path.exists(os.path.join(data_dir, "meta.json")) \
            else None
        tests = [("1_macro_f1", lambda: t1_metrics()),
                 ("2_window_packing", lambda: t2_packing()),
                 ("3_output_lengths_and_validator", lambda: t3_outputs(real or syn)),
                 ("4_peft_techniques", lambda: t4_peft(syn, root)),
                 ("5_prefix_prompt_visibility", lambda: t5_visibility(syn, root)),
                 ("6_overfit_32", lambda: t6_overfit(syn, root)),
                 ("7_kill_resume", lambda: t7_resume(syn, root))]
        if full_size:
            tests.append(("8_full_size_trainable_pct", lambda: t_full_size(root, syn)))
        for name, fn in tests:
            t0 = time.time()
            try:
                res["tests"][name] = {"result": fn(), "seconds": round(time.time() - t0, 1)}
                log(f"   PASS {name} ({time.time() - t0:.1f}s)")
            except Exception as e:
                res["failed"].append(name)
                res["tests"][name] = {"error": str(e), "trace": traceback.format_exc()[-3000:]}
                log(f"   FAIL {name}: {e}")
    res["ok"] = not res["failed"]
    return res


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=None)
    ap.add_argument("--data-dir", default=None)
    ap.add_argument("--full-size", action="store_true")
    a = ap.parse_args()
    res = run_all(a.data_dir, a.full_size)
    if a.out:
        write_json(a.out, res)
    print(json.dumps({"ok": res["ok"], "failed": res["failed"]}))
    sys.exit(0 if res["ok"] else 1)


if __name__ == "__main__":
    main()


In [ ]:
# every module must be byte-identical to the generated version (same code in all notebooks)
bad = []
for rel, h in EXPECTED_MD5.items():
    p = os.path.join("/kaggle/working/phase3", rel)
    if hashlib.md5(open(p, "rb").read()).hexdigest() != h:
        bad.append(rel)
assert not bad, f"modules differ from the generated version: {bad}"
print(f"all {len(EXPECTED_MD5)} modules verified")
sys.path.insert(0, "/kaggle/working/phase3")
from rr_common import SPEC, default_input_roots, find_files, find_phase1_models
print("Phase-1 weights:", json.dumps(find_phase1_models(default_input_roots()), indent=1))
print("bpe_model.json:", find_files("bpe_model.json", default_input_roots())[:4])
print("labels (", len(SPEC['labels']), "):", SPEC['labels'])


In [ ]:
# data discovery + gates G0 (schema, ID collisions), G1 (tokenisation), G2 (packing and
# validator round trips); reuses an attached phase3_data cache when it is valid
!cd /kaggle/working/phase3 && python rr_data.py build --out /kaggle/working/phase3_data


In [ ]:
!cd /kaggle/working/phase3 && python run_experiment.py --exp E0 --resume auto

In [ ]:
OUT = "/kaggle/working/phase3_out"
FOLDER = "E0_full_ft"
print(open(f"{OUT}/INDEX.md").read() if os.path.exists(f"{OUT}/INDEX.md") else "no INDEX.md yet")
p = f"{OUT}/{FOLDER}/RESULTS.json"
rows = json.load(open(p)) if os.path.exists(p) else []
print(f"{'job':<30}{'dev F1':>8}{'CL':>8}{'IT':>8}{'trainable':>13}{'total':>13}{'min':>7}")
for r in rows:
    d = r.get('dev_per_domain_macro_f1') or {}
    print(f"{r['job']:<30}{r['dev_macro_f1']:>8.4f}{d.get('CL', float('nan')):>8.4f}{d.get('IT', float('nan')):>8.4f}"
          f"{r.get('trainable_params') or 0:>13,}{r.get('total_params') or 0:>13,}{r.get('train_minutes') or 0:>7.1f}")
for f in ("SELECTION.json", "validation_report.json"):
    q = f"{OUT}/{FOLDER}/BEST/{f}"
    if os.path.exists(q):
        print(f, json.dumps(json.load(open(q)), indent=1)[:1500])
for f in ("PILOT.json", "REPORT_TABLES.md"):
    q = f"{OUT}/{f}"
    if os.path.exists(q):
        print(open(q).read()[:6000])


In [ ]:
OUT = "/kaggle/working/phase3_out"
for dp, dn, fn in sorted(os.walk(OUT)):
    depth = dp.count(os.sep) - OUT.count(os.sep)
    if depth <= 2:
        size = sum(os.path.getsize(os.path.join(dp, f)) for f in fn) / 1e6
        print("  " * depth + os.path.basename(dp) + f"/  ({size:.1f} MB in files here)")
print("\nUpload to Codabench (validated):")
for z in sorted(glob.glob(f"{OUT}/*/BEST/submission.zip")):
    print("  ", z)


**Run this notebook as a commit** (Save Version -> Save & Run All): it keeps running without the browser, and `/kaggle/working/phase3_out` (+ `phase3_data`) becomes the notebook's output. Attach that output to the next session of this notebook to skip finished jobs / resume an unfinished one, and to E11.

Upload `phase3_out/E0_full_ft/BEST/submission.zip` to Codabench. `python rr_select.py --coverage` lists the winners, the rule compliance and any extra upload needed so that all four Phase-1 models are represented.